In [3]:
# ============================================================
# Zelle 1 — Daten laden + Qualitätscheck + Baseline-Bereinigung
# ============================================================
"""
Analyst-Notiz:
In dieser ersten Zelle lade ich den Datensatz und mache einen kompakten Qualitäts- und Plausibilitätscheck.
Ziel: Wir wollen spätere Modelle/Cluster nicht von Tippfehlern, Extremwerten oder inkonsistenten Zählungen verzerren lassen.

Entscheidungen (bewusst & nachvollziehbar):
1) Familienstand: seltene „Müll“-Kategorien werden harmonisiert (z. B. „Allein“ → „Ledig“, „Absurd“ → „Sonstiges“).
2) Einkommen: offensichtlicher Extremwert (z. B. 666666) wird als Ausreißer behandelt; sehr niedrige Einkommen unter 1%-Quantil ebenfalls.
   → Beides wird zu NaN gesetzt, damit wir später sauber imputen können (statt die Verteilung zu zerstören).
3) Geburtsjahr: Plausibilitätsgrenze Alter 18–90 (Stichtag 2026-01-29). Unplausible Werte → NaN.
4) Kaufzählungen: Rabattkäufe dürfen nicht größer sein als die Summe der Kanal-Käufe → wird auf Kanal-Summe gekappt.
5) Wir markieren Sonderfälle (z. B. Ausgaben > 0, aber Käufe = 0) als Flag, statt blind „wegzuputzen“.
"""

import numpy as np
import pandas as pd
from pathlib import Path

# -------------------------
# 0) CSV im Projektordner finden (robust, ohne harte Pfade)
# -------------------------
dateiname = "Marktkampagne.csv"

kandidaten = [
    Path(dateiname),                       # gleiche Ebene wie das Notebook
    Path("data") / dateiname,              # typischer Unterordner
    Path("daten") / dateiname,             # typischer Unterordner (de)
    Path("input") / dateiname,             # typischer Unterordner
    Path("/mnt/data") / dateiname,         # falls im Sandbox-Umfeld
]

pfad_csv = None

# 1) Direkt-Kandidaten prüfen
for p in kandidaten:
    if p.exists():
        pfad_csv = p
        break

# 2) Falls nicht gefunden: im aktuellen Projektordner suchen (rekursiv, aber begrenzt)
if pfad_csv is None:
    treffer = []
    for p in Path(".").rglob(dateiname):
        treffer.append(p)
        if len(treffer) >= 10:  # Sicherheitsbremse
            break

    if treffer:
        pfad_csv = treffer[0]
        print("⚠️ Hinweis: CSV wurde per Suche gefunden. Trefferliste (max 10):")
        for t in treffer:
            print(" -", t.resolve())
    else:
        raise FileNotFoundError(
            f"CSV '{dateiname}' nicht gefunden.\n"
            f"Aktueller Ordner: {Path.cwd()}\n"
            f"Geprüfte Pfade: {[str(k.resolve()) for k in kandidaten]}\n"
            f"Lege die Datei in den Notebook-Ordner oder passe 'kandidaten' oben an."
        )

# -------------------------
# 1) Daten einlesen
# -------------------------
df_raw = pd.read_csv(pfad_csv, sep=None, engine="python", encoding="utf-8")
df = df_raw.copy()

# Datum sauber parsen (Format: TT-MM-JJJJ)
df["Datum_Kunde"] = pd.to_datetime(df["Datum_Kunde"], format="%d-%m-%Y", errors="coerce")

# -------------------------
# 2) Baseline-Checks (vor Bereinigung)
# -------------------------
print("\n=== DATEN-ÜBERBLICK (RAW) ===")
print(f"Zeilen/Spalten: {df.shape[0]} / {df.shape[1]}")
print(f"Eindeutige IDs: {df['ID'].nunique()} von {len(df)} (sollte identisch sein)")
print(f"Fehlende Einkommen (vorher): {df['Einkommen'].isna().sum()}")
print()

# -------------------------
# 3) Familienstand harmonisieren
# -------------------------
familienstand_map = {
    "Allein": "Ledig",
    "Absurd": "Sonstiges",
    "Man lebt nur einmal": "Sonstiges"
}
familienstand_vor = df["Familienstand"].copy()
df["Familienstand"] = df["Familienstand"].replace(familienstand_map)
familienstand_geaendert = (familienstand_vor != df["Familienstand"]).sum()

# -------------------------
# 4) Einkommen: Ausreißer behandeln (robust)
#    - sehr niedrig: unter 1%-Quantil
#    - extrem hoch: > 200.000 (harte Plausibilitätsgrenze)
# -------------------------
income = df["Einkommen"]
q01 = income.quantile(0.01)
q99 = income.quantile(0.99)
high_cap = 200_000

mask_low = income.notna() & (income < q01)
mask_high = income.notna() & (income > high_cap)

df.loc[mask_low | mask_high, "Einkommen"] = np.nan

# -------------------------
# 5) Geburtsjahr plausibilisieren (Alter 18–90)
# -------------------------
stichtag = pd.Timestamp("2026-01-29")
alter = stichtag.year - df["Geburtsjahr"]
mask_alter_unplausibel = alter.notna() & ((alter < 18) | (alter > 90))

df.loc[mask_alter_unplausibel, "Geburtsjahr"] = np.nan

# -------------------------
# 6) Kaufzählungen: Konsistenz herstellen
#    Rabattkäufe <= Summe(Web+Katalog+Laden)
# -------------------------
kanal_kaeufe = df["Anzahl_Webkäufe"] + df["Anzahl_Katalogkäufe"] + df["Anzahl_Ladeneinkäufe"]
mask_rabatt_inkonsistent = df["Anzahl_Rabattkäufe"] > kanal_kaeufe
df.loc[mask_rabatt_inkonsistent, "Anzahl_Rabattkäufe"] = kanal_kaeufe[mask_rabatt_inkonsistent]

# -------------------------
# 7) Praktische Flags für spätere Analyse
# -------------------------
ausgaben_spalten = [c for c in df.columns if c.startswith("Ausgaben_")]
df["Ausgaben_Gesamt"] = df[ausgaben_spalten].sum(axis=1)
df["Kanal_Käufe_Gesamt"] = kanal_kaeufe
df["Flag_Ausgaben_ohne_Käufe"] = (df["Ausgaben_Gesamt"] > 0) & (df["Kanal_Käufe_Gesamt"] == 0)

# -------------------------
# 8) Kurzreport (nach Bereinigung)
# -------------------------
print("=== BEREINIGUNGS-REPORT (CLEAN BASELINE) ===")
print(f"Familienstand angepasst (Anzahl Werte): {familienstand_geaendert}")
print(f"Einkommen: 1%-Quantil = {q01:,.1f} | 99%-Quantil = {q99:,.1f}")
print(f"Einkommen als Ausreißer -> NaN gesetzt: niedrig={mask_low.sum()} | hoch={mask_high.sum()}")
print(f"Fehlende Einkommen (nachher): {df['Einkommen'].isna().sum()}")
print(f"Geburtsjahr unplausibel (Alter <18 oder >90) -> NaN gesetzt: {mask_alter_unplausibel.sum()}")
print(f"Rabattkäufe gekappt (Rabatt > Kanal-Käufe): {mask_rabatt_inkonsistent.sum()}")
print(f"Flag_Ausgaben_ohne_Käufe (Ausgaben > 0 bei Käufe=0): {df['Flag_Ausgaben_ohne_Käufe'].sum()}")
print()

print("=== FAMILIENSTAND (TOP) ===")
print(df["Familienstand"].value_counts().head(10))



=== DATEN-ÜBERBLICK (RAW) ===
Zeilen/Spalten: 2240 / 27
Eindeutige IDs: 2240 von 2240 (sollte identisch sein)
Fehlende Einkommen (vorher): 24

=== BEREINIGUNGS-REPORT (CLEAN BASELINE) ===
Familienstand angepasst (Anzahl Werte): 7
Einkommen: 1%-Quantil = 7,579.2 | 99%-Quantil = 94,458.8
Einkommen als Ausreißer -> NaN gesetzt: niedrig=23 | hoch=1
Fehlende Einkommen (nachher): 48
Geburtsjahr unplausibel (Alter <18 oder >90) -> NaN gesetzt: 3
Rabattkäufe gekappt (Rabatt > Kanal-Käufe): 3
Flag_Ausgaben_ohne_Käufe (Ausgaben > 0 bei Käufe=0): 6

=== FAMILIENSTAND (TOP) ===
Familienstand
Verheiratet       864
Zusammenlebend    580
Ledig             483
Geschieden        232
Verwitwet          77
Sonstiges           4
Name: count, dtype: int64


In [4]:
# ============================================================
# Zelle 2 — Feature-Engineering Blueprint (Plan + Designregeln)
# ============================================================
"""
Analyst-Notiz:
Wir haben jetzt bereinigte Rohdaten. Bevor wir Features „bauen“, definieren wir ein Blueprint:
- Welche Features wollen wir?
- Wie werden sie berechnet?
- Welcher Datentyp ist es (numerisch/kategorisch/Flag)?
- Für welche Aufgaben sind sie geeignet (Klassifikation/Regression/Clustering)?
- Gibt es Leakage-Risiko (also Informationen, die das Ziel unzulässig verraten)?

Wichtig: Wir bauen bewusst ein „Super-Set“ an Features, aber modular.
Später können wir je nach Aufgabe ein Feature-Subset auswählen, ohne alles neu zu erfinden.

Designregeln:
1) Keine Zielspalte in Features. Keine Features, die direkt aus dem Ziel abgeleitet sind.
2) Bei Quotienten (Divisionen) immer Schutz vor Division durch 0.
3) Bei stark schiefen Verteilungen (z. B. Ausgaben) planen wir optional Log-Features ein (log1p).
4) RFM-Logik (Recency/Frequency/Monetary) ist Kern, weil sie für Marketing interpretierbar ist.
"""

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1) Blueprint als strukturierte Tabelle
# ------------------------------------------------------------
# Spalten:
# - feature: Name des Features
# - beschreibung: Kurzbeschreibung in Business-Sprache
# - formel_quelle: Wie wird es gebildet (Spalten / Formel)
# - typ: numerisch / kategorisch / flag / score
# - geeignet_fuer: Klassifikation | Regression | Clustering
# - leakage_risiko: nein / prüfen / ja
# - hinweis: technische Hinweise (Skalierung, Log, Missing etc.)

blueprint = [
    # -------------------------
    # A) Zeit / Kundenbeziehung
    # -------------------------
    {
        "feature": "Kunde_seit_Tagen",
        "beschreibung": "Dauer der Kundenbeziehung (Tenure) in Tagen",
        "formel_quelle": "Stichtag - Datum_Kunde",
        "typ": "numerisch",
        "geeignet_fuer": "Klassifikation, Regression, Clustering",
        "leakage_risiko": "nein",
        "hinweis": "Stichtag definieren; Datum_Kunde muss datetime sein"
    },
    {
        "feature": "Kunde_seit_Monaten",
        "beschreibung": "Tenure in Monaten (robuster für Interpretation als Tage)",
        "formel_quelle": "Kunde_seit_Tagen / 30.44",
        "typ": "numerisch",
        "geeignet_fuer": "Klassifikation, Regression, Clustering",
        "leakage_risiko": "nein",
        "hinweis": "Rundung optional; Skalierung bei Clustering"
    },
    {
        "feature": "Flag_Neukunde",
        "beschreibung": "Neukunde-Flag (z. B. Kunde seit <= 90 Tage)",
        "formel_quelle": "Kunde_seit_Tagen <= 90",
        "typ": "flag",
        "geeignet_fuer": "Klassifikation, Clustering",
        "leakage_risiko": "nein",
        "hinweis": "Schwelle später begründen/anpassen"
    },

    # -------------------------
    # B) Demografie / Haushalt
    # -------------------------
    {
        "feature": "Alter",
        "beschreibung": "Alter in Jahren",
        "formel_quelle": "Stichtag_Jahr - Geburtsjahr",
        "typ": "numerisch",
        "geeignet_fuer": "Klassifikation, Regression, Clustering",
        "leakage_risiko": "nein",
        "hinweis": "Unplausible Geburtsjahre wurden bereits zu NaN gesetzt; später imputen"
    },
    {
        "feature": "Haushaltsgroesse",
        "beschreibung": "Haushaltsgröße als einfache Näherung",
        "formel_quelle": "1 + Kinder_im_Haushalt + Teenager_im_Haushalt",
        "typ": "numerisch",
        "geeignet_fuer": "Klassifikation, Regression, Clustering",
        "leakage_risiko": "nein",
        "hinweis": "Interpretation: Haushaltsstruktur, kein echter Zensus"
    },
    {
        "feature": "Flag_Kinder",
        "beschreibung": "Flag: Kinder im Haushalt vorhanden",
        "formel_quelle": "Kinder_im_Haushalt > 0",
        "typ": "flag",
        "geeignet_fuer": "Klassifikation, Clustering",
        "leakage_risiko": "nein",
        "hinweis": "Hilft bei Segmenten (Familie vs ohne Kinder)"
    },
    {
        "feature": "Flag_Teenager",
        "beschreibung": "Flag: Teenager im Haushalt vorhanden",
        "formel_quelle": "Teenager_im_Haushalt > 0",
        "typ": "flag",
        "geeignet_fuer": "Klassifikation, Clustering",
        "leakage_risiko": "nein",
        "hinweis": "Kann Produktpräferenzen beeinflussen"
    },

    # -------------------------
    # C) RFM / Kaufverhalten
    # -------------------------
    {
        "feature": "Recency_Tage",
        "beschreibung": "Tage seit letztem Kauf (Recency)",
        "formel_quelle": "Tage_letzter_Kauf",
        "typ": "numerisch",
        "geeignet_fuer": "Klassifikation, Regression, Clustering",
        "leakage_risiko": "nein",
        "hinweis": "Zentrale Marketingvariable; Skalierung bei Clustering"
    },
    {
        "feature": "Frequency_Kaeufe_Gesamt",
        "beschreibung": "Gesamte Kaufhäufigkeit über alle Kanäle",
        "formel_quelle": "Anzahl_Webkäufe + Anzahl_Katalogkäufe + Anzahl_Ladeneinkäufe",
        "typ": "numerisch",
        "geeignet_fuer": "Klassifikation, Regression, Clustering",
        "leakage_risiko": "nein",
        "hinweis": "Basis für Warenkorb- und Quotenfeatures"
    },
    {
        "feature": "Monetary_Ausgaben_Gesamt",
        "beschreibung": "Gesamtausgaben (Monetary)",
        "formel_quelle": "Summe aller Ausgaben_* Spalten",
        "typ": "numerisch",
        "geeignet_fuer": "Klassifikation, Regression, Clustering",
        "leakage_risiko": "nein",
        "hinweis": "Stark schief -> optional log1p(Monetary)"
    },
    {
        "feature": "Warenkorb_Durchschnitt",
        "beschreibung": "Durchschnittlicher Warenkorb pro Kauf",
        "formel_quelle": "Monetary_Ausgaben_Gesamt / Frequency_Kaeufe_Gesamt",
        "typ": "numerisch",
        "geeignet_fuer": "Regression, Clustering, Klassifikation",
        "leakage_risiko": "nein",
        "hinweis": "Nur berechnen, wenn Frequency > 0, sonst 0 oder NaN"
    },
    {
        "feature": "Rabattquote",
        "beschreibung": "Anteil der Rabattkäufe an allen Käufen",
        "formel_quelle": "Anzahl_Rabattkäufe / Frequency_Kaeufe_Gesamt",
        "typ": "numerisch",
        "geeignet_fuer": "Klassifikation, Clustering",
        "leakage_risiko": "nein",
        "hinweis": "Division durch 0 absichern; interpretierbar als Preis-Sensitivität"
    },
    {
        "feature": "Flag_Beschwerde",
        "beschreibung": "Beschwerde vorhanden (Service-/Zufriedenheitsindikator)",
        "formel_quelle": "Beschwerde",
        "typ": "flag",
        "geeignet_fuer": "Klassifikation, Clustering",
        "leakage_risiko": "nein",
        "hinweis": "Kann Kampagnenantwort negativ beeinflussen"
    },

    # -------------------------
    # D) Produktmix / Präferenzen
    # -------------------------
    {
        "feature": "Produktmix_Diversity",
        "beschreibung": "Anzahl Produktkategorien mit Ausgaben > 0",
        "formel_quelle": "Count(Ausgaben_* > 0)",
        "typ": "numerisch",
        "geeignet_fuer": "Clustering, Klassifikation",
        "leakage_risiko": "nein",
        "hinweis": "Breite des Einkaufsverhaltens; gut für Segmente"
    },
    {
        "feature": "Anteil_Wein",
        "beschreibung": "Anteil der Weinausgaben an Gesamtausgaben",
        "formel_quelle": "Ausgaben_Wein / Monetary_Ausgaben_Gesamt",
        "typ": "numerisch",
        "geeignet_fuer": "Clustering, Klassifikation",
        "leakage_risiko": "nein",
        "hinweis": "Division durch 0 absichern; Produktpräferenz-Proxy"
    },
    {
        "feature": "Anteil_Fleisch",
        "beschreibung": "Anteil der Fleischausgaben an Gesamtausgaben",
        "formel_quelle": "Ausgaben_Fleisch / Monetary_Ausgaben_Gesamt",
        "typ": "numerisch",
        "geeignet_fuer": "Clustering, Klassifikation",
        "leakage_risiko": "nein",
        "hinweis": "Wie oben; später analog für weitere Kategorien"
    },
    {
        "feature": "Anteil_Suesses",
        "beschreibung": "Anteil der Süßigkeiten-Ausgaben an Gesamtausgaben",
        "formel_quelle": "Ausgaben_Süßes / Monetary_Ausgaben_Gesamt",
        "typ": "numerisch",
        "geeignet_fuer": "Clustering, Klassifikation",
        "leakage_risiko": "nein",
        "hinweis": "Wie oben; später analog für weitere Kategorien"
    },

    # -------------------------
    # E) Kanalpräferenz / digital
    # -------------------------
    {
        "feature": "Anteil_Webkaeufe",
        "beschreibung": "Anteil der Webkäufe an allen Käufen",
        "formel_quelle": "Anzahl_Webkäufe / Frequency_Kaeufe_Gesamt",
        "typ": "numerisch",
        "geeignet_fuer": "Clustering, Klassifikation",
        "leakage_risiko": "nein",
        "hinweis": "Division durch 0 absichern; Kanalpräferenz"
    },
    {
        "feature": "Anteil_Katalogkaeufe",
        "beschreibung": "Anteil der Katalogkäufe an allen Käufen",
        "formel_quelle": "Anzahl_Katalogkäufe / Frequency_Kaeufe_Gesamt",
        "typ": "numerisch",
        "geeignet_fuer": "Clustering, Klassifikation",
        "leakage_risiko": "nein",
        "hinweis": "Division durch 0 absichern; Kanalpräferenz"
    },
    {
        "feature": "Anteil_Ladeneinkaeufe",
        "beschreibung": "Anteil der Ladeneinkäufe an allen Käufen",
        "formel_quelle": "Anzahl_Ladeneinkäufe / Frequency_Kaeufe_Gesamt",
        "typ": "numerisch",
        "geeignet_fuer": "Clustering, Klassifikation",
        "leakage_risiko": "nein",
        "hinweis": "Division durch 0 absichern; Kanalpräferenz"
    },
    {
        "feature": "Web_Interaktion_Intensitaet",
        "beschreibung": "Webbesuche pro Webkauf (Proxy für Recherche/Entscheidungsdauer)",
        "formel_quelle": "Anzahl_WebBesuche_Monat / (Anzahl_Webkäufe + 1)",
        "typ": "numerisch",
        "geeignet_fuer": "Clustering, Klassifikation",
        "leakage_risiko": "nein",
        "hinweis": "Stabilisiert durch +1 im Nenner; bei Clustering skalieren"
    },

    # -------------------------
    # F) Kampagnenhistorie (nur bedingt nutzen)
    # -------------------------
    {
        "feature": "Kampagnen_Historie_Summe",
        "beschreibung": "Wie oft Kunde auf frühere Kampagnen reagiert hat (historisch)",
        "formel_quelle": "Summe(Kampagne_1..Kampagne_5)",
        "typ": "numerisch",
        "geeignet_fuer": "Klassifikation",
        "leakage_risiko": "prüfen",
        "hinweis": "Nur nutzen, wenn Kampagne_1..5 zeitlich vor Zielkampagne liegen (Leakage prüfen)"
    },
    {
        "feature": "Flag_Zuvor_Reagiert",
        "beschreibung": "Flag: Kunde hat schon einmal auf eine Kampagne reagiert",
        "formel_quelle": "Kampagnen_Historie_Summe > 0",
        "typ": "flag",
        "geeignet_fuer": "Klassifikation",
        "leakage_risiko": "prüfen",
        "hinweis": "Wie oben; sehr starkes Merkmal, daher zeitliche Logik absichern"
    },
]

blueprint_df = pd.DataFrame(blueprint)

print("=== FEATURE-ENGINEERING BLUEPRINT ===")
print(f"Anzahl geplanter Features: {len(blueprint_df)}")
print()
print(blueprint_df[["feature", "typ", "geeignet_fuer", "leakage_risiko"]])

print("\n=== DETAILANSICHT (erste 10 Zeilen) ===")
print(blueprint_df.head(10))


=== FEATURE-ENGINEERING BLUEPRINT ===
Anzahl geplanter Features: 23

                        feature        typ  \
0              Kunde_seit_Tagen  numerisch   
1            Kunde_seit_Monaten  numerisch   
2                 Flag_Neukunde       flag   
3                         Alter  numerisch   
4              Haushaltsgroesse  numerisch   
5                   Flag_Kinder       flag   
6                 Flag_Teenager       flag   
7                  Recency_Tage  numerisch   
8       Frequency_Kaeufe_Gesamt  numerisch   
9      Monetary_Ausgaben_Gesamt  numerisch   
10       Warenkorb_Durchschnitt  numerisch   
11                  Rabattquote  numerisch   
12              Flag_Beschwerde       flag   
13         Produktmix_Diversity  numerisch   
14                  Anteil_Wein  numerisch   
15               Anteil_Fleisch  numerisch   
16               Anteil_Suesses  numerisch   
17             Anteil_Webkaeufe  numerisch   
18         Anteil_Katalogkaeufe  numerisch   
19        A

In [8]:
# ============================================================
# Zelle 3 — Feature Engineering (Umsetzung) + Feature-QA
# ============================================================
"""
Analyst-Notiz:
Diese Zelle setzt den Blueprint (Zelle 2) in echte Features um – basierend auf den
tatsächlichen Spaltennamen aus Marktkampagne.csv.

Wichtige Prinzipien:
1) Reproduzierbarkeit: fester Stichtag für Alter/Tenure.
2) Robustheit: sichere Divisionen, saubere Defaults bei 0 im Nenner.
3) Trennung: Features werden getrennt von potenziellen Zielvariablen gehalten.
4) Qualität: am Ende ein Feature-QA-Report (Missing, Ranges, Plausibilität).
"""

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0) Voraussetzung: df muss aus Zelle 1 existieren
# ------------------------------------------------------------
if "df" not in globals():
    raise RuntimeError("df ist nicht vorhanden. Bitte zuerst Zelle 1 ausführen (Daten laden und bereinigen).")

# ------------------------------------------------------------
# 1) Spaltenprüfung (konkret nach CSV)
# ------------------------------------------------------------
required_cols = [
    "ID",
    "Geburtsjahr",
    "Kinder_zu_Hause",
    "Teenager_zu_Hause",
    "Datum_Kunde",
    "Letzter_Kauf_Tage",
    "Beschwerde",
    "Anzahl_Rabattkäufe",
    "Anzahl_Webkäufe",
    "Anzahl_Katalogkäufe",
    "Anzahl_Ladeneinkäufe",
    "Anzahl_WebBesuche_Monat",
    "Ausgaben_Wein",
    "Ausgaben_Fleisch",
    "Ausgaben_Süßigkeiten",
    "Kampagne_1_Akzeptiert",
    "Kampagne_2_Akzeptiert",
    "Kampagne_3_Akzeptiert",
    "Kampagne_4_Akzeptiert",
    "Kampagne_5_Akzeptiert",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(
        "Fehlende Spalten:\n- " + "\n- ".join(missing) +
        "\n\nVorhandene Spalten sind:\n- " + "\n- ".join(df.columns.astype(str).tolist())
    )

ausgaben_spalten = [c for c in df.columns if c.startswith("Ausgaben_")]
if not ausgaben_spalten:
    raise KeyError("Keine Ausgaben_* Spalten gefunden. Prüfe die CSV-Spaltennamen.")

# ------------------------------------------------------------
# 2) Hilfsfunktion: sichere Division
# ------------------------------------------------------------
def safe_divide(numerator, denominator, default=0.0):
    num = pd.to_numeric(numerator, errors="coerce")
    den = pd.to_numeric(denominator, errors="coerce")
    out = np.where((den == 0) | pd.isna(den), default, num / den)
    return pd.Series(out, index=num.index)

# ------------------------------------------------------------
# 3) Stichtag festlegen (reproduzierbar)
# ------------------------------------------------------------
STICHTAG = pd.Timestamp("2026-01-29")

# Datum sicherstellen
df["Datum_Kunde"] = pd.to_datetime(df["Datum_Kunde"], errors="coerce")

# ------------------------------------------------------------
# 4) Feature Engineering nach Blueprint (Zelle 2)
# ------------------------------------------------------------
kanal_kaeufe = df["Anzahl_Webkäufe"] + df["Anzahl_Katalogkäufe"] + df["Anzahl_Ladeneinkäufe"]
ausgaben_gesamt = df[ausgaben_spalten].apply(pd.to_numeric, errors="coerce").sum(axis=1)

features = pd.DataFrame(index=df.index)
features["ID"] = df["ID"]

# A) Zeit / Kundenbeziehung
features["Kunde_seit_Tagen"] = (STICHTAG - df["Datum_Kunde"]).dt.days
features["Kunde_seit_Monaten"] = features["Kunde_seit_Tagen"] / 30.44
features["Flag_Neukunde"] = (features["Kunde_seit_Tagen"] <= 90).astype(int)

# B) Demografie / Haushalt
features["Alter"] = STICHTAG.year - pd.to_numeric(df["Geburtsjahr"], errors="coerce")
features["Haushaltsgroesse"] = 1 + pd.to_numeric(df["Kinder_zu_Hause"], errors="coerce") + pd.to_numeric(df["Teenager_zu_Hause"], errors="coerce")
features["Flag_Kinder"] = (pd.to_numeric(df["Kinder_zu_Hause"], errors="coerce") > 0).astype(int)
features["Flag_Teenager"] = (pd.to_numeric(df["Teenager_zu_Hause"], errors="coerce") > 0).astype(int)

# C) RFM / Kaufverhalten
features["Recency_Tage"] = pd.to_numeric(df["Letzter_Kauf_Tage"], errors="coerce")
features["Frequency_Kaeufe_Gesamt"] = pd.to_numeric(kanal_kaeufe, errors="coerce")
features["Monetary_Ausgaben_Gesamt"] = pd.to_numeric(ausgaben_gesamt, errors="coerce")

# Ø Warenkorb: wenn keine Käufe -> NaN (nicht definiert)
features["Warenkorb_Durchschnitt"] = safe_divide(
    features["Monetary_Ausgaben_Gesamt"],
    features["Frequency_Kaeufe_Gesamt"],
    default=np.nan
)

# Rabattquote: wenn keine Käufe -> 0
features["Rabattquote"] = safe_divide(
    df["Anzahl_Rabattkäufe"],
    features["Frequency_Kaeufe_Gesamt"],
    default=0.0
)

features["Flag_Beschwerde"] = pd.to_numeric(df["Beschwerde"], errors="coerce").fillna(0).astype(int)

# D) Produktmix / Präferenzen
df_spend = df[ausgaben_spalten].apply(pd.to_numeric, errors="coerce").fillna(0)
features["Produktmix_Diversity"] = (df_spend > 0).sum(axis=1).astype(int)

features["Anteil_Wein"] = safe_divide(df["Ausgaben_Wein"], features["Monetary_Ausgaben_Gesamt"], default=0.0)
features["Anteil_Fleisch"] = safe_divide(df["Ausgaben_Fleisch"], features["Monetary_Ausgaben_Gesamt"], default=0.0)
features["Anteil_Suesses"] = safe_divide(df["Ausgaben_Süßigkeiten"], features["Monetary_Ausgaben_Gesamt"], default=0.0)

# E) Kanalpräferenz / digital
features["Anteil_Webkaeufe"] = safe_divide(df["Anzahl_Webkäufe"], features["Frequency_Kaeufe_Gesamt"], default=0.0)
features["Anteil_Katalogkaeufe"] = safe_divide(df["Anzahl_Katalogkäufe"], features["Frequency_Kaeufe_Gesamt"], default=0.0)
features["Anteil_Ladeneinkaeufe"] = safe_divide(df["Anzahl_Ladeneinkäufe"], features["Frequency_Kaeufe_Gesamt"], default=0.0)

features["Web_Interaktion_Intensitaet"] = safe_divide(
    df["Anzahl_WebBesuche_Monat"],
    (pd.to_numeric(df["Anzahl_Webkäufe"], errors="coerce") + 1),
    default=0.0
)

# F) Kampagnenhistorie (Leakage später prüfen, hier nur berechnet)
kampagnen_cols = [
    "Kampagne_1_Akzeptiert",
    "Kampagne_2_Akzeptiert",
    "Kampagne_3_Akzeptiert",
    "Kampagne_4_Akzeptiert",
    "Kampagne_5_Akzeptiert",
]
features["Kampagnen_Historie_Summe"] = df[kampagnen_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1).astype(int)
features["Flag_Zuvor_Reagiert"] = (features["Kampagnen_Historie_Summe"] > 0).astype(int)

# Optional: Flag aus Zelle 1 übernehmen, falls vorhanden
if "Flag_Ausgaben_ohne_Käufe" in df.columns:
    features["Flag_Ausgaben_ohne_Käufe"] = pd.to_numeric(df["Flag_Ausgaben_ohne_Käufe"], errors="coerce").fillna(0).astype(int)

# ------------------------------------------------------------
# 5) Feature-Gruppen / Typen (für spätere Subsets & Pipelines)
# ------------------------------------------------------------
feature_groups = {
    "zeit_tenure": ["Kunde_seit_Tagen", "Kunde_seit_Monaten", "Flag_Neukunde"],
    "demografie_haushalt": ["Alter", "Haushaltsgroesse", "Flag_Kinder", "Flag_Teenager"],
    "rfm_kaufverhalten": [
        "Recency_Tage",
        "Frequency_Kaeufe_Gesamt",
        "Monetary_Ausgaben_Gesamt",
        "Warenkorb_Durchschnitt",
        "Rabattquote",
        "Flag_Beschwerde",
    ],
    "produktmix": ["Produktmix_Diversity", "Anteil_Wein", "Anteil_Fleisch", "Anteil_Suesses"],
    "kanal_digital": ["Anteil_Webkaeufe", "Anteil_Katalogkaeufe", "Anteil_Ladeneinkaeufe", "Web_Interaktion_Intensitaet"],
    "kampagnen_optional": ["Kampagnen_Historie_Summe", "Flag_Zuvor_Reagiert"],
}

flag_features = [c for c in features.columns if c.startswith("Flag_")]
numeric_features = [c for c in features.columns if c not in (["ID"] + flag_features)]

# ------------------------------------------------------------
# 6) Feature-QA Report
# ------------------------------------------------------------
print("=== FEATURE ENGINEERING OUTPUT ===")
print(f"df_features Shape: {features.shape[0]} Zeilen / {features.shape[1]} Spalten")
print()

print("=== MISSING VALUES (Top 15) ===")
missing_counts = features.isna().sum().sort_values(ascending=False)
print(missing_counts.head(15))
print()

print("=== PLAUSIBILITÄTSCHECKS ===")
anteil_cols = [
    "Anteil_Wein",
    "Anteil_Fleisch",
    "Anteil_Suesses",
    "Anteil_Webkaeufe",
    "Anteil_Katalogkaeufe",
    "Anteil_Ladeneinkaeufe",
]
out_of_range = {c: int(((features[c] < -1e-9) | (features[c] > 1 + 1e-9)).sum()) for c in anteil_cols}
print("Anteil-Spalten außerhalb [0,1] (sollte 0 sein):")
print(out_of_range)
print()

flag_issues = {}
for c in flag_features:
    vals = set(features[c].dropna().unique().tolist())
    bad = vals - {0, 1}
    if bad:
        flag_issues[c] = bad
print("Flag-Spalten mit Werten ungleich {0,1} (sollte leer sein):")
print(flag_issues if flag_issues else {})
print()

print("=== NUMERIC DESCRIBE (Auszug) ===")
with pd.option_context("display.max_columns", 120):
    print(features[numeric_features].describe().T[["count", "mean", "std", "min", "50%", "max"]])
print()

print("=== HEAD (erste 5 Zeilen) ===")
with pd.option_context("display.max_columns", 120):
    print(features.head(5))

# Ergebnisobjekte für die nächsten Schritte
df_features = features.copy()
print("\nObjekte bereitgestellt: df_features, feature_groups, numeric_features, flag_features")


=== FEATURE ENGINEERING OUTPUT ===
df_features Shape: 2240 Zeilen / 25 Spalten

=== MISSING VALUES (Top 15) ===
Warenkorb_Durchschnitt      6
Alter                       3
ID                          0
Kunde_seit_Monaten          0
Kunde_seit_Tagen            0
Haushaltsgroesse            0
Flag_Kinder                 0
Flag_Teenager               0
Flag_Neukunde               0
Recency_Tage                0
Frequency_Kaeufe_Gesamt     0
Monetary_Ausgaben_Gesamt    0
Rabattquote                 0
Flag_Beschwerde             0
Produktmix_Diversity        0
dtype: int64

=== PLAUSIBILITÄTSCHECKS ===
Anteil-Spalten außerhalb [0,1] (sollte 0 sein):
{'Anteil_Wein': 0, 'Anteil_Fleisch': 0, 'Anteil_Suesses': 0, 'Anteil_Webkaeufe': 0, 'Anteil_Katalogkaeufe': 0, 'Anteil_Ladeneinkaeufe': 0}

Flag-Spalten mit Werten ungleich {0,1} (sollte leer sein):
{}

=== NUMERIC DESCRIBE (Auszug) ===
                              count         mean          std         min  \
Kunde_seit_Tagen             2240

In [9]:
# ============================================================
# Zelle 4 — Feature-Set Auswahl + Preprocessing-Setup (bereit für Cluster & Modelle)
# ============================================================
"""
Analyst-Notiz:
Wir haben jetzt df_features aus Zelle 3 (Feature Engineering Umsetzung) und feature_groups.
In dieser Zelle definieren wir die „arbeitsfähigen“ Feature-Sets und bereiten sie technisch so vor,
dass wir damit direkt weiterarbeiten können:

1) Feature-Set Cluster (Segmentierung):
   - Fokus auf Verhalten (RFM), Produktmix, Kanäle und Tenure.
   - Kampagnenhistorie wird bewusst NICHT genutzt (Leakage-/Interpretationsrisiko).

2) Feature-Set Model (Supervised Learning):
   - Safe Features + optional Kampagnenhistorie (bleibt erstmal als Option).
   - Zielvariablen werden hier noch nicht festgelegt; wir erzeugen nur X.

3) Preprocessing-Regeln:
   - Numerische Features: Median-Imputation (robust gegenüber Ausreißern).
   - Flags (0/1): bleiben wie sie sind.
   - Skalierung:
       * Für Clustering: Standardisierung (damit Monetary nicht alles dominiert).
       * Für Modelle: hier erstmal unskaliert bereitstellen; Skalierung erfolgt modellabhängig in der Modellzelle.
"""

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0) Voraussetzungen: df_features + feature_groups müssen existieren
# ------------------------------------------------------------
if "df_features" not in globals():
    raise RuntimeError("df_features fehlt. Bitte zuerst Zelle 3 ausführen.")
if "feature_groups" not in globals():
    raise RuntimeError("feature_groups fehlt. Bitte zuerst Zelle 3 ausführen.")

# ------------------------------------------------------------
# 1) Feature-Sets definieren (modular nach Zelle 2 Blueprint)
# ------------------------------------------------------------
# Cluster-Set (ohne Kampagnenhistorie)
features_cluster = (
    feature_groups["zeit_tenure"]
    + feature_groups["demografie_haushalt"]
    + feature_groups["rfm_kaufverhalten"]
    + feature_groups["produktmix"]
    + feature_groups["kanal_digital"]
)

# Model-Set: Safe-Basis (ohne kampagnen_optional). Kampagnen können später optional ergänzt werden.
features_model_safe = features_cluster.copy()

# Optionales Model-Set inkl. Kampagnenhistorie (später Leakage prüfen)
features_model_with_campaign = features_model_safe + feature_groups["kampagnen_optional"]

# ------------------------------------------------------------
# 2) X-Matrizen erzeugen (nur Features, keine ID)
# ------------------------------------------------------------
X_cluster_raw = df_features[features_cluster].copy()
X_model_safe_raw = df_features[features_model_safe].copy()
X_model_with_campaign_raw = df_features[features_model_with_campaign].copy()

# ------------------------------------------------------------
# 3) Imputation-Regeln (Median) — nur numerische Spalten
#    Flags werden mitgeführt, aber median funktioniert auch, da 0/1 numerisch ist.
# ------------------------------------------------------------
def median_impute(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    med = out.median(numeric_only=True)
    out = out.fillna(med)
    return out

X_cluster_imputed = median_impute(X_cluster_raw)
X_model_safe_imputed = median_impute(X_model_safe_raw)
X_model_with_campaign_imputed = median_impute(X_model_with_campaign_raw)

# ------------------------------------------------------------
# 4) Skalierung für Clustering (Standardisierung)
#    (Wir implementieren StandardScaler "per Hand", damit keine Imports nötig sind.)
# ------------------------------------------------------------
def standardize(frame: pd.DataFrame):
    """
    Standardisierung: (x - mean) / std
    Gibt zurück:
    - X_scaled
    - scaler_params (mean, std) für Reproduzierbarkeit / inverse_transform falls nötig
    """
    out = frame.copy()
    mean = out.mean(numeric_only=True)
    std = out.std(numeric_only=True).replace(0, 1)  # Schutz: std=0 -> 1
    out = (out - mean) / std
    params = {"mean": mean, "std": std}
    return out, params

X_cluster_scaled, scaler_cluster = standardize(X_cluster_imputed)

# Für Modelle geben wir erstmal nur "imputed" aus.
# Skalierung (falls nötig) machen wir im Modellblock abhängig vom Algorithmus.
X_model_safe = X_model_safe_imputed.copy()
X_model_with_campaign = X_model_with_campaign_imputed.copy()

# ------------------------------------------------------------
# 5) Mini-QA (damit wir sofort sehen, ob alles passt)
# ------------------------------------------------------------
print("=== ZELLE 4: OUTPUT CHECK ===")
print(f"Cluster-Features: {len(features_cluster)}")
print(f"Model-Features (safe): {len(features_model_safe)}")
print(f"Model-Features (mit Kampagnen): {len(features_model_with_campaign)}")
print()

print("Shapes:")
print("X_cluster_scaled:", X_cluster_scaled.shape)
print("X_model_safe:", X_model_safe.shape)
print("X_model_with_campaign:", X_model_with_campaign.shape)
print()

# Missing-Check nach Imputation
print("Missing nach Imputation (sollte 0 sein):")
print("X_cluster_imputed:", int(X_cluster_imputed.isna().sum().sum()))
print("X_model_safe:", int(X_model_safe.isna().sum().sum()))
print("X_model_with_campaign:", int(X_model_with_campaign.isna().sum().sum()))
print()

# Grobe Plausibilität: Standardisierung sollte ~0 mean haben
print("Cluster-Scaling Quickcheck (Mittelwerte, erste 10 Features):")
print(X_cluster_scaled.mean().head(10).round(3))
print()

print("Objekte bereitgestellt:")
print("- features_cluster, features_model_safe, features_model_with_campaign")
print("- X_cluster_scaled, scaler_cluster")
print("- X_model_safe, X_model_with_campaign")


=== ZELLE 4: OUTPUT CHECK ===
Cluster-Features: 21
Model-Features (safe): 21
Model-Features (mit Kampagnen): 23

Shapes:
X_cluster_scaled: (2240, 21)
X_model_safe: (2240, 21)
X_model_with_campaign: (2240, 23)

Missing nach Imputation (sollte 0 sein):
X_cluster_imputed: 0
X_model_safe: 0
X_model_with_campaign: 0

Cluster-Scaling Quickcheck (Mittelwerte, erste 10 Features):
Kunde_seit_Tagen            0.0
Kunde_seit_Monaten         -0.0
Flag_Neukunde               0.0
Alter                       0.0
Haushaltsgroesse            0.0
Flag_Kinder                -0.0
Flag_Teenager               0.0
Recency_Tage               -0.0
Frequency_Kaeufe_Gesamt    -0.0
Monetary_Ausgaben_Gesamt    0.0
dtype: float64

Objekte bereitgestellt:
- features_cluster, features_model_safe, features_model_with_campaign
- X_cluster_scaled, scaler_cluster
- X_model_safe, X_model_with_campaign


In [10]:
# ============================================================
# Zelle 5 — Clustering (KMeans): Clusteranzahl finden + Clusterprofile
# ============================================================
"""
Analyst-Notiz:
Jetzt, wo X_cluster_scaled aus Zelle 4 bereitsteht, können wir Kundensegmente bilden.

Vorgehen:
1) Wir testen mehrere Clusterzahlen (k) und messen:
   - Inertia (SSE) für den Elbow-Ansatz
   - Silhouette Score als Qualitätsmaß der Trennung (höher ist besser)
2) Wir wählen danach ein sinnvolles k (meist 3–6, abhängig von Stabilität/Interpretierbarkeit).
3) Wir erstellen Clusterprofile:
   - Mittelwerte der (nicht skalierten) Features pro Cluster für Interpretation
   - Größen der Cluster (Anzahl, Anteil)

Wichtig:
- Wir clustern auf skalierten Daten (X_cluster_scaled), interpretieren aber auf den Original-Skalen
  (X_cluster_imputed), damit man die Segmente fachlich erklären kann.
"""

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
if "X_cluster_scaled" not in globals():
    raise RuntimeError("X_cluster_scaled fehlt. Bitte zuerst Zelle 4 ausführen.")
if "X_cluster_imputed" not in globals():
    raise RuntimeError("X_cluster_imputed fehlt. Bitte zuerst Zelle 4 ausführen.")

# ------------------------------------------------------------
# 1) k-Range definieren und evaluieren
# ------------------------------------------------------------
k_values = list(range(2, 9))  # 2..8 ist für 2240 Zeilen ein guter Start
results = []

for k in k_values:
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = km.fit_predict(X_cluster_scaled)

    inertia = km.inertia_
    sil = silhouette_score(X_cluster_scaled, labels)

    results.append({"k": k, "inertia": inertia, "silhouette": sil})

eval_df = pd.DataFrame(results)

print("=== KMEANS EVALUATION (Elbow + Silhouette) ===")
print(eval_df)

# ------------------------------------------------------------
# 2) k automatisch vorschlagen (höchster Silhouette-Score)
#    Hinweis: Das ist ein Vorschlag. Business-Interpretierbarkeit zählt ebenfalls.
# ------------------------------------------------------------
best_row = eval_df.sort_values("silhouette", ascending=False).iloc[0]
k_best = int(best_row["k"])

print("\n=== AUTO-VORSCHLAG ===")
print(f"Bestes k nach Silhouette: k = {k_best} (Silhouette = {best_row['silhouette']:.4f})")

# ------------------------------------------------------------
# 3) Finales Modell fitten (mit k_best)
# ------------------------------------------------------------
kmeans_final = KMeans(n_clusters=k_best, n_init=50, random_state=42)
cluster_labels = kmeans_final.fit_predict(X_cluster_scaled)

# Labels in DataFrame ablegen
cluster_series = pd.Series(cluster_labels, index=X_cluster_scaled.index, name="Cluster")

# ------------------------------------------------------------
# 4) Clustergrößen
# ------------------------------------------------------------
cluster_sizes = cluster_series.value_counts().sort_index()
cluster_share = (cluster_sizes / len(cluster_series)).round(4)

cluster_overview = pd.DataFrame({
    "Cluster": cluster_sizes.index,
    "Anzahl": cluster_sizes.values,
    "Anteil": cluster_share.values
}).sort_values("Cluster")

print("\n=== CLUSTERGRÖSSEN ===")
print(cluster_overview)

# ------------------------------------------------------------
# 5) Clusterprofile (Interpretation auf Original-Skalen)
# ------------------------------------------------------------
X_profile = X_cluster_imputed.copy()
X_profile["Cluster"] = cluster_series

cluster_profile_mean = X_profile.groupby("Cluster").mean(numeric_only=True)
cluster_profile_median = X_profile.groupby("Cluster").median(numeric_only=True)

print("\n=== CLUSTERPROFILE (MEAN, Original-Skalen) ===")
with pd.option_context("display.max_columns", 120):
    print(cluster_profile_mean)

print("\n=== CLUSTERPROFILE (MEDIAN, Original-Skalen) ===")
with pd.option_context("display.max_columns", 120):
    print(cluster_profile_median)

# ------------------------------------------------------------
# 6) Ergebnisobjekte für die nächsten Zellen
# ------------------------------------------------------------
k_best_chosen = k_best
df_clustered = df_features.copy()
df_clustered["Cluster"] = cluster_series

print("\nObjekte bereitgestellt:")
print("- eval_df (k-Auswertung)")
print("- k_best_chosen")
print("- kmeans_final")
print("- df_clustered (df_features + Cluster)")
print("- cluster_overview")
print("- cluster_profile_mean, cluster_profile_median")


=== KMEANS EVALUATION (Elbow + Silhouette) ===
   k       inertia  silhouette
0  2  35939.950998    0.196860
1  3  32629.878853    0.153608
2  4  30016.405028    0.154646
3  5  27920.466695    0.162035
4  6  26552.762871    0.154180
5  7  25498.607287    0.140852
6  8  24539.627020    0.136790

=== AUTO-VORSCHLAG ===
Bestes k nach Silhouette: k = 2 (Silhouette = 0.1969)

=== CLUSTERGRÖSSEN ===
   Cluster  Anzahl  Anteil
0        0    1109  0.4951
1        1    1131  0.5049

=== CLUSTERPROFILE (MEAN, Original-Skalen) ===
         Kunde_seit_Tagen  Kunde_seit_Monaten  Flag_Neukunde      Alter  \
Cluster                                                                   
0              954.076646           31.342860            0.0  49.742110   
1              913.142352           29.998106            0.0  44.503095   

         Haushaltsgroesse  Flag_Kinder  Flag_Teenager  Recency_Tage  \
Cluster                                                               
0                1.543733     0

In [11]:
# ============================================================
# Zelle 6 — Cluster-Interpretation: Top-Unterschiede + Profil-Report
# ============================================================
"""
Analyst-Notiz:
KMeans liefert uns Clusterlabels, aber der eigentliche Mehrwert entsteht erst in der Interpretation:
- Was unterscheidet die Cluster inhaltlich?
- Welche Variablen treiben die Trennung?
- Wie können wir die Segmente fachlich benennen und nutzbar machen?

In dieser Zelle:
1) Wir erstellen ein Cluster-Profil (Mean/Median) auf Original-Skalen.
2) Wir berechnen Differenzen zwischen Clustern (bei k=2) und sortieren die stärksten Treiber.
3) Wir ergänzen einfache Kennzahlen: Clustergröße, Anteil, sowie „Indexwerte“ (Clustermean / Gesamtmean).
"""

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
if "df_clustered" not in globals():
    raise RuntimeError("df_clustered fehlt. Bitte zuerst Zelle 5 ausführen.")
if "features_cluster" not in globals():
    raise RuntimeError("features_cluster fehlt. Bitte zuerst Zelle 4 ausführen.")

# Wir interpretieren auf den Clusterfeatures, die wir fürs Clustering genutzt haben.
cols_profile = features_cluster.copy()

# Sicherheit: nur Spalten, die wirklich existieren
cols_profile = [c for c in cols_profile if c in df_clustered.columns]

# ------------------------------------------------------------
# 1) Basis: Clustergrößen & zentrale Statistiken
# ------------------------------------------------------------
cluster_counts = df_clustered["Cluster"].value_counts().sort_index()
cluster_share = (cluster_counts / len(df_clustered)).round(4)

overview = pd.DataFrame({
    "Cluster": cluster_counts.index,
    "Anzahl": cluster_counts.values,
    "Anteil": cluster_share.values
}).sort_values("Cluster")

print("=== CLUSTER OVERVIEW ===")
print(overview)

# Mean/Median Profile
profile_mean = df_clustered.groupby("Cluster")[cols_profile].mean(numeric_only=True)
profile_median = df_clustered.groupby("Cluster")[cols_profile].median(numeric_only=True)

print("\n=== CLUSTER PROFILE (MEAN) ===")
with pd.option_context("display.max_columns", 120):
    print(profile_mean)

print("\n=== CLUSTER PROFILE (MEDIAN) ===")
with pd.option_context("display.max_columns", 120):
    print(profile_median)

# ------------------------------------------------------------
# 2) Treiberanalyse (für k=2): Differenzen & Effektstärke (Cohen's d)
# ------------------------------------------------------------
unique_clusters = sorted(df_clustered["Cluster"].unique().tolist())

if len(unique_clusters) == 2:
    c0, c1 = unique_clusters[0], unique_clusters[1]

    df0 = df_clustered[df_clustered["Cluster"] == c0][cols_profile]
    df1 = df_clustered[df_clustered["Cluster"] == c1][cols_profile]

    # Differenzen auf Mean-Basis
    mean0 = df0.mean(numeric_only=True)
    mean1 = df1.mean(numeric_only=True)
    diff = (mean0 - mean1)

    # Cohen's d als standardisierte Effektstärke
    std0 = df0.std(numeric_only=True)
    std1 = df1.std(numeric_only=True)
    pooled_std = np.sqrt((std0**2 + std1**2) / 2).replace(0, np.nan)
    cohens_d = diff / pooled_std

    # Indexwerte: Clustermean / Gesamtmean
    overall_mean = df_clustered[cols_profile].mean(numeric_only=True).replace(0, np.nan)
    index0 = (mean0 / overall_mean)
    index1 = (mean1 / overall_mean)

    driver_df = pd.DataFrame({
        "mean_cluster_" + str(c0): mean0,
        "mean_cluster_" + str(c1): mean1,
        "diff_" + str(c0) + "_minus_" + str(c1): diff,
        "cohens_d": cohens_d,
        "index_cluster_" + str(c0): index0,
        "index_cluster_" + str(c1): index1,
    })

    # Sortierung: absolute Effektstärke
    driver_df = driver_df.sort_values("cohens_d", key=lambda s: s.abs(), ascending=False)

    print("\n=== TOP TREIBER (nach |Cohen's d|) ===")
    with pd.option_context("display.max_rows", 30, "display.max_columns", 120):
        print(driver_df.head(20))

    # Kurzer, interpretierbarer Fokus-Report: Top 8 Treiber
    top = driver_df.head(8).copy()
    top["richtung"] = np.where(top["diff_" + str(c0) + "_minus_" + str(c1)] > 0, f"Cluster {c0} höher", f"Cluster {c1} höher")

    print("\n=== INTERPRETATIONS-KURZLISTE (Top 8) ===")
    with pd.option_context("display.max_columns", 120):
        print(top[[
            "richtung",
            "mean_cluster_" + str(c0),
            "mean_cluster_" + str(c1),
            "diff_" + str(c0) + "_minus_" + str(c1),
            "cohens_d"
        ]])

else:
    print("\nHinweis: Treiberanalyse ist hier für k=2 ausgelegt. Für k>2 machen wir Ranking pro Cluster gegen Gesamtmittel.")

# ------------------------------------------------------------
# 3) Segmentnamen (neutral) vorschlagen — datenbasiert anhand Monetary/Frequency
# ------------------------------------------------------------
if "Monetary_Ausgaben_Gesamt" in profile_mean.columns and "Frequency_Kaeufe_Gesamt" in profile_mean.columns:
    seg = profile_mean[["Monetary_Ausgaben_Gesamt", "Frequency_Kaeufe_Gesamt"]].copy()
    seg["Segmentlabel_Vorschlag"] = ""

    # Simple Heuristik: High/Low via Median der Clustermeans
    m_med = seg["Monetary_Ausgaben_Gesamt"].median()
    f_med = seg["Frequency_Kaeufe_Gesamt"].median()

    for cl in seg.index:
        m = seg.loc[cl, "Monetary_Ausgaben_Gesamt"]
        f = seg.loc[cl, "Frequency_Kaeufe_Gesamt"]

        if (m >= m_med) and (f >= f_med):
            label = "High-Value / Frequent"
        elif (m >= m_med) and (f < f_med):
            label = "High-Value / Infrequent"
        elif (m < m_med) and (f >= f_med):
            label = "Low-Value / Frequent"
        else:
            label = "Low-Value / Infrequent"

        seg.loc[cl, "Segmentlabel_Vorschlag"] = label

    print("\n=== SEGMENTLABEL (VORSCHLAG) ===")
    print(seg)
else:
    print("\nSegmentlabel: Monetary/Frequency nicht gefunden (Spaltennamen prüfen).")

# Ergebnisobjekte für spätere Schritte
cluster_profile_mean = profile_mean
cluster_profile_median = profile_median
cluster_driver_table = driver_df if len(unique_clusters) == 2 else None

print("\nObjekte bereitgestellt:")
print("- cluster_profile_mean, cluster_profile_median")
print("- cluster_driver_table (nur bei k=2)")
print("- overview (Clustergrößen)")


=== CLUSTER OVERVIEW ===
   Cluster  Anzahl  Anteil
0        0    1109  0.4951
1        1    1131  0.5049

=== CLUSTER PROFILE (MEAN) ===
         Kunde_seit_Tagen  Kunde_seit_Monaten  Flag_Neukunde      Alter  \
Cluster                                                                   
0              954.076646           31.342860            0.0  49.745487   
1              913.142352           29.998106            0.0  44.500443   

         Haushaltsgroesse  Flag_Kinder  Flag_Teenager  Recency_Tage  \
Cluster                                                               
0                1.543733     0.045987       0.473399     49.775473   
1                2.349248     0.792219       0.492485     48.456233   

         Frequency_Kaeufe_Gesamt  Monetary_Ausgaben_Gesamt  \
Cluster                                                      
0                      18.396754               2153.440938   
1                       6.791335                288.072502   

         Warenkorb_Durchsch

In [12]:
# ============================================================
# Zelle 7 — Segment-Report: Labels + KPI-Übersicht + Quantile
# ============================================================
"""
Analyst-Notiz:
Ziel dieser Zelle ist, die Clusterergebnisse "berichtsfähig" zu machen:

1) Segmentlabels vergeben (neutral, datenbasiert über Monetary/Frequency).
2) KPI-Übersicht pro Segment:
   - Mean pro Cluster (Original-Skalen)
   - Indexwerte (Clustermean / Gesamtmean) -> leicht interpretierbar
3) Quantile-Report (q10/Median/q90) für zentrale KPIs, um Streuung sichtbar zu machen.

Ergebnis:
- df_clustered bekommt eine neue Spalte "Segment"
- segment_overview, segment_kpi_mean, segment_kpi_index, segment_quantiles stehen für den nächsten Schritt bereit
"""

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
if "df_clustered" not in globals():
    raise RuntimeError("df_clustered fehlt. Bitte zuerst Zelle 5 ausführen.")

# ------------------------------------------------------------
# 1) Segmentlabels (datenbasiert über Monetary + Frequency)
# ------------------------------------------------------------
need_cols = ["Monetary_Ausgaben_Gesamt", "Frequency_Kaeufe_Gesamt"]
for c in need_cols:
    if c not in df_clustered.columns:
        raise KeyError(f"Spalte fehlt: {c}. Bitte Zelle 3/5 prüfen.")

seg_means = df_clustered.groupby("Cluster")[need_cols].mean(numeric_only=True)

m_med = seg_means["Monetary_Ausgaben_Gesamt"].median()
f_med = seg_means["Frequency_Kaeufe_Gesamt"].median()

segment_labels = {}
for cl in seg_means.index:
    m = seg_means.loc[cl, "Monetary_Ausgaben_Gesamt"]
    f = seg_means.loc[cl, "Frequency_Kaeufe_Gesamt"]

    # Heuristik: High/Low relativ zu den Cluster-Medians
    if (m >= m_med) and (f >= f_med):
        label = "Segment A: High-Value / Frequent"
    elif (m >= m_med) and (f < f_med):
        label = "Segment B: High-Value / Infrequent"
    elif (m < m_med) and (f >= f_med):
        label = "Segment C: Low-Value / Frequent"
    else:
        label = "Segment D: Low-Value / Infrequent"

    segment_labels[int(cl)] = label

df_clustered = df_clustered.copy()
df_clustered["Segment"] = df_clustered["Cluster"].map(segment_labels)

print("=== SEGMENTLABELS ===")
for k in sorted(segment_labels.keys()):
    print(f"Cluster {k}: {segment_labels[k]}")

# ------------------------------------------------------------
# 2) Segment-Overview (Größe/Anteil)
# ------------------------------------------------------------
counts = df_clustered["Cluster"].value_counts().sort_index()
share = (counts / len(df_clustered)).round(4)

segment_overview = pd.DataFrame({
    "Cluster": counts.index,
    "Anzahl": counts.values,
    "Anteil": share.values,
    "Segment": [segment_labels[int(x)] for x in counts.index]
}).sort_values("Cluster")

print("\n=== SEGMENT OVERVIEW ===")
print(segment_overview)

# ------------------------------------------------------------
# 3) KPI-Report (Mean + Index)
# ------------------------------------------------------------
kpis = [
    "Kunde_seit_Tagen", "Alter", "Haushaltsgroesse",
    "Recency_Tage", "Frequency_Kaeufe_Gesamt", "Monetary_Ausgaben_Gesamt",
    "Warenkorb_Durchschnitt", "Rabattquote",
    "Produktmix_Diversity",
    "Anteil_Wein", "Anteil_Fleisch", "Anteil_Suesses",
    "Anteil_Webkaeufe", "Anteil_Katalogkaeufe", "Anteil_Ladeneinkaeufe",
    "Web_Interaktion_Intensitaet",
    "Flag_Beschwerde"
]
kpis = [c for c in kpis if c in df_clustered.columns]

segment_kpi_mean = df_clustered.groupby("Cluster")[kpis].mean(numeric_only=True)

overall_mean = df_clustered[kpis].mean(numeric_only=True).replace(0, np.nan)
segment_kpi_index = segment_kpi_mean.div(overall_mean, axis=1)

print("\n=== KPI MEAN (pro Cluster) ===")
with pd.option_context("display.max_columns", 120):
    print(segment_kpi_mean)

print("\n=== KPI INDEX (Clustermean / Gesamtmean) ===")
with pd.option_context("display.max_columns", 120):
    print(segment_kpi_index.round(3))

# ------------------------------------------------------------
# 4) Quantile-Report (Streuung sichtbar machen)
# ------------------------------------------------------------
quant_cols = [c for c in ["Monetary_Ausgaben_Gesamt", "Frequency_Kaeufe_Gesamt", "Warenkorb_Durchschnitt", "Rabattquote", "Recency_Tage"] if c in df_clustered.columns]

rows = []
for cl, sub in df_clustered.groupby("Cluster"):
    for col in quant_cols:
        s = pd.to_numeric(sub[col], errors="coerce").dropna()
        rows.append({
            "Cluster": int(cl),
            "Segment": segment_labels[int(cl)],
            "Feature": col,
            "q10": s.quantile(0.10),
            "median": s.quantile(0.50),
            "q90": s.quantile(0.90),
        })

segment_quantiles = pd.DataFrame(rows)

print("\n=== QUANTILE REPORT (q10 / median / q90) ===")
with pd.option_context("display.max_rows", 200, "display.max_columns", 20):
    print(segment_quantiles)

print("\nObjekte bereitgestellt:")
print("- df_clustered (jetzt mit Segment)")
print("- segment_overview, segment_kpi_mean, segment_kpi_index, segment_quantiles")


=== SEGMENTLABELS ===
Cluster 0: Segment A: High-Value / Frequent
Cluster 1: Segment D: Low-Value / Infrequent

=== SEGMENT OVERVIEW ===
   Cluster  Anzahl  Anteil                            Segment
0        0    1109  0.4951   Segment A: High-Value / Frequent
1        1    1131  0.5049  Segment D: Low-Value / Infrequent

=== KPI MEAN (pro Cluster) ===
         Kunde_seit_Tagen      Alter  Haushaltsgroesse  Recency_Tage  \
Cluster                                                                
0              954.076646  49.745487          1.543733     49.775473   
1              913.142352  44.500443          2.349248     48.456233   

         Frequency_Kaeufe_Gesamt  Monetary_Ausgaben_Gesamt  \
Cluster                                                      
0                      18.396754               2153.440938   
1                       6.791335                288.072502   

         Warenkorb_Durchschnitt  Rabattquote  Produktmix_Diversity  \
Cluster                              

In [13]:
# ============================================================
# Zelle 8 — Supervised Learning (Klassifikation): Kampagnenantwort vorhersagen
# ============================================================
"""
Analyst-Notiz:
Nach Segmentierung (Clustering) gehen wir jetzt in die Vorhersage (Supervised Learning).

Ziel:
- Wir sagen voraus, ob ein Kunde auf die letzte Kampagne reagiert (1) oder nicht (0).

Wichtige Punkte:
1) Leakage vermeiden:
   - Kampagnenhistorie (AcceptedCmp1..5 / Kampagne_*_Akzeptiert) kann helfen,
     ist aber potenziell „zu nah“ an der Zielvariable. Deshalb testen wir zwei Varianten:
       A) Nur SAFE-Features (ohne Kampagnenhistorie)
       B) Features + Kampagnenhistorie (optional, später kritisch bewerten)

2) Baseline-Modell:
   - Logistische Regression (gut erklärbar, solide Baseline)
   - Metriken: ROC-AUC, Accuracy, Precision, Recall, F1, Confusion Matrix

Outputs:
- model_safe, model_with_campaign
- metrics_table (Vergleich)
"""

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
if "df" not in globals():
    raise RuntimeError("df fehlt. Bitte Zelle 1 ausführen.")
if "X_model_safe" not in globals():
    raise RuntimeError("X_model_safe fehlt. Bitte Zelle 4 ausführen.")
if "X_model_with_campaign" not in globals():
    raise RuntimeError("X_model_with_campaign fehlt. Bitte Zelle 4 ausführen.")

# ------------------------------------------------------------
# 1) Zielvariable finden (konkret nach CSV)
#    Wir prüfen typische Namen und nehmen den ersten Treffer.
# ------------------------------------------------------------
target_candidates = [
    "Antwort_Letzte_Kampagne",
    "Response",
    "Letzte_Kampagne_Antwort",
    "Kampagne_Letzte_Antwort",
]

target_col = None
for c in target_candidates:
    if c in df.columns:
        target_col = c
        break

if target_col is None:
    raise KeyError(
        "Keine Zielvariable gefunden. Erwartet eine dieser Spalten:\n- "
        + "\n- ".join(target_candidates)
        + "\n\nVorhandene Spalten sind:\n- "
        + "\n- ".join(df.columns.astype(str).tolist())
    )

y = pd.to_numeric(df[target_col], errors="coerce").fillna(0).astype(int)

print("=== ZIELVARIABLE ===")
print("Spalte:", target_col)
print("Verteilung (0/1):")
print(y.value_counts(dropna=False).sort_index())

# ------------------------------------------------------------
# 2) Train/Test Split (stratifiziert)
# ------------------------------------------------------------
X_safe = X_model_safe.copy()
X_camp = X_model_with_campaign.copy()

X_train_s, X_test_s, y_train, y_test = train_test_split(
    X_safe, y, test_size=0.25, random_state=42, stratify=y
)

X_train_c, X_test_c, _, _ = train_test_split(
    X_camp, y, test_size=0.25, random_state=42, stratify=y
)

# ------------------------------------------------------------
# 3) Pipeline: Imputer + Scaler + Logistic Regression
# ------------------------------------------------------------
pipe_lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])

# A) SAFE
model_safe = pipe_lr.fit(X_train_s, y_train)
proba_safe = model_safe.predict_proba(X_test_s)[:, 1]
pred_safe = (proba_safe >= 0.5).astype(int)

# B) WITH CAMPAIGN HISTORY
model_with_campaign = pipe_lr.fit(X_train_c, y_train)
proba_camp = model_with_campaign.predict_proba(X_test_c)[:, 1]
pred_camp = (proba_camp >= 0.5).astype(int)

# ------------------------------------------------------------
# 4) Metriken berechnen
# ------------------------------------------------------------
def metric_row(name, y_true, proba, pred):
    return {
        "Modell": name,
        "ROC_AUC": roc_auc_score(y_true, proba),
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ConfusionMatrix": confusion_matrix(y_true, pred).tolist(),
    }

metrics = [
    metric_row("LogReg SAFE (ohne Kampagnenhistorie)", y_test, proba_safe, pred_safe),
    metric_row("LogReg + Kampagnenhistorie (optional)", y_test, proba_camp, pred_camp),
]

metrics_table = pd.DataFrame(metrics)

print("\n=== METRIKEN-VERGLEICH ===")
print(metrics_table[["Modell", "ROC_AUC", "Accuracy", "Precision", "Recall", "F1"]])

print("\n=== CONFUSION MATRICES (als Liste) ===")
for _, r in metrics_table.iterrows():
    print("\n", r["Modell"])
    print(r["ConfusionMatrix"])

# ------------------------------------------------------------
# 5) Ergebnisobjekte für nächste Schritte
# ------------------------------------------------------------
print("\nObjekte bereitgestellt:")
print("- model_safe, model_with_campaign")
print("- metrics_table")
print("- (y_test, proba_safe, pred_safe) und (proba_camp, pred_camp) für Threshold-Tuning in Zelle 9")

y_test_holdout = y_test.copy()
proba_safe_holdout = proba_safe.copy()
pred_safe_holdout = pred_safe.copy()
proba_camp_holdout = proba_camp.copy()
pred_camp_holdout = pred_camp.copy()


=== ZIELVARIABLE ===
Spalte: Antwort_Letzte_Kampagne
Verteilung (0/1):
Antwort_Letzte_Kampagne
0    1906
1     334
Name: count, dtype: int64

=== METRIKEN-VERGLEICH ===
                                  Modell   ROC_AUC  Accuracy  Precision  \
0   LogReg SAFE (ohne Kampagnenhistorie)  0.857152  0.785714   0.390533   
1  LogReg + Kampagnenhistorie (optional)  0.886047  0.825000   0.443609   

     Recall        F1  
0  0.795181  0.523810  
1  0.710843  0.546296  

=== CONFUSION MATRICES (als Liste) ===

 LogReg SAFE (ohne Kampagnenhistorie)
[[374, 103], [17, 66]]

 LogReg + Kampagnenhistorie (optional)
[[403, 74], [24, 59]]

Objekte bereitgestellt:
- model_safe, model_with_campaign
- metrics_table
- (y_test, proba_safe, pred_safe) und (proba_camp, pred_camp) für Threshold-Tuning in Zelle 9


In [14]:
# ============================================================
# Zelle 9 — Threshold-Tuning (Schwellenanalyse) + Kampagnen-Decision
# ============================================================
"""
Analyst-Notiz:
Das Modell liefert Wahrscheinlichkeiten (proba). Die Standard-Schwelle 0.50 ist selten optimal,
weil wir ein unausgeglichenes Ziel haben (0: 1906 vs 1: 334).

In dieser Zelle:
1) Wir testen viele Schwellen (z. B. 0.05 bis 0.95).
2) Wir berechnen pro Schwelle: Precision, Recall, F1, Accuracy + Confusion Matrix Werte.
3) Wir identifizieren:
   - beste Schwelle nach F1 (balanciert Precision/Recall)
   - beste Schwelle nach Precision (wenn Budget eng)
   - beste Schwelle nach Recall (wenn wir möglichst viele Reagierer erwischen wollen)
4) Wir vergleichen SAFE vs. optional (mit Kampagnenhistorie).

Outputs:
- tab_safe, tab_camp (Schwellentabellen)
- best_safe_f1, best_camp_f1 (empfohlene Schwellen nach F1)
"""

import numpy as np
import pandas as pd

from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
required = ["y_test_holdout", "proba_safe_holdout", "proba_camp_holdout"]
for r in required:
    if r not in globals():
        raise RuntimeError(f"{r} fehlt. Bitte zuerst Zelle 8 ausführen.")

y_true = y_test_holdout.copy()

# ------------------------------------------------------------
# 1) Schwellenanalyse-Funktion
# ------------------------------------------------------------
def schwellen_auswertung(y_true, proba, name="Modell", schwellen=None):
    if schwellen is None:
        schwellen = np.round(np.arange(0.05, 0.96, 0.05), 2)

    rows = []
    for t in schwellen:
        y_pred = (proba >= t).astype(int)

        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        acc = accuracy_score(y_true, y_pred)

        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel()

        rows.append({
            "modell": name,
            "threshold": float(t),
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "accuracy": acc,
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
            "pred_pos": int(tp + fp),
            "pred_neg": int(tn + fn),
        })

    tab = pd.DataFrame(rows)

    # Rankings
    best_f1 = tab.sort_values(["f1", "precision", "recall"], ascending=False).iloc[0]
    best_precision = tab.sort_values(["precision", "recall"], ascending=False).iloc[0]
    best_recall = tab.sort_values(["recall", "precision"], ascending=False).iloc[0]

    return tab, best_f1, best_precision, best_recall

# ------------------------------------------------------------
# 2) Auswertung SAFE vs. optional
# ------------------------------------------------------------
schwellen_grid = np.round(np.arange(0.05, 0.96, 0.05), 2)

tab_safe, best_safe_f1, best_safe_prec, best_safe_rec = schwellen_auswertung(
    y_true, proba_safe_holdout, name="LogReg SAFE", schwellen=schwellen_grid
)

tab_camp, best_camp_f1, best_camp_prec, best_camp_rec = schwellen_auswertung(
    y_true, proba_camp_holdout, name="LogReg + Kampagnenhistorie", schwellen=schwellen_grid
)

print("=== SCHWELLEN-TABELLE (SAFE) — Top 10 nach F1 ===")
print(tab_safe.sort_values(["f1", "precision", "recall"], ascending=False).head(10))

print("\n=== SCHWELLEN-TABELLE (CAMP) — Top 10 nach F1 ===")
print(tab_camp.sort_values(["f1", "precision", "recall"], ascending=False).head(10))

# ------------------------------------------------------------
# 3) Empfehlungen (3 Perspektiven) — jeweils SAFE und CAMP
# ------------------------------------------------------------
def print_best(best_row, title):
    print(title)
    print(f"threshold={best_row['threshold']:.2f} | precision={best_row['precision']:.3f} | recall={best_row['recall']:.3f} | f1={best_row['f1']:.3f} | acc={best_row['accuracy']:.3f}")
    print(f"Confusion: TN={best_row['tn']} FP={best_row['fp']} FN={best_row['fn']} TP={best_row['tp']} | pred_pos={best_row['pred_pos']}")

print("\n=== EMPFEHLUNGEN — SAFE ===")
print_best(best_safe_f1,  "Best nach F1 (Balance):")
print_best(best_safe_prec, "Best nach Precision (Budget eng):")
print_best(best_safe_rec,  "Best nach Recall (max Treffer):")

print("\n=== EMPFEHLUNGEN — CAMP (optional) ===")
print_best(best_camp_f1,  "Best nach F1 (Balance):")
print_best(best_camp_prec, "Best nach Precision (Budget eng):")
print_best(best_camp_rec,  "Best nach Recall (max Treffer):")

# ------------------------------------------------------------
# 4) Eine “Arbeits-Schwelle” festlegen (Default: bestes F1 je Modell)
# ------------------------------------------------------------
threshold_safe_empfohlen = float(best_safe_f1["threshold"])
threshold_camp_empfohlen = float(best_camp_f1["threshold"])

print("\n=== ARBEITSSCHWELLEN (Default nach F1) ===")
print("SAFE threshold:", threshold_safe_empfohlen)
print("CAMP threshold:", threshold_camp_empfohlen)

# ------------------------------------------------------------
# 5) Ergebnisobjekte für die nächste Zelle (z. B. Feature-Importance / Koeffizienten)
# ------------------------------------------------------------
best_thresholds = {
    "safe_f1": best_safe_f1.to_dict(),
    "safe_precision": best_safe_prec.to_dict(),
    "safe_recall": best_safe_rec.to_dict(),
    "camp_f1": best_camp_f1.to_dict(),
    "camp_precision": best_camp_prec.to_dict(),
    "camp_recall": best_camp_rec.to_dict(),
}

print("\nObjekte bereitgestellt:")
print("- tab_safe, tab_camp")
print("- best_thresholds")
print("- threshold_safe_empfohlen, threshold_camp_empfohlen")


=== SCHWELLEN-TABELLE (SAFE) — Top 10 nach F1 ===
         modell  threshold  precision    recall        f1  accuracy   tn   fp  \
9   LogReg SAFE       0.50   0.390533  0.795181  0.523810  0.785714  374  103   
11  LogReg SAFE       0.60   0.423077  0.662651  0.516432  0.816071  402   75   
12  LogReg SAFE       0.65   0.453704  0.590361  0.513089  0.833929  418   59   
8   LogReg SAFE       0.45   0.368984  0.831325  0.511111  0.764286  359  118   
10  LogReg SAFE       0.55   0.400000  0.698795  0.508772  0.800000  390   87   
7   LogReg SAFE       0.40   0.346341  0.855422  0.493056  0.739286  343  134   
13  LogReg SAFE       0.70   0.471910  0.506024  0.488372  0.842857  430   47   
6   LogReg SAFE       0.35   0.316456  0.903614  0.468750  0.696429  315  162   
14  LogReg SAFE       0.75   0.500000  0.409639  0.450331  0.851786  443   34   
5   LogReg SAFE       0.30   0.291188  0.915663  0.441860  0.657143  292  185   

    fn  tp  pred_pos  pred_neg  
9   17  66       169     

In [16]:
# ============================================================
# Zelle 10 — Modell-Interpretation (LogReg): Koeffizienten + Treiberreport (robust)
# ============================================================
"""
Analyst-Notiz:
Wir machen das LogReg-Modell erklärbar, indem wir die Koeffizienten sauber aus der Pipeline ziehen.
Der Fehler „All arrays must be of the same length“ kommt typischerweise daher, dass die Anzahl
der Koeffizienten (n_features) nicht zu der Feature-Liste passt (z. B. durch abweichende Spaltenliste
oder ein anderes Fit-Input-Layout).

Diese Version ist robust:
- Feature-Namen bevorzugt aus pipeline.feature_names_in_ (falls vorhanden)
- Fallback: Spalten aus X_model_* (DataFrame)
- Sicherheitscheck: Längenabgleich (min length), plus Debug-Print der Längen
- Ausgabe: Top positive/negative Treiber, sowie Odds Ratios (exp(coef))
"""

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
if "model_safe" not in globals():
    raise RuntimeError("model_safe fehlt. Bitte zuerst Zelle 8 ausführen.")
if "model_with_campaign" not in globals():
    raise RuntimeError("model_with_campaign fehlt. Bitte zuerst Zelle 8 ausführen.")
if "X_model_safe" not in globals():
    raise RuntimeError("X_model_safe fehlt. Bitte zuerst Zelle 4 ausführen.")
if "X_model_with_campaign" not in globals():
    raise RuntimeError("X_model_with_campaign fehlt. Bitte zuerst Zelle 4 ausführen.")

# ------------------------------------------------------------
# 1) Hilfsfunktion: Feature-Namen & Koeffizienten robust extrahieren
# ------------------------------------------------------------
def _get_feature_names(pipeline, X_fallback: pd.DataFrame):
    """
    Holt Feature-Namen aus dem fit-Input der Pipeline, falls verfügbar.
    Fallback: X_fallback.columns
    """
    if hasattr(pipeline, "feature_names_in_"):
        return list(pipeline.feature_names_in_)
    return list(X_fallback.columns.astype(str).tolist())

def extract_logreg_table(pipeline, X_fallback: pd.DataFrame, label: str):
    """
    Erwartet Pipeline: imputer -> scaler -> model(LogisticRegression)
    Baut eine Koeffizienten-Tabelle, auch wenn Feature-Namen-Länge abweicht.
    """
    if "model" not in pipeline.named_steps:
        raise ValueError(f"[{label}] Pipeline hat keinen Step 'model'.")

    lr = pipeline.named_steps["model"]
    coefs = np.asarray(lr.coef_).ravel()  # (n_features,)

    feature_names = _get_feature_names(pipeline, X_fallback)
    feature_names = list(feature_names)

    # Debug-Längen
    n_coef = int(len(coefs))
    n_feat = int(len(feature_names))
    print(f"\n[{label}] Feature-Namen: {n_feat} | Koeffizienten: {n_coef}")

    # Falls es nicht passt: auf gemeinsame Länge schneiden (robuste Notlösung)
    if n_feat != n_coef:
        n = min(n_feat, n_coef)
        print(f"[{label}] WARNUNG: Länge passt nicht. Verwende min-Länge = {n}.")
        feature_names = feature_names[:n]
        coefs = coefs[:n]

    dfc = pd.DataFrame({
        "feature": feature_names,
        "coef": coefs
    })

    dfc["abs_coef"] = dfc["coef"].abs()
    dfc["odds_ratio"] = np.exp(dfc["coef"])  # exp(coef): >1 erhöht Odds, <1 senkt Odds
    dfc["richtung"] = np.where(
        dfc["coef"] >= 0,
        "positiv (Antwort wahrscheinlicher)",
        "negativ (Antwort unwahrscheinlicher)"
    )

    dfc = dfc.sort_values("abs_coef", ascending=False).reset_index(drop=True)
    return dfc

# ------------------------------------------------------------
# 2) Tabellen erstellen (SAFE / CAMP)
# ------------------------------------------------------------
coef_table_safe = extract_logreg_table(model_safe, X_model_safe, label="SAFE")
coef_table_camp = extract_logreg_table(model_with_campaign, X_model_with_campaign, label="CAMP")

# ------------------------------------------------------------
# 3) Top-Treiber ausgeben
# ------------------------------------------------------------
def print_top_blocks(dfc: pd.DataFrame, title: str, n: int = 10):
    pos = dfc[dfc["coef"] > 0].head(n)
    neg = dfc[dfc["coef"] < 0].head(n)

    print(f"\n=== {title} ===")

    print("\nTop Positiv-Treiber:")
    with pd.option_context("display.max_rows", n, "display.max_columns", 10):
        print(pos[["feature", "coef", "odds_ratio", "abs_coef"]])

    print("\nTop Negativ-Treiber:")
    with pd.option_context("display.max_rows", n, "display.max_columns", 10):
        print(neg[["feature", "coef", "odds_ratio", "abs_coef"]])

print_top_blocks(coef_table_safe, "LogReg SAFE (ohne Kampagnenhistorie)", n=10)
print_top_blocks(coef_table_camp, "LogReg + Kampagnenhistorie (optional)", n=10)

print("\n=== TREIBERREPORT (Top 15 |coef|) — SAFE ===")
with pd.option_context("display.max_rows", 20, "display.max_columns", 10):
    print(coef_table_safe.head(15)[["feature", "coef", "richtung", "odds_ratio", "abs_coef"]])

print("\n=== TREIBERREPORT (Top 15 |coef|) — CAMP ===")
with pd.option_context("display.max_rows", 20, "display.max_columns", 10):
    print(coef_table_camp.head(15)[["feature", "coef", "richtung", "odds_ratio", "abs_coef"]])

print("\nObjekte bereitgestellt:")
print("- coef_table_safe, coef_table_camp")



[SAFE] Feature-Namen: 23 | Koeffizienten: 23

[CAMP] Feature-Namen: 23 | Koeffizienten: 23

=== LogReg SAFE (ohne Kampagnenhistorie) ===

Top Positiv-Treiber:
                     feature      coef  odds_ratio  abs_coef
0   Kampagnen_Historie_Summe  0.966828    2.629591  0.966828
3         Kunde_seit_Monaten  0.524999    1.690457  0.524999
4           Kunde_seit_Tagen  0.524999    1.690457  0.524999
6   Monetary_Ausgaben_Gesamt  0.485083    1.624310  0.485083
7       Anteil_Katalogkaeufe  0.465653    1.593054  0.465653
8             Anteil_Fleisch  0.446625    1.563028  0.446625
9           Anteil_Webkaeufe  0.385246    1.469976  0.385246
10               Rabattquote  0.313888    1.368737  0.313888
13                     Alter  0.252378    1.287083  0.252378
15       Flag_Zuvor_Reagiert  0.182802    1.200576  0.182802

Top Negativ-Treiber:
                        feature      coef  odds_ratio  abs_coef
1                  Recency_Tage -0.892295    0.409714  0.892295
2       Frequency_K

In [17]:
# ============================================================
# Zelle 11 — Leakage-Check + Finales Scoring (Target-Liste) + Segment-Performance
# ============================================================
"""
Analyst-Notiz:
Bevor wir aus dem Modell echte Kampagnen-Entscheidungen ableiten, machen wir zwei Dinge:

1) Leakage-Check:
   Das "SAFE"-Modell soll OHNE Kampagnenhistorie laufen.
   In deinem Output tauchen aber Kampagnen_Historie_Summe / Flag_Zuvor_Reagiert im SAFE-Report auf.
   Falls diese Spalten in X_model_safe enthalten sind, bauen wir ein wirkliches SAFE-Set (ohne diese Features)
   und trainieren die SAFE-Baseline einmal sauber neu.

2) Finales Scoring:
   - Wir nutzen das gewählte Modell und berechnen pro Kunde eine Antwortwahrscheinlichkeit.
   - Wir wenden eine Schwelle an (Default: threshold_safe_empfohlen aus Zelle 9, sonst 0.50).
   - Wir erstellen:
       a) Top-Target-Liste (höchste Wahrscheinlichkeiten)
       b) Segment-Report (Predicted Positives, Mittelwert pro Segment, echte Response-Rate falls vorhanden)
"""

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
req_vars = ["df", "df_clustered", "X_model_safe", "model_safe"]
for v in req_vars:
    if v not in globals():
        raise RuntimeError(f"{v} fehlt. Bitte vorherige Zellen ausführen.")

# Zielvariable (wie in Zelle 8)
if "Antwort_Letzte_Kampagne" not in df.columns:
    raise KeyError("Zielspalte 'Antwort_Letzte_Kampagne' fehlt in df. Bitte Zelle 8 prüfen.")
y = pd.to_numeric(df["Antwort_Letzte_Kampagne"], errors="coerce").fillna(0).astype(int)

# ------------------------------------------------------------
# 1) Leakage-Check: Kampagnenhistorie darf nicht im SAFE-Set sein
# ------------------------------------------------------------
campaign_optional_cols = [c for c in ["Kampagnen_Historie_Summe", "Flag_Zuvor_Reagiert"] if c in X_model_safe.columns]
needs_safe_fix = len(campaign_optional_cols) > 0

print("=== LEAKAGE-CHECK (SAFE) ===")
if needs_safe_fix:
    print("SAFE enthält kampagnennahe Features und wird bereinigt:", campaign_optional_cols)
else:
    print("SAFE ist sauber (keine kampagnennahe Features im X_model_safe).")

# Falls nötig: echtes SAFE bauen und neu trainieren (vergleichbar zur Zelle 8)
if needs_safe_fix:
    X_safe_clean = X_model_safe.drop(columns=campaign_optional_cols).copy()
else:
    X_safe_clean = X_model_safe.copy()

# Pipeline wie Zelle 8
pipe_lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])

# Holdout-Split (wie vorher)
X_train, X_test, y_train, y_test = train_test_split(
    X_safe_clean, y, test_size=0.25, random_state=42, stratify=y
)

model_safe_final = pipe_lr.fit(X_train, y_train)
proba_test = model_safe_final.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.5).astype(int)

print("\n=== SAFE FINAL (Holdout-Metriken, Threshold=0.50) ===")
print("ROC_AUC:", round(roc_auc_score(y_test, proba_test), 6))
print("Accuracy:", round(accuracy_score(y_test, pred_test), 6))
print("Precision:", round(precision_score(y_test, pred_test, zero_division=0), 6))
print("Recall:", round(recall_score(y_test, pred_test, zero_division=0), 6))
print("F1:", round(f1_score(y_test, pred_test, zero_division=0), 6))
print("ConfusionMatrix:", confusion_matrix(y_test, pred_test).tolist())

# ------------------------------------------------------------
# 2) Schwelle festlegen (wenn aus Zelle 9 vorhanden, sonst 0.50)
# ------------------------------------------------------------
if "threshold_safe_empfohlen" in globals():
    threshold_final = float(threshold_safe_empfohlen)
else:
    threshold_final = 0.50

print("\n=== SCHWELLE (Final) ===")
print("threshold_final:", threshold_final)

# ------------------------------------------------------------
# 3) Scoring auf allen Kunden (finales Modell auf Voll-Daten fitten)
#    (nachdem die Holdout-Leistung geprüft ist)
# ------------------------------------------------------------
model_safe_final_full = pipe_lr.fit(X_safe_clean, y)
proba_all = model_safe_final_full.predict_proba(X_safe_clean)[:, 1]
pred_all = (proba_all >= threshold_final).astype(int)

# Scoring-Tabelle
df_scoring = pd.DataFrame({
    "ID": df_clustered["ID"] if "ID" in df_clustered.columns else df["ID"],
    "Cluster": df_clustered["Cluster"] if "Cluster" in df_clustered.columns else np.nan,
    "Segment": df_clustered["Segment"] if "Segment" in df_clustered.columns else np.nan,
    "proba_antwort": proba_all,
    "pred_antwort": pred_all,
    "ist_antwort": y.values
})

df_scoring = df_scoring.sort_values("proba_antwort", ascending=False).reset_index(drop=True)

print("\n=== TOP 25 TARGETS (höchste Wahrscheinlichkeit) ===")
with pd.option_context("display.max_rows", 25, "display.max_columns", 10):
    print(df_scoring.head(25))

# ------------------------------------------------------------
# 4) Segment-Performance (Predicted Positives, echte Response-Rate)
# ------------------------------------------------------------
seg_report = df_scoring.groupby(["Segment"], dropna=False).agg(
    kunden=("ID", "count"),
    mean_proba=("proba_antwort", "mean"),
    pred_pos=("pred_antwort", "sum"),
    actual_rate=("ist_antwort", "mean"),
).reset_index()

seg_report["pred_pos_rate"] = seg_report["pred_pos"] / seg_report["kunden"]
seg_report = seg_report.sort_values("mean_proba", ascending=False)

print("\n=== SEGMENT REPORT ===")
with pd.option_context("display.max_rows", 50, "display.max_columns", 20):
    print(seg_report)

# ------------------------------------------------------------
# 5) Ergebnisobjekte für nächste Schritte
# ------------------------------------------------------------
print("\nObjekte bereitgestellt:")
print("- model_safe_final (Holdout geprüft)")
print("- model_safe_final_full (für Scoring)")
print("- df_scoring (Target-Liste)")
print("- seg_report (Segment-Performance)")

# optional: persistente Namen
model_final = model_safe_final_full


=== LEAKAGE-CHECK (SAFE) ===
SAFE ist sauber (keine kampagnennahe Features im X_model_safe).

=== SAFE FINAL (Holdout-Metriken, Threshold=0.50) ===
ROC_AUC: 0.857152
Accuracy: 0.785714
Precision: 0.390533
Recall: 0.795181
F1: 0.52381
ConfusionMatrix: [[374, 103], [17, 66]]

=== SCHWELLE (Final) ===
threshold_final: 0.5

=== TOP 25 TARGETS (höchste Wahrscheinlichkeit) ===
       ID  Cluster                            Segment  proba_antwort  \
0   10749        1  Segment D: Low-Value / Infrequent       0.999935   
1    1501        0   Segment A: High-Value / Frequent       0.999747   
2    5376        0   Segment A: High-Value / Frequent       0.999523   
3    4931        0   Segment A: High-Value / Frequent       0.998779   
4    4702        0   Segment A: High-Value / Frequent       0.990791   
5    7919        0   Segment A: High-Value / Frequent       0.990132   
6    9595        0   Segment A: High-Value / Frequent       0.986443   
7    1891        0   Segment A: High-Value / Frequ

In [18]:
# ============================================================
# Zelle 12 — Alles anzeigen: Scoring-Tabellen + Segment-Report + Verteilungschecks
# ============================================================
"""
Analyst-Notiz:
Bevor wir exportieren oder Entscheidungen festzurren, schauen wir uns "alles" transparent an:
1) df_scoring: die gesamte Scoring-Tabelle (kurzer Überblick + Tail)
2) Verteilung der Scores (Quantile)
3) Predicted Positives: wie viele Kunden würden wir anschreiben?
4) Segment-Report: Performance pro Segment (mean_proba, pred_pos_rate, actual_rate)
5) Top/Bottom Beispiele zur Plausibilisierung
"""

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
if "df_scoring" not in globals():
    raise RuntimeError("df_scoring fehlt. Bitte zuerst Zelle 11 ausführen.")
if "seg_report" not in globals():
    raise RuntimeError("seg_report fehlt. Bitte zuerst Zelle 11 ausführen.")

# ------------------------------------------------------------
# 1) Überblick über df_scoring
# ------------------------------------------------------------
print("=== DF_SCORING: OVERVIEW ===")
print("Shape:", df_scoring.shape)
print("Spalten:", df_scoring.columns.tolist())
print()

print("=== DF_SCORING: HEAD (Top 15) ===")
with pd.option_context("display.max_rows", 20, "display.max_columns", 20):
    print(df_scoring.head(15))
print()

print("=== DF_SCORING: TAIL (Bottom 15) ===")
with pd.option_context("display.max_rows", 20, "display.max_columns", 20):
    print(df_scoring.tail(15))
print()

# ------------------------------------------------------------
# 2) Score-Verteilung (Quantile)
# ------------------------------------------------------------
print("=== SCORE-VERTEILUNG (Quantile) ===")
q = df_scoring["proba_antwort"].quantile([0.00, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00])
print(q)
print()

# ------------------------------------------------------------
# 3) Predicted Positives / Kampagnenumfang
# ------------------------------------------------------------
pred_pos = int(df_scoring["pred_antwort"].sum())
total = int(len(df_scoring))
rate = pred_pos / total

print("=== KAMPAGNENUMFANG (nach threshold_final) ===")
print("predicted positives (anschreiben):", pred_pos)
print("gesamt:", total)
print("anteil:", round(rate, 4))
print()

# Wie viele echte Reagierer sind in pred_pos enthalten? (auf Gesamtmenge, nur Plausibilitätswert)
tp_all = int(((df_scoring["pred_antwort"] == 1) & (df_scoring["ist_antwort"] == 1)).sum())
fp_all = int(((df_scoring["pred_antwort"] == 1) & (df_scoring["ist_antwort"] == 0)).sum())
fn_all = int(((df_scoring["pred_antwort"] == 0) & (df_scoring["ist_antwort"] == 1)).sum())
tn_all = int(((df_scoring["pred_antwort"] == 0) & (df_scoring["ist_antwort"] == 0)).sum())

print("=== PLAUSI-CHECK (auf Gesamtmenge, nicht Holdout) ===")
print("TN:", tn_all, "FP:", fp_all, "FN:", fn_all, "TP:", tp_all)
print("Precision (gesamt):", round(tp_all / (tp_all + fp_all) if (tp_all + fp_all) else 0, 4))
print("Recall (gesamt):", round(tp_all / (tp_all + fn_all) if (tp_all + fn_all) else 0, 4))
print()

# ------------------------------------------------------------
# 4) Segment-Report anzeigen
# ------------------------------------------------------------
print("=== SEGMENT REPORT (aus Zelle 11) ===")
with pd.option_context("display.max_rows", 50, "display.max_columns", 30):
    print(seg_report)
print()

# ------------------------------------------------------------
# 5) Segmentverteilung in Targets (pred_antwort==1)
# ------------------------------------------------------------
print("=== SEGMENTVERTEILUNG IN TARGETS (pred_antwort==1) ===")
targets = df_scoring[df_scoring["pred_antwort"] == 1].copy()

seg_target_counts = targets["Segment"].value_counts(dropna=False)
seg_target_share = (seg_target_counts / len(targets)).round(4) if len(targets) else seg_target_counts

seg_targets = pd.DataFrame({
    "Segment": seg_target_counts.index,
    "Anzahl_Targets": seg_target_counts.values,
    "Anteil_Targets": seg_target_share.values
})

print(seg_targets)
print()

# ------------------------------------------------------------
# 6) Beispiele: Top/Bottom pro Segment
# ------------------------------------------------------------
print("=== BEISPIELE: TOP 5 je Segment ===")
for seg in df_scoring["Segment"].dropna().unique().tolist():
    sub = df_scoring[df_scoring["Segment"] == seg].head(5)
    print("\nSegment:", seg)
    with pd.option_context("display.max_rows", 10, "display.max_columns", 10):
        print(sub[["ID", "Segment", "proba_antwort", "pred_antwort", "ist_antwort"]])

print("\n=== BEISPIELE: BOTTOM 5 je Segment ===")
for seg in df_scoring["Segment"].dropna().unique().tolist():
    sub = df_scoring[df_scoring["Segment"] == seg].tail(5)
    print("\nSegment:", seg)
    with pd.option_context("display.max_rows", 10, "display.max_columns", 10):
        print(sub[["ID", "Segment", "proba_antwort", "pred_antwort", "ist_antwort"]])

print("\nHinweis: Wenn das alles plausibel aussieht, machen wir als nächstes den Export-Block (Zelle 13).")


=== DF_SCORING: OVERVIEW ===
Shape: (2240, 6)
Spalten: ['ID', 'Cluster', 'Segment', 'proba_antwort', 'pred_antwort', 'ist_antwort']

=== DF_SCORING: HEAD (Top 15) ===
       ID  Cluster                            Segment  proba_antwort  \
0   10749        1  Segment D: Low-Value / Infrequent       0.999935   
1    1501        0   Segment A: High-Value / Frequent       0.999747   
2    5376        0   Segment A: High-Value / Frequent       0.999523   
3    4931        0   Segment A: High-Value / Frequent       0.998779   
4    4702        0   Segment A: High-Value / Frequent       0.990791   
5    7919        0   Segment A: High-Value / Frequent       0.990132   
6    9595        0   Segment A: High-Value / Frequent       0.986443   
7    1891        0   Segment A: High-Value / Frequent       0.986059   
8    7627        0   Segment A: High-Value / Frequent       0.986002   
9    8707        0   Segment A: High-Value / Frequent       0.983710   
10   1859        0   Segment A: High-Valu

In [19]:
# ============================================================
# Zelle 13 — Export: Scoring, Target-Liste, Segment-Report, Kurzsummary
# ============================================================
"""
Analyst-Notiz:
Wir speichern die wichtigsten Ergebnisse als CSV, damit sie weiterverwendbar sind
(z. B. Kampagnenliste, Reporting, Übergabe an CRM/Marketing).

Exports:
1) scoring_all.csv         -> alle Kunden mit Score + Prediction + Segment
2) targets_threshold.csv   -> alle Kunden, die nach threshold_final angeschrieben würden
3) targets_topN.csv        -> Top-N Kunden nach Score (optional, budgetorientiert)
4) segment_report.csv      -> Segment-KPIs (mean_proba, pred_pos_rate, actual_rate)
5) run_summary.txt         -> kurze Zusammenfassung (Threshold, Counts, Lift)

Hinweis:
- Für die Performance-Bewertung zählt der Holdout aus Zelle 8/11.
- Das Vollfit-Modell ist hier nur für das Scoring (praktische Einsatzliste).
"""

import os
from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
if "df_scoring" not in globals():
    raise RuntimeError("df_scoring fehlt. Bitte zuerst Zelle 11/12 ausführen.")
if "seg_report" not in globals():
    raise RuntimeError("seg_report fehlt. Bitte zuerst Zelle 11/12 ausführen.")

# Schwelle (falls vorhanden)
threshold_final = float(globals().get("threshold_final", 0.50))

# ------------------------------------------------------------
# 1) Export-Ordner
# ------------------------------------------------------------
export_dir = Path("exports")
export_dir.mkdir(exist_ok=True)

# ------------------------------------------------------------
# 2) Scoring-Tabelle aufbereiten (Rank + saubere Spaltenreihenfolge)
# ------------------------------------------------------------
scoring_out = df_scoring.copy()
scoring_out["rank_score"] = np.arange(1, len(scoring_out) + 1)

# Spaltenordnung
cols_order = ["rank_score", "ID", "Cluster", "Segment", "proba_antwort", "pred_antwort", "ist_antwort"]
cols_order = [c for c in cols_order if c in scoring_out.columns]
scoring_out = scoring_out[cols_order]

# ------------------------------------------------------------
# 3) Targets erzeugen (Threshold + TopN)
# ------------------------------------------------------------
targets_threshold = scoring_out[scoring_out["pred_antwort"] == 1].copy()

TOP_N = 200  # Budget-orientiert anpassbar
targets_topN = scoring_out.head(TOP_N).copy()

# ------------------------------------------------------------
# 4) Dateinamen
# ------------------------------------------------------------
file_scoring = export_dir / "scoring_all.csv"
file_targets_thr = export_dir / "targets_threshold.csv"
file_targets_topN = export_dir / f"targets_top{TOP_N}.csv"
file_segment = export_dir / "segment_report.csv"
file_summary = export_dir / "run_summary.txt"

# ------------------------------------------------------------
# 5) CSV schreiben
# ------------------------------------------------------------
scoring_out.to_csv(file_scoring, index=False, sep=";")
targets_threshold.to_csv(file_targets_thr, index=False, sep=";")
targets_topN.to_csv(file_targets_topN, index=False, sep=";")
seg_report.to_csv(file_segment, index=False, sep=";")

# ------------------------------------------------------------
# 6) Kurzsummary schreiben
# ------------------------------------------------------------
total = len(scoring_out)
pred_pos = int((scoring_out["pred_antwort"] == 1).sum())
base_rate = float(scoring_out["ist_antwort"].mean())

tp = int(((scoring_out["pred_antwort"] == 1) & (scoring_out["ist_antwort"] == 1)).sum())
fp = int(((scoring_out["pred_antwort"] == 1) & (scoring_out["ist_antwort"] == 0)).sum())

precision_all = tp / (tp + fp) if (tp + fp) else 0.0
lift = precision_all / base_rate if base_rate else np.nan

summary_lines = [
    "RUN SUMMARY",
    f"threshold_final: {threshold_final}",
    f"gesamt_kunden: {total}",
    f"targets_threshold: {pred_pos}",
    f"base_rate (Antwort=1): {base_rate:.6f}",
    f"precision_in_targets (gesamt): {precision_all:.6f}",
    f"lift (targets_precision / base_rate): {lift:.3f}",
    "",
    "Dateien:",
    str(file_scoring),
    str(file_targets_thr),
    str(file_targets_topN),
    str(file_segment),
]

file_summary.write_text("\n".join(summary_lines), encoding="utf-8")

# ------------------------------------------------------------
# 7) Ausgabe
# ------------------------------------------------------------
print("=== EXPORT FERTIG ===")
print("scoring_all:", file_scoring)
print("targets_threshold:", file_targets_thr)
print("targets_topN:", file_targets_topN)
print("segment_report:", file_segment)
print("summary:", file_summary)

print("\nKurzübersicht:")
print("threshold_final:", threshold_final)
print("gesamt:", total, "| targets:", pred_pos, "| base_rate:", round(base_rate, 4))
print("precision_in_targets:", round(precision_all, 4), "| lift:", round(lift, 2))


=== EXPORT FERTIG ===
scoring_all: exports\scoring_all.csv
targets_threshold: exports\targets_threshold.csv
targets_topN: exports\targets_top200.csv
segment_report: exports\segment_report.csv
summary: exports\run_summary.txt

Kurzübersicht:
threshold_final: 0.5
gesamt: 2240 | targets: 700 | base_rate: 0.1491
precision_in_targets: 0.3757 | lift: 2.52


In [20]:
# ============================================================
# Zelle 14 — Abschluss-Report (Management Summary) + nächste Schritte
# ============================================================
"""
Analyst-Notiz:
Diese Zelle erzeugt eine kompakte, prüfungs-/berichtsfähige Zusammenfassung dessen,
was wir getan haben und was dabei herausgekommen ist — ohne Marketing-Floskeln,
sondern als Data-Analyst-Report.

Inhalte:
1) Daten & Zielvariable (Klassenverhältnis)
2) Feature Engineering (Blueprint -> df_features)
3) Segmentierung (KMeans) Ergebnis in 1-2 Sätzen
4) Supervised Modell (LogReg SAFE) Leistung + Interpretation
5) Kampagnen-Output (Threshold, Targets, Lift)
6) Empfehlung / nächste Schritte (optional: Kalibrierung, alternative Modelle)
"""

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
needed = ["df", "df_features", "df_clustered", "metrics_table", "threshold_final", "seg_report"]
missing = [v for v in needed if v not in globals()]
if missing:
    raise RuntimeError("Fehlende Objekte: " + ", ".join(missing) + ". Bitte vorherige Zellen ausführen.")

# ------------------------------------------------------------
# 1) Kennzahlen einsammeln
# ------------------------------------------------------------
n_total = len(df)
target_col = "Antwort_Letzte_Kampagne"
pos = int((df[target_col] == 1).sum())
neg = int((df[target_col] == 0).sum())
base_rate = pos / n_total if n_total else 0

# Modellmetriken (aus Zelle 8) — SAFE
m_safe = metrics_table.iloc[0].to_dict()
auc_safe = float(m_safe["ROC_AUC"])
prec_safe = float(m_safe["Precision"])
rec_safe = float(m_safe["Recall"])
f1_safe = float(m_safe["F1"])

# Kampagnenoutput (aus Zelle 12/13)
targets = int(df_scoring["pred_antwort"].sum()) if "df_scoring" in globals() else None
precision_targets = None
lift = None
if "df_scoring" in globals():
    tp = int(((df_scoring["pred_antwort"] == 1) & (df_scoring["ist_antwort"] == 1)).sum())
    fp = int(((df_scoring["pred_antwort"] == 1) & (df_scoring["ist_antwort"] == 0)).sum())
    precision_targets = tp / (tp + fp) if (tp + fp) else 0.0
    lift = precision_targets / base_rate if base_rate else np.nan

# Segment-Highlights
seg_sorted = seg_report.sort_values("mean_proba", ascending=False).copy()

# ------------------------------------------------------------
# 2) Berichtstext erstellen
# ------------------------------------------------------------
lines = []
lines.append("ABSCHLUSS-REPORT — Marktkampagne (Jan 2026)")
lines.append("-" * 60)
lines.append(f"Datenumfang: {n_total} Kunden")
lines.append(f"Zielvariable: {target_col} (1=Antwort, 0=keine Antwort)")
lines.append(f"Klassenverteilung: 0={neg} | 1={pos} | Basis-Response-Rate={base_rate:.4f}")
lines.append("")

lines.append("1) Feature Engineering")
lines.append("   - Blueprint mit 23 Features (Tenure/Demografie/RFM/Produktmix/Kanal + optional Kampagne)")
lines.append("   - Umsetzung in df_features, anschließend Imputation & Skalierung für Segmentierung")
lines.append("")

lines.append("2) Segmentierung (KMeans)")
lines.append("   - KMeans mit k=2 (beste Silhouette im Testbereich)")
lines.append("   - Ergebnis: zwei grobe Wert-/Verhaltenssegmente")
lines.append("     * Segment A: High-Value / Frequent")
lines.append("     * Segment D: Low-Value / Infrequent")
lines.append("")

lines.append("3) Vorhersagemodell (Logistische Regression, SAFE)")
lines.append("   - Modellvariante ohne Kampagnenhistorie (Leakage-Check bestanden)")
lines.append(f"   - Holdout-Leistung (Threshold=0.50): ROC-AUC={auc_safe:.3f} | Precision={prec_safe:.3f} | Recall={rec_safe:.3f} | F1={f1_safe:.3f}")
lines.append("   - Interpretation: Modell trennt gut (AUC hoch), arbeitet bei 0.50 eher recall-orientiert (viele Reagierer finden).")
lines.append("")

lines.append("4) Kampagnen-Output (Scoring & Targeting)")
lines.append(f"   - Schwelle (Threshold): {float(threshold_final):.2f}")
if targets is not None:
    lines.append(f"   - Targets (pred=1): {targets} von {n_total} ({targets/n_total:.4f})")
    lines.append(f"   - Erwartete Trefferqualität (auf Gesamt-Scoring plausibilisiert): Precision≈{precision_targets:.4f} (vs Basis {base_rate:.4f})")
    lines.append(f"   - Lift: {lift:.2f}×")
lines.append("")

lines.append("5) Segment-Performance (Überblick)")
for _, r in seg_sorted.iterrows():
    seg = r["Segment"]
    lines.append(
        f"   - {seg}: kunden={int(r['kunden'])}, actual_rate={float(r['actual_rate']):.4f}, "
        f"pred_pos_rate={float(r['pred_pos_rate']):.4f}, mean_proba={float(r['mean_proba']):.4f}"
    )
lines.append("")

lines.append("6) Empfehlung / nächste Schritte")
lines.append("   - Schwelle strategisch wählen (Budget vs. Reichweite):")
lines.append("     * Budget eng -> Threshold erhöhen (mehr Precision, weniger Targets)")
lines.append("     * Reichweite -> Threshold senken (mehr Recall, mehr Targets)")
lines.append("   - Optional zur Absicherung:")
lines.append("     * Kalibrierung prüfen (Probability Calibration)")
lines.append("     * Alternative Modelle testen (RandomForest / Gradient Boosting) und gegen LogReg vergleichen")
lines.append("     * Stabilität der Segmente (k=3..5) testen, falls feinere Marketingtypen benötigt werden")
lines.append("")

report_text = "\n".join(lines)

print(report_text)

# ------------------------------------------------------------
# 3) Als Datei speichern (für Abgabe)
# ------------------------------------------------------------
from pathlib import Path
export_dir = Path("exports")
export_dir.mkdir(exist_ok=True)
file_report = export_dir / "abschluss_report.txt"
file_report.write_text(report_text, encoding="utf-8")

print("\n=== REPORT GESPEICHERT ===")
print(file_report)


ABSCHLUSS-REPORT — Marktkampagne (Jan 2026)
------------------------------------------------------------
Datenumfang: 2240 Kunden
Zielvariable: Antwort_Letzte_Kampagne (1=Antwort, 0=keine Antwort)
Klassenverteilung: 0=1906 | 1=334 | Basis-Response-Rate=0.1491

1) Feature Engineering
   - Blueprint mit 23 Features (Tenure/Demografie/RFM/Produktmix/Kanal + optional Kampagne)
   - Umsetzung in df_features, anschließend Imputation & Skalierung für Segmentierung

2) Segmentierung (KMeans)
   - KMeans mit k=2 (beste Silhouette im Testbereich)
   - Ergebnis: zwei grobe Wert-/Verhaltenssegmente
     * Segment A: High-Value / Frequent
     * Segment D: Low-Value / Infrequent

3) Vorhersagemodell (Logistische Regression, SAFE)
   - Modellvariante ohne Kampagnenhistorie (Leakage-Check bestanden)
   - Holdout-Leistung (Threshold=0.50): ROC-AUC=0.857 | Precision=0.391 | Recall=0.795 | F1=0.524
   - Interpretation: Modell trennt gut (AUC hoch), arbeitet bei 0.50 eher recall-orientiert (viele Reagier

In [21]:
# ============================================================
# Zelle 15 — Modellvergleich (Baseline vs. Tree-Modelle) + Ranking
# ============================================================
"""
Analyst-Notiz:
Jetzt machen wir den nächsten professionellen Schritt: Modellvergleich.
Logistische Regression ist unsere erklärbare Baseline. Wir testen dagegen 2 nichtlineare Modelle,
die oft bei tabellarischen Kundendaten stärker sind:

1) RandomForestClassifier  (robust, wenig Tuning nötig)
2) GradientBoostingClassifier (stärker, kann Nichtlinearität abbilden)

Wir vergleichen auf dem gleichen Holdout:
- ROC-AUC (Hauptmetrik)
- Precision / Recall / F1 (bei Threshold 0.50 als Standard)
Zusätzlich:
- Wir speichern die besten Modelle und Wahrscheinlichkeiten für Zelle 16 (z. B. Kalibrierung / SHAP / Feature Importance).

Wichtig:
- Wir nutzen das SAFE-Feature-Set (ohne Kampagnenhistorie), damit es methodisch sauber bleibt.
"""

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
if "df" not in globals():
    raise RuntimeError("df fehlt. Bitte Zelle 1 ausführen.")
if "X_model_safe" not in globals():
    raise RuntimeError("X_model_safe fehlt. Bitte Zelle 4 ausführen.")

target_col = "Antwort_Letzte_Kampagne"
if target_col not in df.columns:
    raise KeyError(f"Zielspalte '{target_col}' fehlt in df.")

X = X_model_safe.copy()
y = pd.to_numeric(df[target_col], errors="coerce").fillna(0).astype(int)

# Stabiler Split (gleiche Parameter wie bisher)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ------------------------------------------------------------
# 1) Modelle definieren
# ------------------------------------------------------------
# Baseline: LogReg (wie gehabt)
pipe_logreg = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])

# RandomForest: keine Skalierung nötig
pipe_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        class_weight="balanced_subsample",
        n_jobs=-1
    ))
])

# GradientBoosting: keine Skalierung nötig
pipe_gb = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", GradientBoostingClassifier(
        random_state=42
    ))
])

models = [
    ("LogReg_SAFE", pipe_logreg),
    ("RandomForest_SAFE", pipe_rf),
    ("GradientBoosting_SAFE", pipe_gb),
]

# ------------------------------------------------------------
# 2) Fit + Evaluation
# ------------------------------------------------------------
def eval_model(name, pipe):
    pipe.fit(X_train, y_train)

    # proba für ROC-AUC
    proba = pipe.predict_proba(X_test)[:, 1]

    # Standard-Threshold 0.50
    pred = (proba >= 0.50).astype(int)

    out = {
        "Modell": name,
        "ROC_AUC": roc_auc_score(y_test, proba),
        "Accuracy": accuracy_score(y_test, pred),
        "Precision@0.50": precision_score(y_test, pred, zero_division=0),
        "Recall@0.50": recall_score(y_test, pred, zero_division=0),
        "F1@0.50": f1_score(y_test, pred, zero_division=0),
        "ConfusionMatrix": confusion_matrix(y_test, pred).tolist(),
        "pipe": pipe,
        "proba": proba
    }
    return out

rows = []
for name, pipe in models:
    rows.append(eval_model(name, pipe))

# Tabelle bauen (pipe/proba nicht mit ausgeben)
compare_df = pd.DataFrame([{
    "Modell": r["Modell"],
    "ROC_AUC": r["ROC_AUC"],
    "Accuracy": r["Accuracy"],
    "Precision@0.50": r["Precision@0.50"],
    "Recall@0.50": r["Recall@0.50"],
    "F1@0.50": r["F1@0.50"],
} for r in rows]).sort_values(["ROC_AUC", "F1@0.50"], ascending=False)

print("=== MODELLVERGLEICH (SAFE, Holdout) ===")
print(compare_df)

print("\n=== CONFUSION MATRICES (Threshold 0.50) ===")
for r in rows:
    print("\n", r["Modell"])
    print(r["ConfusionMatrix"])

# ------------------------------------------------------------
# 3) Bestes Modell auswählen (nach ROC-AUC)
# ------------------------------------------------------------
best_name = compare_df.iloc[0]["Modell"]
best_obj = [r for r in rows if r["Modell"] == best_name][0]

best_model_name = best_name
best_model_pipe = best_obj["pipe"]
best_model_proba = best_obj["proba"]

print("\n=== BESTES MODELL (nach ROC-AUC) ===")
print("best_model_name:", best_model_name)

# ------------------------------------------------------------
# 4) Ergebnisobjekte für nächste Zellen
# ------------------------------------------------------------
model_compare_table = compare_df.copy()

# Für Zelle 16 (z. B. Kalibrierung / Threshold-Tuning fürs best_model)
y_test_holdout_v2 = y_test.copy()
proba_best_holdout = best_model_proba.copy()

print("\nObjekte bereitgestellt:")
print("- model_compare_table")
print("- best_model_name, best_model_pipe")
print("- y_test_holdout_v2, proba_best_holdout")


=== MODELLVERGLEICH (SAFE, Holdout) ===
                  Modell   ROC_AUC  Accuracy  Precision@0.50  Recall@0.50  \
1      RandomForest_SAFE  0.875111  0.855357        0.513158     0.469880   
0            LogReg_SAFE  0.857152  0.785714        0.390533     0.795181   
2  GradientBoosting_SAFE  0.849196  0.857143        0.548387     0.204819   

    F1@0.50  
1  0.490566  
0  0.523810  
2  0.298246  

=== CONFUSION MATRICES (Threshold 0.50) ===

 LogReg_SAFE
[[374, 103], [17, 66]]

 RandomForest_SAFE
[[440, 37], [44, 39]]

 GradientBoosting_SAFE
[[463, 14], [66, 17]]

=== BESTES MODELL (nach ROC-AUC) ===
best_model_name: RandomForest_SAFE

Objekte bereitgestellt:
- model_compare_table
- best_model_name, best_model_pipe
- y_test_holdout_v2, proba_best_holdout


In [22]:
# ============================================================
# Zelle 16 — Threshold-Tuning für BESTES MODELL (aus Zelle 15)
# ============================================================
"""
Analyst-Notiz:
In Zelle 15 war RandomForest nach ROC-AUC das beste Modell.
Aber: Die Standard-Schwelle 0.50 ist selten optimal, vor allem bei unbalancierten Klassen.

In dieser Zelle:
1) Wir testen Schwellen (0.05 bis 0.95).
2) Wir berechnen pro Schwelle: Precision, Recall, F1, Accuracy + Confusion-Werte.
3) Wir wählen:
   - beste Schwelle nach F1 (Balance)
   - beste Schwelle nach Precision (Budget eng)
   - beste Schwelle nach Recall (max Treffer)
4) Optional: Vergleich gegen LogReg (falls proba_safe_holdout vorhanden).

Outputs:
- tab_best (Schwellentabelle)
- best_bestmodel (dict mit empfohlenen Schwellen)
"""

import numpy as np
import pandas as pd

from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
if "best_model_name" not in globals():
    raise RuntimeError("best_model_name fehlt. Bitte zuerst Zelle 15 ausführen.")
if "y_test_holdout_v2" not in globals():
    raise RuntimeError("y_test_holdout_v2 fehlt. Bitte zuerst Zelle 15 ausführen.")
if "proba_best_holdout" not in globals():
    raise RuntimeError("proba_best_holdout fehlt. Bitte zuerst Zelle 15 ausführen.")

y_true = y_test_holdout_v2.copy()
proba = np.array(proba_best_holdout)

# ------------------------------------------------------------
# 1) Schwellen-Auswertung
# ------------------------------------------------------------
def threshold_table(y_true, proba, thresholds):
    rows = []
    for t in thresholds:
        y_pred = (proba >= t).astype(int)

        prec = precision_score(y_true, y_pred, zero_division=0)
        rec  = recall_score(y_true, y_pred, zero_division=0)
        f1   = f1_score(y_true, y_pred, zero_division=0)
        acc  = accuracy_score(y_true, y_pred)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        rows.append({
            "threshold": float(t),
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "accuracy": acc,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "pred_pos": int(tp + fp),
            "pred_neg": int(tn + fn),
        })
    return pd.DataFrame(rows)

thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
tab_best = threshold_table(y_true, proba, thresholds)

# Rankings
best_f1 = tab_best.sort_values(["f1", "precision", "recall"], ascending=False).iloc[0]
best_prec = tab_best.sort_values(["precision", "recall"], ascending=False).iloc[0]
best_rec = tab_best.sort_values(["recall", "precision"], ascending=False).iloc[0]

print(f"=== THRESHOLD-TUNING: {best_model_name} (Holdout) ===")
print("\nTop 10 nach F1:")
print(tab_best.sort_values(["f1", "precision", "recall"], ascending=False).head(10))

def print_pick(row, title):
    print("\n" + title)
    print(f"threshold={row['threshold']:.2f} | precision={row['precision']:.3f} | recall={row['recall']:.3f} | f1={row['f1']:.3f} | acc={row['accuracy']:.3f}")
    print(f"Confusion: TN={row['tn']} FP={row['fp']} FN={row['fn']} TP={row['tp']} | pred_pos={row['pred_pos']}")

print_pick(best_f1,  "Empfehlung nach F1 (Balance):")
print_pick(best_prec,"Empfehlung nach Precision (Budget eng):")
print_pick(best_rec, "Empfehlung nach Recall (max Treffer):")

# ------------------------------------------------------------
# 2) Optional: Gegen LogReg vergleichen (falls vorhanden)
# ------------------------------------------------------------
if "proba_safe_holdout" in globals() and "y_test_holdout" in globals():
    # Achtung: nur sinnvoll, wenn y_test_holdout identisch gesplittet ist (bei euch i.d.R. ja)
    y_lr = y_test_holdout
    if len(y_lr) == len(y_true):
        tab_lr = threshold_table(y_lr, np.array(proba_safe_holdout), thresholds)
        best_lr_f1 = tab_lr.sort_values(["f1", "precision", "recall"], ascending=False).iloc[0]

        print("\n=== OPTIONAL: Vergleich LogReg vs BestModel (beste F1-Schwelle) ===")
        print("LogReg best F1:", f"t={best_lr_f1['threshold']:.2f}, f1={best_lr_f1['f1']:.3f}, prec={best_lr_f1['precision']:.3f}, rec={best_lr_f1['recall']:.3f}")
        print(best_model_name, "best F1:", f"t={best_f1['threshold']:.2f}, f1={best_f1['f1']:.3f}, prec={best_f1['precision']:.3f}, rec={best_f1['recall']:.3f}")
    else:
        print("\nHinweis: LogReg-Vergleich übersprungen (Holdout-Längen nicht identisch).")

# ------------------------------------------------------------
# 3) Ergebnisobjekte
# ------------------------------------------------------------
best_bestmodel = {
    "best_f1": best_f1.to_dict(),
    "best_precision": best_prec.to_dict(),
    "best_recall": best_rec.to_dict(),
}

threshold_best_f1 = float(best_f1["threshold"])

print("\nObjekte bereitgestellt:")
print("- tab_best (Schwellentabelle BestModel)")
print("- best_bestmodel (Empfehlungen)")
print("- threshold_best_f1 (Default für nächste Zelle)")


=== THRESHOLD-TUNING: RandomForest_SAFE (Holdout) ===

Top 10 nach F1:
    threshold  precision    recall        f1  accuracy   tn   fp  fn  tp  \
7        0.40   0.459016  0.674699  0.546341  0.833929  411   66  27  56   
5        0.30   0.409639  0.819277  0.546185  0.798214  379   98  15  68   
4        0.25   0.377551  0.891566  0.530466  0.766071  355  122   9  74   
8        0.45   0.484848  0.578313  0.527473  0.846429  426   51  35  48   
6        0.35   0.409722  0.710843  0.519824  0.805357  392   85  24  59   
9        0.50   0.513158  0.469880  0.490566  0.855357  440   37  44  39   
3        0.20   0.320988  0.939759  0.478528  0.696429  312  165   5  78   
10       0.55   0.508197  0.373494  0.430556  0.853571  447   30  52  31   
2        0.15   0.270270  0.963855  0.422164  0.608929  261  216   3  80   
11       0.60   0.625000  0.301205  0.406504  0.869643  462   15  58  25   

    pred_pos  pred_neg  
7        122       438  
5        166       394  
4        196     

In [23]:
# ============================================================
# Zelle 17 — Bestes Modell anwenden: Scoring + Targets + Segment-Report + Export
#            (RandomForest_SAFE mit optimaler F1-Schwelle aus Zelle 16)
# ============================================================
"""
Analyst-Notiz:
Wir nutzen jetzt das beste Modell aus dem Vergleich (Zelle 15) und die optimale Schwelle (Zelle 16),
um eine „kampagnenfertige“ Target-Liste zu erzeugen.

Ablauf:
1) Holdout-Check für die gewählte Schwelle (t = threshold_best_f1)
2) Modell auf Voll-Daten fitten (für Scoring/Ranking in der Praxis)
3) Scoring-Tabelle bauen (ID, Segment, proba, pred)
4) Segment-Report erstellen
5) Export als CSV (separate Dateien für BestModel, damit nichts überschrieben wird)

Outputs:
- df_scoring_best
- seg_report_best
- Exports: exports/scoring_bestmodel.csv, exports/targets_bestmodel_threshold.csv, exports/targets_bestmodel_topN.csv, exports/segment_report_bestmodel.csv
"""

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix, roc_auc_score

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
req = ["df", "X_model_safe", "df_clustered", "best_model_pipe", "best_model_name", "threshold_best_f1", "y_test_holdout_v2", "proba_best_holdout"]
missing = [r for r in req if r not in globals()]
if missing:
    raise RuntimeError("Fehlende Objekte: " + ", ".join(missing) + ". Bitte Zelle 15/16 prüfen.")

target_col = "Antwort_Letzte_Kampagne"
if target_col not in df.columns:
    raise KeyError(f"Zielspalte '{target_col}' fehlt in df.")

X_all = X_model_safe.copy()
y_all = pd.to_numeric(df[target_col], errors="coerce").fillna(0).astype(int)

t = float(threshold_best_f1)

# ------------------------------------------------------------
# 1) Holdout-Check bei gewählter Schwelle (zur Dokumentation)
# ------------------------------------------------------------
y_true = y_test_holdout_v2.copy()
proba_holdout = np.array(proba_best_holdout)
pred_holdout = (proba_holdout >= t).astype(int)

tn, fp, fn, tp = confusion_matrix(y_true, pred_holdout).ravel()

print("=== BESTMODEL HOLDOUТ CHECK ===")
print("Modell:", best_model_name)
print("Threshold (F1-optimal):", t)
print("ROC_AUC:", round(roc_auc_score(y_true, proba_holdout), 6))
print("Accuracy:", round(accuracy_score(y_true, pred_holdout), 6))
print("Precision:", round(precision_score(y_true, pred_holdout, zero_division=0), 6))
print("Recall:", round(recall_score(y_true, pred_holdout, zero_division=0), 6))
print("F1:", round(f1_score(y_true, pred_holdout, zero_division=0), 6))
print("ConfusionMatrix:", [[int(tn), int(fp)], [int(fn), int(tp)]])
print("pred_pos:", int(tp + fp))

# ------------------------------------------------------------
# 2) Modell auf Voll-Daten fitten (für Scoring-Liste)
# ------------------------------------------------------------
best_model_full = best_model_pipe.fit(X_all, y_all)

proba_all = best_model_full.predict_proba(X_all)[:, 1]
pred_all = (proba_all >= t).astype(int)

# ------------------------------------------------------------
# 3) Scoring-Tabelle bauen
# ------------------------------------------------------------
# ID / Segment aus df_clustered ziehen (robust)
if "ID" in df_clustered.columns:
    ids = df_clustered["ID"]
elif "ID" in df.columns:
    ids = df["ID"]
else:
    ids = pd.Series(np.arange(len(df)), name="ID")

cluster_col = df_clustered["Cluster"] if "Cluster" in df_clustered.columns else np.nan
segment_col = df_clustered["Segment"] if "Segment" in df_clustered.columns else np.nan

df_scoring_best = pd.DataFrame({
    "ID": ids,
    "Cluster": cluster_col,
    "Segment": segment_col,
    "proba_antwort": proba_all,
    "pred_antwort": pred_all,
    "ist_antwort": y_all.values
}).sort_values("proba_antwort", ascending=False).reset_index(drop=True)

df_scoring_best["rank_score"] = np.arange(1, len(df_scoring_best) + 1)

print("\n=== DF_SCORING_BEST: OVERVIEW ===")
print("Shape:", df_scoring_best.shape)
print("Targets (pred=1):", int(df_scoring_best["pred_antwort"].sum()), "von", len(df_scoring_best))

print("\n=== TOP 25 TARGETS (BestModel) ===")
with pd.option_context("display.max_rows", 30, "display.max_columns", 20):
    print(df_scoring_best.head(25)[["rank_score","ID","Segment","proba_antwort","pred_antwort","ist_antwort"]])

# ------------------------------------------------------------
# 4) Segment-Report (BestModel)
# ------------------------------------------------------------
seg_report_best = df_scoring_best.groupby(["Segment"], dropna=False).agg(
    kunden=("ID", "count"),
    mean_proba=("proba_antwort", "mean"),
    pred_pos=("pred_antwort", "sum"),
    actual_rate=("ist_antwort", "mean"),
).reset_index()

seg_report_best["pred_pos_rate"] = seg_report_best["pred_pos"] / seg_report_best["kunden"]
seg_report_best = seg_report_best.sort_values("mean_proba", ascending=False)

print("\n=== SEGMENT REPORT (BestModel) ===")
with pd.option_context("display.max_rows", 50, "display.max_columns", 30):
    print(seg_report_best)

# ------------------------------------------------------------
# 5) Export (separate Dateien fürs BestModel)
# ------------------------------------------------------------
export_dir = Path("exports")
export_dir.mkdir(exist_ok=True)

TOP_N = 200

file_scoring = export_dir / "scoring_bestmodel.csv"
file_targets_thr = export_dir / "targets_bestmodel_threshold.csv"
file_targets_topN = export_dir / f"targets_bestmodel_top{TOP_N}.csv"
file_segment = export_dir / "segment_report_bestmodel.csv"
file_summary = export_dir / "run_summary_bestmodel.txt"

# Ziel-Tabellen
targets_threshold = df_scoring_best[df_scoring_best["pred_antwort"] == 1].copy()
targets_topN = df_scoring_best.head(TOP_N).copy()

# Export (semicolon für DE-Excel)
df_scoring_best.to_csv(file_scoring, index=False, sep=";")
targets_threshold.to_csv(file_targets_thr, index=False, sep=";")
targets_topN.to_csv(file_targets_topN, index=False, sep=";")
seg_report_best.to_csv(file_segment, index=False, sep=";")

# Summary (Lift grob, auf Vollmenge nur Plausi)
base_rate = float(df_scoring_best["ist_antwort"].mean())
tp_all = int(((df_scoring_best["pred_antwort"] == 1) & (df_scoring_best["ist_antwort"] == 1)).sum())
fp_all = int(((df_scoring_best["pred_antwort"] == 1) & (df_scoring_best["ist_antwort"] == 0)).sum())
precision_targets = tp_all / (tp_all + fp_all) if (tp_all + fp_all) else 0.0
lift = precision_targets / base_rate if base_rate else np.nan

summary_lines = [
    "RUN SUMMARY — BESTMODEL",
    f"model: {best_model_name}",
    f"threshold_final: {t}",
    f"gesamt_kunden: {len(df_scoring_best)}",
    f"targets_threshold: {int(df_scoring_best['pred_antwort'].sum())}",
    f"base_rate (Antwort=1): {base_rate:.6f}",
    f"precision_in_targets (gesamt, Plausi): {precision_targets:.6f}",
    f"lift (targets_precision / base_rate): {lift:.3f}",
    "",
    "Dateien:",
    str(file_scoring),
    str(file_targets_thr),
    str(file_targets_topN),
    str(file_segment),
]

file_summary.write_text("\n".join(summary_lines), encoding="utf-8")

print("\n=== EXPORT BESTMODEL FERTIG ===")
print("scoring_bestmodel:", file_scoring)
print("targets_bestmodel_threshold:", file_targets_thr)
print("targets_bestmodel_topN:", file_targets_topN)
print("segment_report_bestmodel:", file_segment)
print("summary:", file_summary)

print("\nObjekte bereitgestellt:")
print("- df_scoring_best")
print("- seg_report_best")
print("- best_model_full (als best_model_full)")
best_model_full = best_model_full


=== BESTMODEL HOLDOUТ CHECK ===
Modell: RandomForest_SAFE
Threshold (F1-optimal): 0.4
ROC_AUC: 0.875111
Accuracy: 0.833929
Precision: 0.459016
Recall: 0.674699
F1: 0.546341
ConfusionMatrix: [[411, 66], [27, 56]]
pred_pos: 122

=== DF_SCORING_BEST: OVERVIEW ===
Shape: (2240, 7)
Targets (pred=1): 506 von 2240

=== TOP 25 TARGETS (BestModel) ===
    rank_score     ID                            Segment  proba_antwort  \
0            1   6072   Segment A: High-Value / Frequent       0.955989   
1            2   8362   Segment A: High-Value / Frequent       0.949780   
2            3   5547   Segment A: High-Value / Frequent       0.949780   
3            4   9064   Segment A: High-Value / Frequent       0.935730   
4            5   2407   Segment A: High-Value / Frequent       0.935730   
5            6   9560   Segment A: High-Value / Frequent       0.922075   
6            7   3690   Segment A: High-Value / Frequent       0.920703   
7            8   1859   Segment A: High-Value / Frequen

In [24]:
# ============================================================
# Zelle 18 — Vergleich SAFE(LogReg) vs BESTMODEL(RandomForest): Targets, Lift, Segment, Overlap
# ============================================================
"""
Analyst-Notiz:
Wir haben jetzt zwei kampagnenfähige Outputs:
- SAFE-LogReg (threshold_final=0.50)  -> df_scoring
- BestModel (RandomForest, threshold_best_f1=0.40) -> df_scoring_best

In dieser Zelle vergleichen wir die beiden Ansätze entlang von:
1) Kampagnenumfang (Anzahl Targets)
2) Treffergenauigkeit (Precision in Targets) + Lift (gegen Basisrate)
3) Segmentverteilung in Targets
4) Overlap: Welche Kunden werden von beiden Modellen ausgewählt? (Schnittmenge)
5) "Unique Targets": Wen nimmt nur LogReg / nur BestModel?

Outputs:
- compare_campaign_df (Vergleichstabelle)
- overlap_summary (Overlap-Kennzahlen)
- seg_targets_lr, seg_targets_best (Segmentverteilungen)
"""

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
need = ["df", "df_scoring", "df_scoring_best"]
missing = [x for x in need if x not in globals()]
if missing:
    raise RuntimeError("Fehlende Objekte: " + ", ".join(missing) + ". Bitte Zelle 11-13 und Zelle 17 ausführen.")

target_col = "Antwort_Letzte_Kampagne"
base_rate = float(pd.to_numeric(df[target_col], errors="coerce").fillna(0).astype(int).mean())

# ------------------------------------------------------------
# 1) Helper: Kampagnenkennzahlen
# ------------------------------------------------------------
def campaign_stats(scoring_df, label):
    total = len(scoring_df)
    targets = scoring_df[scoring_df["pred_antwort"] == 1].copy()
    n_targets = int(len(targets))

    tp = int(((targets["ist_antwort"] == 1)).sum())
    fp = int(((targets["ist_antwort"] == 0)).sum())

    precision_targets = tp / (tp + fp) if (tp + fp) else 0.0
    lift = precision_targets / base_rate if base_rate else np.nan

    # Segmentverteilung in Targets
    seg_counts = targets["Segment"].value_counts(dropna=False)
    seg_share = (seg_counts / n_targets).round(4) if n_targets else seg_counts

    seg_df = pd.DataFrame({
        "Segment": seg_counts.index.astype(str),
        "Anzahl_Targets": seg_counts.values,
        "Anteil_Targets": seg_share.values
    })

    return {
        "Modell": label,
        "gesamt": total,
        "targets": n_targets,
        "targets_rate": n_targets / total if total else 0.0,
        "precision_in_targets": precision_targets,
        "lift": lift,
        "tp_in_targets": tp,
        "fp_in_targets": fp,
        "seg_targets": seg_df
    }

stats_lr = campaign_stats(df_scoring, "LogReg_SAFE (t=0.50)")
stats_best = campaign_stats(df_scoring_best, "RandomForest_SAFE (t=best_f1=0.40)")

compare_campaign_df = pd.DataFrame([
    {k: v for k, v in stats_lr.items() if k != "seg_targets"},
    {k: v for k, v in stats_best.items() if k != "seg_targets"},
])

print("=== KAMPAGNEN-VERGLEICH ===")
print(compare_campaign_df[["Modell","targets","targets_rate","precision_in_targets","lift","tp_in_targets","fp_in_targets"]])

print("\nBasisrate:", round(base_rate, 4))

# Segmentverteilungen anzeigen
seg_targets_lr = stats_lr["seg_targets"]
seg_targets_best = stats_best["seg_targets"]

print("\n=== SEGMENTVERTEILUNG IN TARGETS — LogReg ===")
print(seg_targets_lr)

print("\n=== SEGMENTVERTEILUNG IN TARGETS — BestModel ===")
print(seg_targets_best)

# ------------------------------------------------------------
# 2) Overlap-Analyse (ID Schnittmenge)
# ------------------------------------------------------------
lr_targets_ids = set(df_scoring.loc[df_scoring["pred_antwort"] == 1, "ID"].astype(int).tolist())
best_targets_ids = set(df_scoring_best.loc[df_scoring_best["pred_antwort"] == 1, "ID"].astype(int).tolist())

intersect_ids = lr_targets_ids.intersection(best_targets_ids)
only_lr_ids = lr_targets_ids.difference(best_targets_ids)
only_best_ids = best_targets_ids.difference(lr_targets_ids)

overlap_summary = {
    "lr_targets": len(lr_targets_ids),
    "best_targets": len(best_targets_ids),
    "intersection": len(intersect_ids),
    "only_lr": len(only_lr_ids),
    "only_best": len(only_best_ids),
    "overlap_rate_vs_lr": len(intersect_ids)/len(lr_targets_ids) if len(lr_targets_ids) else 0.0,
    "overlap_rate_vs_best": len(intersect_ids)/len(best_targets_ids) if len(best_targets_ids) else 0.0,
}

print("\n=== OVERLAP SUMMARY (Targets) ===")
for k, v in overlap_summary.items():
    if isinstance(v, float):
        print(k + ":", round(v, 4))
    else:
        print(k + ":", v)

# ------------------------------------------------------------
# 3) Beispiele: Unique Targets (Top nach Score)
# ------------------------------------------------------------
print("\n=== TOP 15 UNIQUE TARGETS — nur LogReg ===")
only_lr_df = df_scoring[(df_scoring["pred_antwort"] == 1) & (~df_scoring["ID"].isin(list(intersect_ids)))].copy()
only_lr_df = only_lr_df.sort_values("proba_antwort", ascending=False).head(15)
with pd.option_context("display.max_rows", 20, "display.max_columns", 10):
    print(only_lr_df[["ID","Segment","proba_antwort","ist_antwort"]])

print("\n=== TOP 15 UNIQUE TARGETS — nur BestModel ===")
only_best_df = df_scoring_best[(df_scoring_best["pred_antwort"] == 1) & (~df_scoring_best["ID"].isin(list(intersect_ids)))].copy()
only_best_df = only_best_df.sort_values("proba_antwort", ascending=False).head(15)
with pd.option_context("display.max_rows", 20, "display.max_columns", 10):
    print(only_best_df[["ID","Segment","proba_antwort","ist_antwort"]])

print("\nObjekte bereitgestellt:")
print("- compare_campaign_df")
print("- overlap_summary")
print("- seg_targets_lr, seg_targets_best")


=== KAMPAGNEN-VERGLEICH ===
                               Modell  targets  targets_rate  \
0                LogReg_SAFE (t=0.50)      700      0.312500   
1  RandomForest_SAFE (t=best_f1=0.40)      506      0.225893   

   precision_in_targets      lift  tp_in_targets  fp_in_targets  
0              0.375714  2.519760            263            437  
1              0.660079  4.426877            334            172  

Basisrate: 0.1491

=== SEGMENTVERTEILUNG IN TARGETS — LogReg ===
                             Segment  Anzahl_Targets  Anteil_Targets
0   Segment A: High-Value / Frequent             484          0.6914
1  Segment D: Low-Value / Infrequent             216          0.3086

=== SEGMENTVERTEILUNG IN TARGETS — BestModel ===
                             Segment  Anzahl_Targets  Anteil_Targets
0   Segment A: High-Value / Frequent             343          0.6779
1  Segment D: Low-Value / Infrequent             163          0.3221

=== OVERLAP SUMMARY (Targets) ===
lr_targets: 700


In [25]:
# ============================================================
# Zelle 19 — Kampagnenstrategie: 3 Listen (Intersection / Union / High-Precision)
# ============================================================
"""
Analyst-Notiz:
Aus Zelle 18 haben wir jetzt die wichtigste Erkenntnis:
- LogReg liefert mehr Targets (700) aber geringere Precision (~0.376)
- RandomForest liefert weniger Targets (506) aber deutlich höhere Precision (~0.661)
- Overlap ist groß (Intersection 415) -> das sind die "sicheren" Kandidaten

In der Praxis baut man daraus 3 sinnvolle Listen:

A) SAFE-Liste (Intersection):
   - Kunden, die BEIDE Modelle targetieren -> hoher Konsens, meist robust.

B) Coverage-Liste (Union):
   - Kunden, die mindestens ein Modell targetiert -> maximaler Umfang.

C) High-Precision-Liste (BestModel-only Threshold):
   - Nur RandomForest-Targets (t=0.40) -> beste Trefferquote bei geringerem Budget.

Zusätzlich geben wir je Liste:
- Größe, Segmentverteilung
- Precision (auf Gesamtmenge als Plausi) + Lift

Exports:
- exports/targets_intersection.csv
- exports/targets_union.csv
- exports/targets_best_only.csv
"""

import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
need = ["df", "df_scoring", "df_scoring_best"]
missing = [x for x in need if x not in globals()]
if missing:
    raise RuntimeError("Fehlende Objekte: " + ", ".join(missing) + ". Bitte Zelle 11-13 + 17-18 ausführen.")

target_col = "Antwort_Letzte_Kampagne"
base_rate = float(pd.to_numeric(df[target_col], errors="coerce").fillna(0).astype(int).mean())

# Targets-IDs
lr_targets = set(df_scoring.loc[df_scoring["pred_antwort"] == 1, "ID"].astype(int).tolist())
best_targets = set(df_scoring_best.loc[df_scoring_best["pred_antwort"] == 1, "ID"].astype(int).tolist())

ids_intersection = lr_targets.intersection(best_targets)
ids_union = lr_targets.union(best_targets)
ids_best_only = best_targets.difference(lr_targets)

# Master-Scoring: wir nehmen BestModel-Scores als Rankingbasis (kannst du ändern)
# Vorteil: bestmodel ist präziser; für Union/Intersection trotzdem gut sortierbar.
master = df_scoring_best.copy()

# ------------------------------------------------------------
# 1) Helper: Liste bauen + Kennzahlen
# ------------------------------------------------------------
def build_list(master_df, id_set, label):
    sub = master_df[master_df["ID"].astype(int).isin(list(id_set))].copy()
    sub = sub.sort_values("proba_antwort", ascending=False).reset_index(drop=True)
    sub["rank_in_list"] = np.arange(1, len(sub) + 1)

    # Plausi-KPIs auf Gesamt
    tp = int(((sub["ist_antwort"] == 1)).sum())
    fp = int(((sub["ist_antwort"] == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    lift = precision / base_rate if base_rate else np.nan

    seg_counts = sub["Segment"].value_counts(dropna=False)
    seg_share = (seg_counts / len(sub)).round(4) if len(sub) else seg_counts

    seg_df = pd.DataFrame({
        "Segment": seg_counts.index.astype(str),
        "Anzahl": seg_counts.values,
        "Anteil": seg_share.values
    })

    summary = {
        "liste": label,
        "anzahl": int(len(sub)),
        "precision_plausi": float(precision),
        "lift": float(lift),
        "tp": tp,
        "fp": fp
    }

    return sub, seg_df, summary

list_intersection, seg_intersection, sum_intersection = build_list(master, ids_intersection, "Intersection (beide Modelle)")
list_union, seg_union, sum_union = build_list(master, ids_union, "Union (mind. ein Modell)")
list_best_only, seg_best_only, sum_best_only = build_list(master, ids_best_only, "BestModel-only (nur RF)")

summary_df = pd.DataFrame([sum_intersection, sum_union, sum_best_only])

print("=== LISTEN-SUMMARY ===")
print(summary_df[["liste","anzahl","precision_plausi","lift","tp","fp"]])

print("\nBasisrate:", round(base_rate, 4))

print("\n=== SEGMENTVERTEILUNG: INTERSECTION ===")
print(seg_intersection)
print("\n=== SEGMENTVERTEILUNG: UNION ===")
print(seg_union)
print("\n=== SEGMENTVERTEILUNG: BESTMODEL-ONLY ===")
print(seg_best_only)

print("\n=== TOP 20 — INTERSECTION ===")
with pd.option_context("display.max_rows", 25, "display.max_columns", 20):
    print(list_intersection.head(20)[["rank_in_list","ID","Segment","proba_antwort","pred_antwort","ist_antwort"]])

print("\n=== TOP 20 — BESTMODEL-ONLY ===")
with pd.option_context("display.max_rows", 25, "display.max_columns", 20):
    print(list_best_only.head(20)[["rank_in_list","ID","Segment","proba_antwort","pred_antwort","ist_antwort"]])

# ------------------------------------------------------------
# 2) Export
# ------------------------------------------------------------
export_dir = Path("exports")
export_dir.mkdir(exist_ok=True)

file_inter = export_dir / "targets_intersection.csv"
file_union = export_dir / "targets_union.csv"
file_best_only = export_dir / "targets_best_only.csv"
file_summary = export_dir / "targets_strategy_summary.csv"

# Semikolon für Excel DE
list_intersection.to_csv(file_inter, index=False, sep=";")
list_union.to_csv(file_union, index=False, sep=";")
list_best_only.to_csv(file_best_only, index=False, sep=";")
summary_df.to_csv(file_summary, index=False, sep=";")

print("\n=== EXPORT STRATEGIE-LISTEN FERTIG ===")
print("intersection:", file_inter)
print("union:", file_union)
print("best_only:", file_best_only)
print("summary:", file_summary)

print("\nObjekte bereitgestellt:")
print("- list_intersection, list_union, list_best_only")
print("- summary_df, seg_intersection, seg_union, seg_best_only")


=== LISTEN-SUMMARY ===
                          liste  anzahl  precision_plausi      lift   tp   fp
0  Intersection (beide Modelle)     415          0.633735  4.250198  263  152
1      Union (mind. ein Modell)     791          0.422250  2.831858  334  457
2       BestModel-only (nur RF)      91          0.780220  5.232612   71   20

Basisrate: 0.1491

=== SEGMENTVERTEILUNG: INTERSECTION ===
                             Segment  Anzahl  Anteil
0   Segment A: High-Value / Frequent     296  0.7133
1  Segment D: Low-Value / Infrequent     119  0.2867

=== SEGMENTVERTEILUNG: UNION ===
                             Segment  Anzahl  Anteil
0   Segment A: High-Value / Frequent     531  0.6713
1  Segment D: Low-Value / Infrequent     260  0.3287

=== SEGMENTVERTEILUNG: BESTMODEL-ONLY ===
                             Segment  Anzahl  Anteil
0   Segment A: High-Value / Frequent      47  0.5165
1  Segment D: Low-Value / Infrequent      44  0.4835

=== TOP 20 — INTERSECTION ===
    rank_in_list    

In [26]:
# ============================================================
# Zelle 20 — Finaler Abschluss-Text (für Abgabe) + Export als TXT
# ============================================================
"""
Analyst-Notiz:
Diese Zelle erzeugt einen finalen, abgabefähigen Textblock, der die komplette Pipeline erklärt
und die Ergebnisse/Empfehlungen zusammenfasst.

Der Text ist bewusst:
- kurz, fachlich, nachvollziehbar
- ohne Deko
- mit Zahlen (Basisrate, Lift, Targets)
"""

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
need = ["df", "model_compare_table", "summary_df"]
missing = [x for x in need if x not in globals()]
if missing:
    raise RuntimeError("Fehlende Objekte: " + ", ".join(missing) + ". Bitte vorherige Zellen ausführen.")

target_col = "Antwort_Letzte_Kampagne"
y = pd.to_numeric(df[target_col], errors="coerce").fillna(0).astype(int)

n_total = len(y)
pos = int((y == 1).sum())
neg = int((y == 0).sum())
base_rate = float(pos / n_total) if n_total else 0.0

# Modellvergleich: bestes ROC-AUC Modell
best_row = model_compare_table.sort_values(["ROC_AUC", "F1@0.50"], ascending=False).iloc[0]
best_model_name = str(best_row["Modell"])
best_auc = float(best_row["ROC_AUC"])

# LogReg-Werte (falls vorhanden)
logreg_row = model_compare_table[model_compare_table["Modell"].str.contains("LogReg", case=False)].head(1)
if len(logreg_row):
    logreg_row = logreg_row.iloc[0]
    logreg_auc = float(logreg_row["ROC_AUC"])
    logreg_f1 = float(logreg_row["F1@0.50"])
else:
    logreg_auc = np.nan
    logreg_f1 = np.nan

# Strategie-Listen Summary (Intersection/Union/Best-only)
# summary_df Spalten: liste, anzahl, precision_plausi, lift, tp, fp
summ = summary_df.copy()

# ------------------------------------------------------------
# 1) Text bauen
# ------------------------------------------------------------
lines = []
lines.append("FINAL SUMMARY — Marktkampagne (Jan 2026)")
lines.append("-" * 60)
lines.append(f"Ziel: Vorhersage der Kampagnenantwort (Antwort_Letzte_Kampagne: 1/0) und Erstellung einer Target-Liste.")
lines.append("")
lines.append(f"Daten: {n_total} Kunden | 1={pos} | 0={neg} | Basis-Response-Rate={base_rate:.4f}")
lines.append("")
lines.append("Feature Engineering:")
lines.append("- Blueprint mit 23 Features (Tenure/Demografie/RFM/Produktmix/Kanal; kampagnennahe Features optional, SAFE-Variante ohne Leakage).")
lines.append("")
lines.append("Segmentierung:")
lines.append("- KMeans (k=2) zur groben Kundenstruktur: Segment A (High-Value/Frequent) vs Segment D (Low-Value/Infrequent).")
lines.append("")
lines.append("Modellierung (SAFE):")
lines.append(f"- Modellvergleich (Holdout): bestes Modell nach ROC-AUC = {best_model_name} (ROC-AUC={best_auc:.3f}).")
if not np.isnan(logreg_auc):
    lines.append(f"- Baseline LogReg_SAFE: ROC-AUC={logreg_auc:.3f}, F1@0.50={logreg_f1:.3f} (gut erklärbar, recall-stark bei t=0.50).")
lines.append("")
lines.append("Kampagnenstrategie (Targets):")
for _, r in summ.iterrows():
    lines.append(
        f"- {r['liste']}: n={int(r['anzahl'])}, precision≈{float(r['precision_plausi']):.4f}, lift≈{float(r['lift']):.2f}× (Plausi auf Gesamtdaten)"
    )
lines.append("")
lines.append("Empfehlung:")
lines.append("- Wenn Budget knapp: BestModel-only Liste (höhere Trefferquote, weniger Kontakte).")
lines.append("- Wenn Sicherheit wichtig (robust): Intersection-Liste (beide Modelle stimmen überein).")
lines.append("- Wenn Reichweite wichtig: Union-Liste (größter Umfang).")
lines.append("")
lines.append("Hinweis zur Validierung:")
lines.append("- Kampagnen-Listen basieren auf Holdout-validierten Modellen; Scoring auf Vollmenge dient der operativen Rangliste.")
lines.append("- Optional zur Absicherung: Kalibrierung der Wahrscheinlichkeiten und Cross-Validation.")

final_text = "\n".join(lines)
print(final_text)

# ------------------------------------------------------------
# 2) Export
# ------------------------------------------------------------
export_dir = Path("exports")
export_dir.mkdir(exist_ok=True)

file_final = export_dir / "final_summary.txt"
file_final.write_text(final_text, encoding="utf-8")

print("\n=== FINAL SUMMARY GESPEICHERT ===")
print(file_final)


FINAL SUMMARY — Marktkampagne (Jan 2026)
------------------------------------------------------------
Ziel: Vorhersage der Kampagnenantwort (Antwort_Letzte_Kampagne: 1/0) und Erstellung einer Target-Liste.

Daten: 2240 Kunden | 1=334 | 0=1906 | Basis-Response-Rate=0.1491

Feature Engineering:
- Blueprint mit 23 Features (Tenure/Demografie/RFM/Produktmix/Kanal; kampagnennahe Features optional, SAFE-Variante ohne Leakage).

Segmentierung:
- KMeans (k=2) zur groben Kundenstruktur: Segment A (High-Value/Frequent) vs Segment D (Low-Value/Infrequent).

Modellierung (SAFE):
- Modellvergleich (Holdout): bestes Modell nach ROC-AUC = RandomForest_SAFE (ROC-AUC=0.875).
- Baseline LogReg_SAFE: ROC-AUC=0.857, F1@0.50=0.524 (gut erklärbar, recall-stark bei t=0.50).

Kampagnenstrategie (Targets):
- Intersection (beide Modelle): n=415, precision≈0.6337, lift≈4.25× (Plausi auf Gesamtdaten)
- Union (mind. ein Modell): n=791, precision≈0.4223, lift≈2.83× (Plausi auf Gesamtdaten)
- BestModel-only (nur RF)

In [27]:
# ============================================================
# Zelle 21 — Marketing-Metriken: Precision@K / Recall@K / Lift@K + Gain-Tabelle
#           (Vergleich: LogReg_SAFE vs RandomForest_SAFE)
# ============================================================
"""
Analyst-Notiz:
Marketing denkt selten in "Threshold 0.50", sondern in Budget/Slots:
"Wir können nur K Kunden anschreiben." -> dann zählen Top-K Metriken.

Diese Zelle berechnet für beide Modelle:
- Precision@K  (Trefferquote in den Top K)
- Recall@K     (wie viele der echten Reagierer wir in Top K abdecken)
- Lift@K       (Precision@K / Basisrate)
- Cumulative Gains (kumulierte Treffer über K)

Voraussetzungen:
- df_scoring        (LogReg SAFE Scoring auf Vollmenge)
- df_scoring_best   (BestModel/RandomForest Scoring auf Vollmenge)
"""

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
need = ["df", "df_scoring", "df_scoring_best"]
missing = [x for x in need if x not in globals()]
if missing:
    raise RuntimeError("Fehlende Objekte: " + ", ".join(missing) + ". Bitte Zelle 11-13 und 17 ausführen.")

target_col = "Antwort_Letzte_Kampagne"
y_all = pd.to_numeric(df[target_col], errors="coerce").fillna(0).astype(int)

n_total = len(y_all)
n_pos = int((y_all == 1).sum())
base_rate = float(y_all.mean())

print("=== BASIS ===")
print("n_total:", n_total, "| n_pos:", n_pos, "| base_rate:", round(base_rate, 4))
print()

# ------------------------------------------------------------
# 1) Helper: Top-K Tabelle
# ------------------------------------------------------------
def topk_table(scoring_df, model_name, ks):
    # erwartet: scoring_df enthält proba_antwort, ist_antwort
    work = scoring_df[["ID", "proba_antwort", "ist_antwort"]].copy()
    work = work.sort_values("proba_antwort", ascending=False).reset_index(drop=True)

    rows = []
    cum_tp = 0
    for k in ks:
        k = int(min(k, len(work)))
        topk = work.head(k)

        tp = int((topk["ist_antwort"] == 1).sum())
        precision_k = tp / k if k else 0.0
        recall_k = tp / n_pos if n_pos else 0.0
        lift_k = precision_k / base_rate if base_rate else np.nan

        rows.append({
            "Modell": model_name,
            "K": k,
            "TP_in_TopK": tp,
            "Precision@K": precision_k,
            "Recall@K": recall_k,
            "Lift@K": lift_k,
        })

    return pd.DataFrame(rows)

# sinnvolle K-Werte (Budget-Szenarien)
ks = [50, 100, 150, 200, 250, 300, 400, 500, 700, 1000]

tab_lr = topk_table(df_scoring, "LogReg_SAFE", ks)
tab_rf = topk_table(df_scoring_best, "RandomForest_SAFE", ks)

topk_compare = pd.concat([tab_lr, tab_rf], ignore_index=True)

# ------------------------------------------------------------
# 2) Vergleichs-Ausgabe
# ------------------------------------------------------------
print("=== TOP-K VERGLEICH (Precision/Recall/Lift) ===")
with pd.option_context("display.max_rows", 200, "display.max_columns", 20):
    print(topk_compare)

# Pivot: schnell vergleichen nach K
pivot_prec = topk_compare.pivot(index="K", columns="Modell", values="Precision@K")
pivot_lift = topk_compare.pivot(index="K", columns="Modell", values="Lift@K")
pivot_recall = topk_compare.pivot(index="K", columns="Modell", values="Recall@K")

print("\n=== PIVOT: Precision@K ===")
print(pivot_prec.round(4))

print("\n=== PIVOT: Lift@K ===")
print(pivot_lift.round(2))

print("\n=== PIVOT: Recall@K ===")
print(pivot_recall.round(4))

# ------------------------------------------------------------
# 3) Gains-Tabelle (kumulierte Treffer über K in % der Positiven)
# ------------------------------------------------------------
def gains_curve_table(scoring_df, model_name, steps=10):
    work = scoring_df[["proba_antwort", "ist_antwort"]].copy()
    work = work.sort_values("proba_antwort", ascending=False).reset_index(drop=True)

    # Dezile: 10%, 20%, ... 100%
    rows = []
    for pct in range(1, steps + 1):
        k = int(np.ceil(len(work) * pct / steps))
        topk = work.head(k)
        tp = int((topk["ist_antwort"] == 1).sum())
        gain = tp / n_pos if n_pos else 0.0
        rows.append({
            "Modell": model_name,
            "Anteil_Kontaktiert": pct / steps,
            "K": k,
            "TP_kumuliert": tp,
            "Gain(Recall_kumuliert)": gain
        })
    return pd.DataFrame(rows)

gain_lr = gains_curve_table(df_scoring, "LogReg_SAFE", steps=10)
gain_rf = gains_curve_table(df_scoring_best, "RandomForest_SAFE", steps=10)
gain_compare = pd.concat([gain_lr, gain_rf], ignore_index=True)

print("\n=== CUMULATIVE GAINS (Dezile) ===")
with pd.option_context("display.max_rows", 200, "display.max_columns", 20):
    print(gain_compare)

gain_pivot = gain_compare.pivot(index="Anteil_Kontaktiert", columns="Modell", values="Gain(Recall_kumuliert)")
print("\n=== PIVOT: CUMULATIVE GAINS ===")
print(gain_pivot.round(4))

# ------------------------------------------------------------
# 4) Objekte bereitstellen
# ------------------------------------------------------------
topk_metrics_table = topk_compare.copy()
gains_table = gain_compare.copy()

print("\nObjekte bereitgestellt:")
print("- topk_metrics_table")
print("- gains_table")


=== BASIS ===
n_total: 2240 | n_pos: 334 | base_rate: 0.1491

=== TOP-K VERGLEICH (Precision/Recall/Lift) ===
               Modell     K  TP_in_TopK  Precision@K  Recall@K    Lift@K
0         LogReg_SAFE    50          35     0.700000  0.104790  4.694611
1         LogReg_SAFE   100          60     0.600000  0.179641  4.023952
2         LogReg_SAFE   150          88     0.586667  0.263473  3.934531
3         LogReg_SAFE   200         111     0.555000  0.332335  3.722156
4         LogReg_SAFE   250         135     0.540000  0.404192  3.621557
5         LogReg_SAFE   300         159     0.530000  0.476048  3.554491
6         LogReg_SAFE   400         199     0.497500  0.595808  3.336527
7         LogReg_SAFE   500         223     0.446000  0.667665  2.991138
8         LogReg_SAFE   700         263     0.375714  0.787425  2.519760
9         LogReg_SAFE  1000         302     0.302000  0.904192  2.025389
10  RandomForest_SAFE    50          50     1.000000  0.149701  6.706587
11  RandomFore

In [28]:
# ============================================================
# Zelle 22 — Korrekte Marketing-Metriken auf HOLDOUT (fairer Vergleich)
#           Precision@K / Recall@K / Lift@K + Gains (LogReg vs BestModel)
# ============================================================
"""
Analyst-Notiz:
Die vorherigen Top-K/Gains waren auf Voll-Daten (in-sample) und dadurch optimistisch.
Hier machen wir es korrekt: Bewertung ausschließlich auf dem Holdout.

Vorgehen:
1) Wir reproduzieren denselben Split (random_state=42, stratify=y, test_size=0.25).
2) Wir fitten LogReg SAFE und BestModel (best_model_pipe) NUR auf Train.
3) Wir erzeugen Probas auf Test (Holdout).
4) Dann berechnen wir Top-K und Gains NUR auf Holdout.

Outputs:
- topk_holdout_table
- gains_holdout_table
"""

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
need = ["df", "X_model_safe", "best_model_pipe"]
missing = [x for x in need if x not in globals()]
if missing:
    raise RuntimeError("Fehlende Objekte: " + ", ".join(missing) + ". Bitte Zelle 15/17 prüfen.")

target_col = "Antwort_Letzte_Kampagne"
y = pd.to_numeric(df[target_col], errors="coerce").fillna(0).astype(int)
X = X_model_safe.copy()

# identischer Split wie bisher
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

n_pos = int((y_test == 1).sum())
base_rate_test = float(y_test.mean())

print("=== HOLDOUT BASIS ===")
print("n_test:", len(y_test), "| n_pos:", n_pos, "| base_rate_test:", round(base_rate_test, 4))
print()

# ------------------------------------------------------------
# 1) Modelle fitten (nur Train) und Probas auf Test erzeugen
# ------------------------------------------------------------
logreg_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])

logreg_fit = logreg_pipe.fit(X_train, y_train)
proba_lr = logreg_fit.predict_proba(X_test)[:, 1]

best_fit = best_model_pipe.fit(X_train, y_train)
proba_best = best_fit.predict_proba(X_test)[:, 1]

# ------------------------------------------------------------
# 2) Helper: Top-K und Gains auf Holdout
# ------------------------------------------------------------
def topk_from_proba(y_true, proba, model_name, ks):
    order = np.argsort(-proba)  # desc
    y_sorted = np.array(y_true)[order]

    rows = []
    for k in ks:
        k = int(min(k, len(y_sorted)))
        tp = int(y_sorted[:k].sum())
        precision_k = tp / k if k else 0.0
        recall_k = tp / n_pos if n_pos else 0.0
        lift_k = precision_k / base_rate_test if base_rate_test else np.nan

        rows.append({
            "Modell": model_name,
            "K": k,
            "TP_in_TopK": tp,
            "Precision@K": precision_k,
            "Recall@K": recall_k,
            "Lift@K": lift_k
        })
    return pd.DataFrame(rows)

def gains_from_proba(y_true, proba, model_name, steps=10):
    order = np.argsort(-proba)
    y_sorted = np.array(y_true)[order]
    rows = []
    for pct in range(1, steps + 1):
        k = int(np.ceil(len(y_sorted) * pct / steps))
        tp = int(y_sorted[:k].sum())
        gain = tp / n_pos if n_pos else 0.0
        rows.append({
            "Modell": model_name,
            "Anteil_Kontaktiert": pct / steps,
            "K": k,
            "TP_kumuliert": tp,
            "Gain(Recall_kumuliert)": gain
        })
    return pd.DataFrame(rows)

ks = [25, 50, 75, 100, 125, 150, 200, 250, 300, 400, 500]

tab_lr = topk_from_proba(y_test, proba_lr, "LogReg_SAFE (Holdout)", ks)
tab_best = topk_from_proba(y_test, proba_best, "BestModel (Holdout)", ks)
topk_holdout_table = pd.concat([tab_lr, tab_best], ignore_index=True)

print("=== TOP-K (HOLDOUT) ===")
with pd.option_context("display.max_rows", 200, "display.max_columns", 20):
    print(topk_holdout_table)

print("\n=== PIVOT: Precision@K (Holdout) ===")
print(topk_holdout_table.pivot(index="K", columns="Modell", values="Precision@K").round(4))

print("\n=== PIVOT: Lift@K (Holdout) ===")
print(topk_holdout_table.pivot(index="K", columns="Modell", values="Lift@K").round(2))

print("\n=== PIVOT: Recall@K (Holdout) ===")
print(topk_holdout_table.pivot(index="K", columns="Modell", values="Recall@K").round(4))

g_lr = gains_from_proba(y_test, proba_lr, "LogReg_SAFE (Holdout)", steps=10)
g_best = gains_from_proba(y_test, proba_best, "BestModel (Holdout)", steps=10)
gains_holdout_table = pd.concat([g_lr, g_best], ignore_index=True)

print("\n=== GAINS (HOLDOUT, Dezile) ===")
with pd.option_context("display.max_rows", 200, "display.max_columns", 20):
    print(gains_holdout_table)

print("\n=== PIVOT: GAINS (Holdout) ===")
print(gains_holdout_table.pivot(index="Anteil_Kontaktiert", columns="Modell", values="Gain(Recall_kumuliert)").round(4))

print("\nObjekte bereitgestellt:")
print("- topk_holdout_table")
print("- gains_holdout_table")


=== HOLDOUT BASIS ===
n_test: 560 | n_pos: 83 | base_rate_test: 0.1482

=== TOP-K (HOLDOUT) ===
                   Modell    K  TP_in_TopK  Precision@K  Recall@K    Lift@K
0   LogReg_SAFE (Holdout)   25          14     0.560000  0.168675  3.778313
1   LogReg_SAFE (Holdout)   50          28     0.560000  0.337349  3.778313
2   LogReg_SAFE (Holdout)   75          37     0.493333  0.445783  3.328514
3   LogReg_SAFE (Holdout)  100          47     0.470000  0.566265  3.171084
4   LogReg_SAFE (Holdout)  125          54     0.432000  0.650602  2.914699
5   LogReg_SAFE (Holdout)  150          59     0.393333  0.710843  2.653815
6   LogReg_SAFE (Holdout)  200          70     0.350000  0.843373  2.361446
7   LogReg_SAFE (Holdout)  250          75     0.300000  0.903614  2.024096
8   LogReg_SAFE (Holdout)  300          79     0.263333  0.951807  1.776707
9   LogReg_SAFE (Holdout)  400          83     0.207500  1.000000  1.400000
10  LogReg_SAFE (Holdout)  500          83     0.166000  1.000000  1

In [29]:
# ============================================================
# Zelle 23 — Kalibrierung (Holdout): Reliability + Brier Score + Erwartete Antworten
# ============================================================
"""
Analyst-Notiz:
Viele Modelle (insb. RandomForest) sind gut im Ranking, aber nicht perfekt kalibriert.
Kalibrierung bedeutet: Wenn das Modell 0.70 sagt, sollen ~70% dieser Fälle wirklich 1 sein.

Wir machen auf Holdout:
1) Brier Score (niedriger = besser)
2) Bin-basierte Calibration Table (Reliability)
3) Kalibrierung mit Platt Scaling (Sigmoid) und Isotonic
4) Vergleich: unkalibriert vs kalibriert
5) Erwartete Antworten für Top-K (Summe der Probabilities) vor/nach Kalibrierung

Outputs:
- calib_table_best (Reliability Tabelle)
- brier_scores_df
- proba_best_cal_sigmoid, proba_best_cal_isotonic
- expected_responses_table (Top-K Erwartungswerte)
"""

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
need = ["df", "X_model_safe", "best_model_pipe"]
missing = [x for x in need if x not in globals()]
if missing:
    raise RuntimeError("Fehlende Objekte: " + ", ".join(missing) + ". Bitte Zelle 15/17 prüfen.")

target_col = "Antwort_Letzte_Kampagne"
y = pd.to_numeric(df[target_col], errors="coerce").fillna(0).astype(int)
X = X_model_safe.copy()

# identischer Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ------------------------------------------------------------
# 1) BestModel: unkalibriert fitten + proba
# ------------------------------------------------------------
best_uncal = best_model_pipe.fit(X_train, y_train)
proba_uncal = best_uncal.predict_proba(X_test)[:, 1]

# ------------------------------------------------------------
# 2) Kalibrierung (Sigmoid / Isotonic) mit CV auf Train
# ------------------------------------------------------------
# Wir kalibrieren auf Train via CV und wenden dann auf Test an
cal_sigmoid = CalibratedClassifierCV(estimator=best_model_pipe, method="sigmoid", cv=5)
cal_isotonic = CalibratedClassifierCV(estimator=best_model_pipe, method="isotonic", cv=5)

cal_sigmoid.fit(X_train, y_train)
cal_isotonic.fit(X_train, y_train)

proba_sigmoid = cal_sigmoid.predict_proba(X_test)[:, 1]
proba_isotonic = cal_isotonic.predict_proba(X_test)[:, 1]

# ------------------------------------------------------------
# 3) Brier Scores
# ------------------------------------------------------------
brier_uncal = brier_score_loss(y_test, proba_uncal)
brier_sig = brier_score_loss(y_test, proba_sigmoid)
brier_iso = brier_score_loss(y_test, proba_isotonic)

brier_scores_df = pd.DataFrame({
    "Variante": ["Uncalibrated", "Sigmoid (Platt)", "Isotonic"],
    "Brier_Score": [brier_uncal, brier_sig, brier_iso]
}).sort_values("Brier_Score")

print("=== BRIER SCORE (niedriger = besser) ===")
print(brier_scores_df)

# ------------------------------------------------------------
# 4) Reliability Table (Bins)
# ------------------------------------------------------------
def reliability_table(y_true, proba, n_bins=10):
    dfc = pd.DataFrame({"y": y_true, "p": proba})
    dfc["bin"] = pd.qcut(dfc["p"], q=n_bins, duplicates="drop")
    out = dfc.groupby("bin").agg(
        count=("y", "count"),
        mean_proba=("p", "mean"),
        actual_rate=("y", "mean"),
    ).reset_index()
    out["gap"] = out["actual_rate"] - out["mean_proba"]
    return out

rel_uncal = reliability_table(y_test, proba_uncal, n_bins=10)
rel_sig = reliability_table(y_test, proba_sigmoid, n_bins=10)
rel_iso = reliability_table(y_test, proba_isotonic, n_bins=10)

print("\n=== RELIABILITY (Uncalibrated) ===")
print(rel_uncal[["count","mean_proba","actual_rate","gap"]])

print("\n=== RELIABILITY (Sigmoid) ===")
print(rel_sig[["count","mean_proba","actual_rate","gap"]])

print("\n=== RELIABILITY (Isotonic) ===")
print(rel_iso[["count","mean_proba","actual_rate","gap"]])

# ------------------------------------------------------------
# 5) Erwartete Antworten in Top-K (Summe Probabilities) vs echte Treffer
# ------------------------------------------------------------
def expected_in_topk(y_true, proba, ks, label):
    order = np.argsort(-proba)
    y_sorted = np.array(y_true)[order]
    p_sorted = np.array(proba)[order]

    rows = []
    for k in ks:
        k = int(min(k, len(y_sorted)))
        exp_resp = float(p_sorted[:k].sum())
        act_resp = int(y_sorted[:k].sum())
        rows.append({
            "Variante": label,
            "K": k,
            "expected_responses_sum_p": exp_resp,
            "actual_responses": act_resp
        })
    return pd.DataFrame(rows)

ks = [25, 50, 100, 150, 200, 250, 300]
exp_uncal = expected_in_topk(y_test, proba_uncal, ks, "Uncalibrated")
exp_sig = expected_in_topk(y_test, proba_sigmoid, ks, "Sigmoid")
exp_iso = expected_in_topk(y_test, proba_isotonic, ks, "Isotonic")

expected_responses_table = pd.concat([exp_uncal, exp_sig, exp_iso], ignore_index=True)

print("\n=== EXPECTED vs ACTUAL (Top-K) ===")
print(expected_responses_table)

# ------------------------------------------------------------
# 6) Objekte bereitstellen
# ------------------------------------------------------------
proba_best_cal_sigmoid = proba_sigmoid
proba_best_cal_isotonic = proba_isotonic

print("\nObjekte bereitgestellt:")
print("- brier_scores_df")
print("- rel_uncal, rel_sig, rel_iso")
print("- expected_responses_table")
print("- proba_best_cal_sigmoid, proba_best_cal_isotonic")


=== BRIER SCORE (niedriger = besser) ===
          Variante  Brier_Score
1  Sigmoid (Platt)     0.095318
2         Isotonic     0.095831
0     Uncalibrated     0.100152

=== RELIABILITY (Uncalibrated) ===
   count  mean_proba  actual_rate       gap
0     56    0.011819     0.017857  0.006038
1     56    0.036141     0.000000 -0.036141
2     56    0.066233     0.000000 -0.066233
3     56    0.101479     0.000000 -0.101479
4     56    0.138525     0.035714 -0.102811
5     56    0.188284     0.053571 -0.134712
6     56    0.251463     0.160714 -0.090748
7     56    0.359754     0.267857 -0.091897
8     56    0.481675     0.428571 -0.053103
9     56    0.677372     0.517857 -0.159515

=== RELIABILITY (Sigmoid) ===
   count  mean_proba  actual_rate       gap
0     56    0.024297     0.017857 -0.006440
1     56    0.028404     0.000000 -0.028404
2     56    0.034357     0.000000 -0.034357
3     56    0.043514     0.000000 -0.043514
4     56    0.054711     0.035714 -0.018997
5     56    0.07

C:\Users\mickh\AppData\Local\Temp\ipykernel_36824\2679510269.py:87: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = dfc.groupby("bin").agg(
C:\Users\mickh\AppData\Local\Temp\ipykernel_36824\2679510269.py:87: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = dfc.groupby("bin").agg(
C:\Users\mickh\AppData\Local\Temp\ipykernel_36824\2679510269.py:87: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = dfc.groupby("bin

In [30]:
# ============================================================
# Zelle 24 — Budget/ROI-Simulation (Holdout) mit kalibrierten Probabilities
#           (Sigmoid vs Isotonic) für Top-K Kampagnenplanung
# ============================================================
"""
Analyst-Notiz:
Wir simulieren Kampagnenentscheidungen für verschiedene Budgets (Top-K).
Wir nutzen kalibrierte Wahrscheinlichkeiten, um erwartete Antworten zu schätzen.

Eingaben (Default, anpassbar):
- cost_per_contact: Kosten pro Kontakt (z.B. E-Mail ~0.05, Print ~0.80)
- value_per_response: Wert pro Antwort (Umsatz/Deckungsbeitrag), z.B. 25€ oder 50€

Outputs:
- roi_table (K, expected_responses, costs, expected_value, expected_profit, ROI)
"""

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
need = ["y_test_holdout_v2", "proba_best_cal_sigmoid", "proba_best_cal_isotonic"]
missing = [x for x in need if x not in globals()]
if missing:
    raise RuntimeError("Fehlende Objekte: " + ", ".join(missing) + ". Bitte Zelle 23 ausführen.")

y_true = np.array(y_test_holdout_v2)
p_sig = np.array(proba_best_cal_sigmoid)
p_iso = np.array(proba_best_cal_isotonic)

# ------------------------------------------------------------
# 1) Parameter (kannst du später ändern)
# ------------------------------------------------------------
cost_per_contact = 0.05       # Default: Email
value_per_response = 25.00    # Default: Deckungsbeitrag pro Antwort

Ks = [25, 50, 100, 150, 200, 250, 300]

# ------------------------------------------------------------
# 2) Helper: ROI Table
# ------------------------------------------------------------
def roi_for_proba(y_true, proba, label, Ks, cost_per_contact, value_per_response):
    order = np.argsort(-proba)
    y_sorted = y_true[order]
    p_sorted = proba[order]

    rows = []
    for k in Ks:
        k = int(min(k, len(y_sorted)))

        expected_responses = float(p_sorted[:k].sum())
        actual_responses = int(y_sorted[:k].sum())

        cost = k * cost_per_contact
        expected_value = expected_responses * value_per_response
        expected_profit = expected_value - cost
        roi = (expected_profit / cost) if cost > 0 else np.nan

        rows.append({
            "Variante": label,
            "K": k,
            "expected_responses": expected_responses,
            "actual_responses(holdout)": actual_responses,
            "cost_total": cost,
            "expected_value": expected_value,
            "expected_profit": expected_profit,
            "ROI": roi
        })
    return pd.DataFrame(rows)

roi_sig = roi_for_proba(y_true, p_sig, "Sigmoid", Ks, cost_per_contact, value_per_response)
roi_iso = roi_for_proba(y_true, p_iso, "Isotonic", Ks, cost_per_contact, value_per_response)

roi_table = pd.concat([roi_sig, roi_iso], ignore_index=True)

print("=== ROI / BUDGET SIMULATION (Holdout, kalibriert) ===")
print("cost_per_contact:", cost_per_contact, "| value_per_response:", value_per_response)
with pd.option_context("display.max_rows", 200, "display.max_columns", 30):
    print(roi_table)

print("\n=== PIVOT: expected_responses ===")
print(roi_table.pivot(index="K", columns="Variante", values="expected_responses").round(2))

print("\n=== PIVOT: expected_profit ===")
print(roi_table.pivot(index="K", columns="Variante", values="expected_profit").round(2))

print("\nObjekte bereitgestellt:")
print("- roi_table")


=== ROI / BUDGET SIMULATION (Holdout, kalibriert) ===
cost_per_contact: 0.05 | value_per_response: 25.0
    Variante    K  expected_responses  actual_responses(holdout)  cost_total  \
0    Sigmoid   25           17.673608                         18        1.25   
1    Sigmoid   50           30.097680                         26        2.50   
2    Sigmoid  100           47.167636                         49        5.00   
3    Sigmoid  150           57.855847                         63        7.50   
4    Sigmoid  200           64.205266                         74       10.00   
5    Sigmoid  250           68.579774                         79       12.50   
6    Sigmoid  300           71.746693                         81       15.00   
7   Isotonic   25           18.788186                         18        1.25   
8   Isotonic   50           32.155713                         27        2.50   
9   Isotonic  100           50.006574                         49        5.00   
10  Isotonic  15

In [31]:
# ============================================================
# Zelle 25 — Sensitivitätsanalyse: ROI unter verschiedenen Annahmen (Isotonic)
# ============================================================
"""
Analyst-Notiz:
ROI hängt vollständig von Annahmen ab (Kosten pro Kontakt, Wert pro Antwort).
Mit dieser Tabelle zeigen wir robuste Entscheidungsfähigkeit:
Wir variieren die Annahmen und sehen, welche K-Strategie stabil profitabel bleibt.

Wir nutzen Isotonic-Probas (kalibriert) für erwartete Antworten.
"""

import numpy as np
import pandas as pd

# Voraussetzungen
need = ["y_test_holdout_v2", "proba_best_cal_isotonic"]
missing = [x for x in need if x not in globals()]
if missing:
    raise RuntimeError("Fehlende Objekte: " + ", ".join(missing) + ". Bitte Zelle 23 ausführen.")

y_true = np.array(y_test_holdout_v2)
p_iso = np.array(proba_best_cal_isotonic)

# Settings (kannst du anpassen)
Ks = [25, 50, 100, 150, 200, 250, 300]
costs = [0.05, 0.20, 0.80]          # Mail / SMS / Print (Beispiele)
values = [5, 10, 25, 50]            # niedriger bis hoher Wert pro Antwort

# Sortierung nach Score
order = np.argsort(-p_iso)
p_sorted = p_iso[order]
y_sorted = y_true[order]

rows = []
for k in Ks:
    k = int(min(k, len(p_sorted)))
    exp_resp = float(p_sorted[:k].sum())
    act_resp = int(y_sorted[:k].sum())

    for c in costs:
        for v in values:
            cost_total = k * c
            exp_value = exp_resp * v
            exp_profit = exp_value - cost_total
            roi = (exp_profit / cost_total) if cost_total > 0 else np.nan

            rows.append({
                "K": k,
                "cost_per_contact": c,
                "value_per_response": v,
                "expected_responses": exp_resp,
                "actual_responses(holdout)": act_resp,
                "expected_profit": exp_profit,
                "ROI": roi
            })

sens_table = pd.DataFrame(rows)

print("=== SENSITIVITÄTSANALYSE (Isotonic, Holdout) ===")
with pd.option_context("display.max_rows", 200, "display.max_columns", 20):
    print(sens_table.sort_values(["value_per_response","cost_per_contact","K"]))

print("\nTipp: Filtere z. B. auf value_per_response=25 und vergleiche K über costs.")
print("\nObjekt bereitgestellt: sens_table")


=== SENSITIVITÄTSANALYSE (Isotonic, Holdout) ===
      K  cost_per_contact  value_per_response  expected_responses  \
0    25              0.05                   5           18.788186   
12   50              0.05                   5           32.155713   
24  100              0.05                   5           50.006574   
36  150              0.05                   5           61.510054   
48  200              0.05                   5           69.307748   
60  250              0.05                   5           74.273409   
72  300              0.05                   5           77.891515   
4    25              0.20                   5           18.788186   
16   50              0.20                   5           32.155713   
28  100              0.20                   5           50.006574   
40  150              0.20                   5           61.510054   
52  200              0.20                   5           69.307748   
64  250              0.20                   5         

In [32]:
# ============================================================
# Zelle 26 — Best-K pro Szenario (max Profit / max ROI) + Break-even Check
# ============================================================
"""
Analyst-Notiz:
Wir nehmen sens_table und bestimmen für jede (cost_per_contact, value_per_response)-Kombination:
- K mit maximalem expected_profit
- K mit maximalem ROI
- ob/ab wann expected_profit wieder fällt (Diminishing Returns / Break-even)

Outputs:
- best_profit_table
- best_roi_table
- profit_curve_flags (zeigt, ob Profit nach Peak fällt)
"""

import numpy as np
import pandas as pd

if "sens_table" not in globals():
    raise RuntimeError("sens_table fehlt. Bitte Zelle 25 ausführen.")

st = sens_table.copy()

# ----------------------------
# 1) Best K nach Profit
# ----------------------------
best_profit_rows = []
for (c, v), g in st.groupby(["cost_per_contact", "value_per_response"], observed=False):
    g2 = g.sort_values("expected_profit", ascending=False).iloc[0]
    best_profit_rows.append({
        "cost_per_contact": c,
        "value_per_response": v,
        "bestK_profit": int(g2["K"]),
        "max_expected_profit": float(g2["expected_profit"]),
        "ROI_at_bestK_profit": float(g2["ROI"]),
        "expected_responses_at_bestK_profit": float(g2["expected_responses"]),
    })

best_profit_table = pd.DataFrame(best_profit_rows).sort_values(
    ["value_per_response", "cost_per_contact"]
)

print("=== BEST K NACH MAX PROFIT ===")
print(best_profit_table)

# ----------------------------
# 2) Best K nach ROI
# ----------------------------
best_roi_rows = []
for (c, v), g in st.groupby(["cost_per_contact", "value_per_response"], observed=False):
    g2 = g.sort_values("ROI", ascending=False).iloc[0]
    best_roi_rows.append({
        "cost_per_contact": c,
        "value_per_response": v,
        "bestK_roi": int(g2["K"]),
        "max_ROI": float(g2["ROI"]),
        "profit_at_bestK_roi": float(g2["expected_profit"]),
        "expected_responses_at_bestK_roi": float(g2["expected_responses"]),
    })

best_roi_table = pd.DataFrame(best_roi_rows).sort_values(
    ["value_per_response", "cost_per_contact"]
)

print("\n=== BEST K NACH MAX ROI ===")
print(best_roi_table)

# ----------------------------
# 3) Profit fällt nach Peak? (Flag)
# ----------------------------
flags = []
for (c, v), g in st.groupby(["cost_per_contact", "value_per_response"], observed=False):
    g_sorted = g.sort_values("K").reset_index(drop=True)
    # finde Peak
    peak_idx = int(g_sorted["expected_profit"].values.argmax())
    peak_k = int(g_sorted.loc[peak_idx, "K"])
    # fällt es danach?
    after = g_sorted.loc[peak_idx+1:, "expected_profit"]
    falls_after_peak = bool((after < g_sorted.loc[peak_idx, "expected_profit"]).any()) if len(after) else False

    flags.append({
        "cost_per_contact": c,
        "value_per_response": v,
        "peakK_profit": peak_k,
        "falls_after_peak": falls_after_peak,
        "profit_at_peak": float(g_sorted.loc[peak_idx, "expected_profit"]),
        "profit_at_maxK": float(g_sorted["expected_profit"].iloc[-1]),
        "maxK_in_table": int(g_sorted["K"].iloc[-1]),
    })

profit_curve_flags = pd.DataFrame(flags).sort_values(["value_per_response", "cost_per_contact"])

print("\n=== PROFIT-KURVE CHECK ===")
print(profit_curve_flags)

print("\nObjekte bereitgestellt:")
print("- best_profit_table")
print("- best_roi_table")
print("- profit_curve_flags")


=== BEST K NACH MAX PROFIT ===
    cost_per_contact  value_per_response  bestK_profit  max_expected_profit  \
0               0.05                   5           300           374.457576   
4               0.20                   5           300           329.457576   
8               0.80                   5           150           187.550271   
1               0.05                  10           300           763.915152   
5               0.20                  10           300           718.915152   
9               0.80                  10           250           542.734093   
2               0.05                  25           300          1932.287881   
6               0.20                  25           300          1887.287881   
10              0.80                  25           300          1707.287881   
3               0.05                  50           300          3879.575762   
7               0.20                  50           300          3834.575762   
11              0.80 

In [33]:
# ============================================================
# Zelle 27 — Erweiterter Best-K Search (bis K_max) + echtes Profit-Optimum
# ============================================================
"""
Analyst-Notiz:
Bisher haben wir nur Ks bis 300 betrachtet. Dadurch ist "K=300" oft künstlich das Maximum.
Hier erweitern wir den Suchraum und finden das echte Optimum.

Wir nutzen Isotonic-Probas (kalibriert) auf Holdout.
K-Raster: 25er Schritte bis K_max (standard: len(holdout)=560)

Outputs:
- bestK_extended_profit (je Szenario echtes K*)
- profit_curve_extended (für Visual/Debug)
"""

import numpy as np
import pandas as pd

need = ["y_test_holdout_v2", "proba_best_cal_isotonic"]
missing = [x for x in need if x not in globals()]
if missing:
    raise RuntimeError("Fehlende Objekte: " + ", ".join(missing) + ". Bitte Zelle 23 ausführen.")

y_true = np.array(y_test_holdout_v2)
p_iso = np.array(proba_best_cal_isotonic)

# Sortierung nach Score
order = np.argsort(-p_iso)
p_sorted = p_iso[order]
y_sorted = y_true[order]

# Parameter: Szenarien wie zuvor
costs = [0.05, 0.20, 0.80]
values = [5, 10, 25, 50]

# K-Raster erweitern
K_max = len(p_sorted)   # 560 (Holdout)
Ks_ext = list(range(25, K_max + 1, 25))
if Ks_ext[-1] != K_max:
    Ks_ext.append(K_max)

rows = []
for k in Ks_ext:
    exp_resp = float(p_sorted[:k].sum())
    act_resp = int(y_sorted[:k].sum())

    for c in costs:
        for v in values:
            cost_total = k * c
            exp_value = exp_resp * v
            exp_profit = exp_value - cost_total
            roi = (exp_profit / cost_total) if cost_total > 0 else np.nan

            rows.append({
                "K": k,
                "cost_per_contact": c,
                "value_per_response": v,
                "expected_responses": exp_resp,
                "actual_responses(holdout)": act_resp,
                "expected_profit": exp_profit,
                "ROI": roi
            })

profit_curve_extended = pd.DataFrame(rows)

# Best-K pro Szenario
best_rows = []
for (c, v), g in profit_curve_extended.groupby(["cost_per_contact","value_per_response"], observed=False):
    g2 = g.sort_values("expected_profit", ascending=False).iloc[0]
    best_rows.append({
        "cost_per_contact": c,
        "value_per_response": v,
        "bestK_profit_ext": int(g2["K"]),
        "max_expected_profit_ext": float(g2["expected_profit"]),
        "ROI_at_bestK": float(g2["ROI"]),
        "expected_responses_at_bestK": float(g2["expected_responses"]),
    })

bestK_extended_profit = pd.DataFrame(best_rows).sort_values(["value_per_response","cost_per_contact"])

print("=== BEST K (EXTENDED) NACH MAX PROFIT ===")
print(bestK_extended_profit)

print("\nObjekte bereitgestellt:")
print("- profit_curve_extended")
print("- bestK_extended_profit")


=== BEST K (EXTENDED) NACH MAX PROFIT ===
    cost_per_contact  value_per_response  bestK_profit_ext  \
0               0.05                   5               425   
4               0.20                   5               325   
8               0.80                   5               175   
1               0.05                  10               475   
5               0.20                  10               375   
9               0.80                  10               250   
2               0.05                  25               475   
6               0.20                  25               425   
10              0.80                  25               350   
3               0.05                  50               475   
7               0.20                  50               475   
11              0.80                  50               400   

    max_expected_profit_ext  ROI_at_bestK  expected_responses_at_bestK  
0                387.191269     18.220766                    81.688254  
4    

In [34]:
# ============================================================
# Zelle 28 — Grenznutzen-Analyse (Marginal Gain) + Break-even K
# ============================================================
"""
Analyst-Notiz:
Hier berechnen wir den Grenznutzen pro zusätzlichem Kontaktblock.
Das ist die sauberste Begründung für "Stop bei K*".

Für jedes K (in 25er-Schritten) berechnen wir:
- marginal_expected_responses = E[Resp](K) - E[Resp](K-25)
- marginal_expected_value     = marginal_expected_responses * value_per_response
- marginal_cost               = 25 * cost_per_contact
- marginal_profit             = marginal_expected_value - marginal_cost

Break-even: erster Punkt, an dem marginal_profit <= 0 (ab da nicht mehr ausweiten).
"""

import numpy as np
import pandas as pd

if "profit_curve_extended" not in globals():
    raise RuntimeError("profit_curve_extended fehlt. Bitte Zelle 27 ausführen.")

pc = profit_curve_extended.copy()

# Wir analysieren für jede (cost, value) separat
marg_rows = []
breakeven_rows = []

for (c, v), g in pc.groupby(["cost_per_contact", "value_per_response"], observed=False):
    g = g.sort_values("K").reset_index(drop=True)

    # marginale Änderungen pro Schritt
    g["marginal_expected_responses"] = g["expected_responses"].diff()
    g["marginal_cost"] = g["K"].diff() * c
    g["marginal_value"] = g["marginal_expected_responses"] * v
    g["marginal_profit"] = g["marginal_value"] - g["marginal_cost"]

    # ersten Schritt diff ist NaN -> raus
    g2 = g.dropna(subset=["marginal_profit"]).copy()

    # Break-even: erstes K, wo marginal_profit <= 0
    be = g2[g2["marginal_profit"] <= 0]
    if len(be):
        be_k = int(be.iloc[0]["K"])
        be_mp = float(be.iloc[0]["marginal_profit"])
    else:
        be_k = int(g["K"].max())
        be_mp = float(g2["marginal_profit"].iloc[-1]) if len(g2) else np.nan

    breakeven_rows.append({
        "cost_per_contact": c,
        "value_per_response": v,
        "breakeven_K_first_nonpositive": be_k,
        "marginal_profit_at_breakeven": be_mp
    })

    # für Output-Tabelle
    out = g2[[
        "K", "expected_responses", "expected_profit",
        "marginal_expected_responses", "marginal_cost", "marginal_value", "marginal_profit"
    ]].copy()
    out["cost_per_contact"] = c
    out["value_per_response"] = v
    marg_rows.append(out)

marginal_table = pd.concat(marg_rows, ignore_index=True)
breakeven_table = pd.DataFrame(breakeven_rows).sort_values(["value_per_response","cost_per_contact"])

print("=== BREAK-EVEN (erster marginal_profit <= 0) ===")
print(breakeven_table)

print("\n=== BEISPIEL: cost=0.80, value=5 (dort kippt es früh) ===")
ex = marginal_table[(marginal_table["cost_per_contact"]==0.80) & (marginal_table["value_per_response"]==5)].copy()
with pd.option_context("display.max_rows", 200, "display.max_columns", 30):
    print(ex)

print("\nObjekte bereitgestellt:")
print("- marginal_table")
print("- breakeven_table")


=== BREAK-EVEN (erster marginal_profit <= 0) ===
    cost_per_contact  value_per_response  breakeven_K_first_nonpositive  \
0               0.05                   5                            450   
4               0.20                   5                            350   
8               0.80                   5                            200   
1               0.05                  10                            500   
5               0.20                  10                            400   
9               0.80                  10                            275   
2               0.05                  25                            500   
6               0.20                  25                            450   
10              0.80                  25                            375   
3               0.05                  50                            500   
7               0.20                  50                            500   
11              0.80                  50           

In [36]:
# ============================================================
# Zelle 29 (FIX) — Deployment: K* aus Holdout -> Kontaktquote -> K*_voll skalieren
#                 + stabiler Tie-Break + Score-Cutoff im Report
# ============================================================

import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV

# ------------------------------------------------------------
# 0) Voraussetzungen
# ------------------------------------------------------------
need = ["df", "X_model_safe", "best_model_pipe", "bestK_extended_profit", "y_test_holdout_v2"]
missing = [x for x in need if x not in globals()]
if missing:
    raise RuntimeError("Fehlende Objekte: " + ", ".join(missing) + ". Bitte Zelle 17 + 27 + Holdout-Setup prüfen.")

# ------------------------------------------------------------
# 1) Szenario-Parameter (anpassen)
# ------------------------------------------------------------
cost_per_contact = 0.80
value_per_response = 5
cal_method = "isotonic"   # oder "sigmoid"

# ------------------------------------------------------------
# 2) K* (Holdout) holen und auf Vollmenge skalieren
# ------------------------------------------------------------
bk = bestK_extended_profit.copy()

row = bk[
    (bk["cost_per_contact"] == cost_per_contact) &
    (bk["value_per_response"] == value_per_response)
]
if row.empty:
    raise RuntimeError("Szenario nicht in bestK_extended_profit gefunden.")

K_star_holdout = int(row.iloc[0]["bestK_profit_ext"])
n_holdout = int(len(y_test_holdout_v2))
contact_rate_star = K_star_holdout / n_holdout

n_full = int(len(df))
K_star_full = int(round(contact_rate_star * n_full))

print("=== SZENARIO (SCALE FIX) ===")
print("cost_per_contact:", cost_per_contact, "| value_per_response:", value_per_response, "| cal_method:", cal_method)
print("Holdout: n=", n_holdout, "| K*_holdout=", K_star_holdout, "| contact_rate=", round(contact_rate_star, 4))
print("Full:    n=", n_full,   "| K*_full=", K_star_full)
print()

# ------------------------------------------------------------
# 3) Kalibriertes Modell fitten (CV-intern) und Voll-Daten scoren
# ------------------------------------------------------------
y_full = pd.to_numeric(df["Antwort_Letzte_Kampagne"], errors="coerce").fillna(0).astype(int)
X_full = X_model_safe.copy()

calibrated = CalibratedClassifierCV(estimator=best_model_pipe, method=cal_method, cv=5)
calibrated.fit(X_full, y_full)

proba_full = calibrated.predict_proba(X_full)[:, 1]

# ------------------------------------------------------------
# 4) Scoring-DF bauen (stabil sortieren: score desc, ID asc)
# ------------------------------------------------------------
id_col = "ID" if "ID" in df.columns else None
ids = df[id_col].values if id_col else np.arange(len(df))

df_scoring_full = pd.DataFrame({
    "ID": ids,
    "score_calibrated": proba_full
})

# optional: Cluster/Segment übernehmen, falls im df vorhanden
for col in ["Cluster", "Segment"]:
    if col in df.columns:
        df_scoring_full[col] = df[col].values

# stabiler Tie-Breaker
df_scoring_full = df_scoring_full.sort_values(
    by=["score_calibrated", "ID"],
    ascending=[False, True]
).reset_index(drop=True)

df_scoring_full["rank"] = np.arange(1, len(df_scoring_full) + 1)
df_scoring_full["target_flag_topKstar"] = (df_scoring_full["rank"] <= K_star_full).astype(int)

# Score-Cutoff (Score an Rang K*)
score_cutoff = float(df_scoring_full.loc[K_star_full - 1, "score_calibrated"])

# ------------------------------------------------------------
# 5) Targetliste + Erwartungswerte
# ------------------------------------------------------------
df_target_topK = df_scoring_full.head(K_star_full).copy()

expected_responses = float(df_target_topK["score_calibrated"].sum())
budget_cost = float(K_star_full * cost_per_contact)
expected_value = float(expected_responses * value_per_response)
expected_profit = float(expected_value - budget_cost)
roi = (expected_profit / budget_cost) if budget_cost > 0 else np.nan

print("=== TOP-K* SUMMARY (FULL, kalibriert) ===")
print("K*_full:", K_star_full)
print("score_cutoff:", round(score_cutoff, 6))
print("expected_responses (sum score):", round(expected_responses, 3))
print("budget_cost:", round(budget_cost, 2))
print("expected_value:", round(expected_value, 2))
print("expected_profit:", round(expected_profit, 2))
print("ROI:", round(roi, 3))
print()

# ------------------------------------------------------------
# 6) Export
# ------------------------------------------------------------
full_scoring_path = "full_scoring_calibrated.csv"
target_path = f"targetliste_top{K_star_full}_calibrated.csv"
report_path = "kampagnen_report.txt"

df_scoring_full.to_csv(full_scoring_path, index=False, sep=";")
df_target_topK.to_csv(target_path, index=False, sep=";")

report_lines = [
    "KAMPAGNEN-REPORT (Deployment Export)",
    f"cal_method={cal_method}",
    f"cost_per_contact={cost_per_contact}",
    f"value_per_response={value_per_response}",
    f"holdout_n={n_holdout}",
    f"K_star_holdout={K_star_holdout}",
    f"contact_rate_star={contact_rate_star:.6f}",
    f"full_n={n_full}",
    f"K_star_full={K_star_full}",
    f"score_cutoff_at_K_star={score_cutoff:.6f}",
    f"expected_responses_sum_score={expected_responses:.6f}",
    f"budget_cost={budget_cost:.2f}",
    f"expected_value={expected_value:.2f}",
    f"expected_profit={expected_profit:.2f}",
    f"ROI={roi:.6f}",
    "",
    "Dateien:",
    f"- {full_scoring_path}",
    f"- {target_path}",
    f"- {report_path}",
]

with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print("=== EXPORT FERTIG (SCALE FIX) ===")
print("-", full_scoring_path)
print("-", target_path)
print("-", report_path)

print("\nPreview Targetliste (Top 10):")
print(df_target_topK.head(10))


=== SZENARIO (SCALE FIX) ===
cost_per_contact: 0.8 | value_per_response: 5 | cal_method: isotonic
Holdout: n= 560 | K*_holdout= 175 | contact_rate= 0.3125
Full:    n= 2240 | K*_full= 700

=== TOP-K* SUMMARY (FULL, kalibriert) ===
K*_full: 700
score_cutoff: 0.170905
expected_responses (sum score): 317.457
budget_cost: 560.0
expected_value: 1587.29
expected_profit: 1027.29
ROI: 1.834

=== EXPORT FERTIG (SCALE FIX) ===
- full_scoring_calibrated.csv
- targetliste_top700_calibrated.csv
- kampagnen_report.txt

Preview Targetliste (Top 10):
     ID  score_calibrated  rank  target_flag_topKstar
0   590          0.971429     1                     1
1  1772          0.971429     2                     1
2  1859          0.971429     3                     1
3  1891          0.971429     4                     1
4  2407          0.971429     5                     1
5  2715          0.971429     6                     1
6  5302          0.971429     7                     1
7  5547          0.971429   

In [38]:
# ============================================================
# Zelle 30 (FIX) — Segment/Cluster-Spalten finden + an df_scoring_full mergen
# ============================================================

import pandas as pd
import numpy as np

def _find_candidate_cols(cols, keywords):
    cols_lower = {c.lower(): c for c in cols}
    hits = []
    for c in cols:
        cl = c.lower()
        if any(k in cl for k in keywords):
            hits.append(c)
    return hits

def _pick_best_source_df():
    # Wir suchen das "beste" DF in globals(), das ID + (Cluster/Segment) enthält
    cand = []
    for name, obj in globals().items():
        if isinstance(obj, pd.DataFrame) and len(obj.columns) > 0:
            cols = list(obj.columns)
            has_id = "id" in [c.lower() for c in cols]
            if not has_id:
                continue
            cluster_hits = _find_candidate_cols(cols, ["cluster", "kmeans"])
            segment_hits = _find_candidate_cols(cols, ["segment", "label"])
            score = (len(cluster_hits) > 0) + (len(segment_hits) > 0) + (len(cluster_hits) + len(segment_hits))*0.01
            if score > 0:
                cand.append((score, name, obj, cluster_hits, segment_hits))
    if not cand:
        return None
    cand.sort(key=lambda x: x[0], reverse=True)
    return cand[0]  # best

def _ensure_id_colname(df_in):
    # normalisiere ID-Spaltenname auf "ID"
    cols = list(df_in.columns)
    id_cols = [c for c in cols if c.lower() == "id"]
    if not id_cols:
        raise KeyError("Keine ID-Spalte gefunden.")
    if id_cols[0] != "ID":
        df_in = df_in.rename(columns={id_cols[0]: "ID"})
    return df_in

print("=== DEBUG: SPALTEN-CHECK ===")
print("df_scoring_full cols (Auszug):", list(df_scoring_full.columns)[:30])
print("df cols (Auszug):", list(df.columns)[:30])
print()

# 1) Falls df_scoring_full Segment/Cluster schon hat: direkt reporten
need_cols = {"Cluster", "Segment"}
if need_cols.issubset(set(df_scoring_full.columns)):
    print("Segment/Cluster sind bereits in df_scoring_full vorhanden.")
else:
    # 2) Beste Quelle suchen (df oder andere DF aus globals)
    best = _pick_best_source_df()
    if best is None:
        print("Keine DataFrame-Quelle mit ID + (Cluster/Segment) gefunden.")
        print("Hinweis: Stelle sicher, dass du die Spalten nach KMeans/Labeling in df (oder ein DF) gespeichert hast.")
    else:
        score, name, src, cluster_hits, segment_hits = best
        print(f"Quelle gewählt: {name}")
        print("Cluster-Kandidaten:", cluster_hits[:10])
        print("Segment-Kandidaten:", segment_hits[:10])

        src2 = _ensure_id_colname(src.copy())

        # Kanonische Spalten bestimmen
        cluster_col = None
        segment_col = None

        # Cluster: bevorzuge exakt "Cluster" oder "cluster" oder kmeans
        pref_cluster = [c for c in src2.columns if c.lower() in ["cluster", "kmeans_cluster", "cluster_kmeans"]]
        if pref_cluster:
            cluster_col = pref_cluster[0]
        elif cluster_hits:
            cluster_col = cluster_hits[0]

        # Segment: bevorzuge exakt "Segment" oder "segment"
        pref_segment = [c for c in src2.columns if c.lower() in ["segment", "segment_label", "label_segment"]]
        if pref_segment:
            segment_col = pref_segment[0]
        elif segment_hits:
            segment_col = segment_hits[0]

        take_cols = ["ID"]
        rename_map = {}

        if cluster_col is not None:
            take_cols.append(cluster_col)
            rename_map[cluster_col] = "Cluster"

        if segment_col is not None:
            take_cols.append(segment_col)
            rename_map[segment_col] = "Segment"

        src_take = src2[take_cols].rename(columns=rename_map).drop_duplicates(subset=["ID"])

        # 3) Merge an df_scoring_full + df_target_topK
        df_scoring_full = df_scoring_full.merge(src_take, on="ID", how="left")
        df_target_topK = df_target_topK.merge(src_take, on="ID", how="left")

        print("\nNach Merge: df_scoring_full hat nun:", [c for c in ["Cluster","Segment"] if c in df_scoring_full.columns])
        print("Fehlende Werte (NaN) in Cluster:", int(df_scoring_full["Cluster"].isna().sum()) if "Cluster" in df_scoring_full.columns else "n/a")
        print("Fehlende Werte (NaN) in Segment:", int(df_scoring_full["Segment"].isna().sum()) if "Segment" in df_scoring_full.columns else "n/a")

# 4) Report-Funktion (wie vorher)
def group_report(df_all, df_tgt, group_col):
    if group_col not in df_all.columns:
        return None

    g_all = df_all.groupby(group_col, dropna=False).agg(
        n_all=("ID", "count"),
        mean_score_all=("score_calibrated", "mean"),
        exp_resp_all=("score_calibrated", "sum")
    ).reset_index()
    g_all["share_all"] = g_all["n_all"] / g_all["n_all"].sum()

    g_tgt = df_tgt.groupby(group_col, dropna=False).agg(
        n_tgt=("ID", "count"),
        mean_score_tgt=("score_calibrated", "mean"),
        exp_resp_tgt=("score_calibrated", "sum")
    ).reset_index()
    g_tgt["share_tgt"] = g_tgt["n_tgt"] / g_tgt["n_tgt"].sum()

    out = g_all.merge(g_tgt, on=group_col, how="left")
    out["n_tgt"] = out["n_tgt"].fillna(0).astype(int)
    out["share_tgt"] = out["share_tgt"].fillna(0.0)
    out["mean_score_tgt"] = out["mean_score_tgt"].fillna(0.0)
    out["exp_resp_tgt"] = out["exp_resp_tgt"].fillna(0.0)

    out["target_vs_all_ratio"] = np.where(out["share_all"] > 0, out["share_tgt"] / out["share_all"], np.nan)
    out = out.sort_values("target_vs_all_ratio", ascending=False)
    return out

print("\n=== SEGMENT/CLUSTER REPORT (Neu) ===")
seg_report = group_report(df_scoring_full, df_target_topK, "Segment")
clu_report = group_report(df_scoring_full, df_target_topK, "Cluster")

if seg_report is not None:
    print("\n--- SEGMENT REPORT ---")
    with pd.option_context("display.max_rows", 100, "display.max_columns", 40):
        print(seg_report)

if clu_report is not None:
    print("\n--- CLUSTER REPORT ---")
    with pd.option_context("display.max_rows", 100, "display.max_columns", 40):
        print(clu_report)

print("\nObjekte bereitgestellt:")
print("- df_scoring_full (aktualisiert)")
print("- df_target_topK (aktualisiert)")
print("- seg_report (oder None)")
print("- clu_report (oder None)")


=== DEBUG: SPALTEN-CHECK ===
df_scoring_full cols (Auszug): ['ID', 'score_calibrated', 'rank', 'target_flag_topKstar']
df cols (Auszug): ['ID', 'Geburtsjahr', 'Bildungsniveau', 'Familienstand', 'Einkommen', 'Kinder_zu_Hause', 'Teenager_zu_Hause', 'Datum_Kunde', 'Letzter_Kauf_Tage', 'Beschwerde', 'Ausgaben_Wein', 'Ausgaben_Obst', 'Ausgaben_Fleisch', 'Ausgaben_Fisch', 'Ausgaben_Süßigkeiten', 'Ausgaben_Gold', 'Anzahl_Rabattkäufe', 'Anzahl_Webkäufe', 'Anzahl_Katalogkäufe', 'Anzahl_Ladeneinkäufe', 'Anzahl_WebBesuche_Monat', 'Kampagne_1_Akzeptiert', 'Kampagne_2_Akzeptiert', 'Kampagne_3_Akzeptiert', 'Kampagne_4_Akzeptiert', 'Kampagne_5_Akzeptiert', 'Antwort_Letzte_Kampagne', 'Ausgaben_Gesamt', 'Kanal_Käufe_Gesamt', 'Flag_Ausgaben_ohne_Käufe']

Quelle gewählt: df_clustered
Cluster-Kandidaten: ['Cluster']
Segment-Kandidaten: ['Segment']

Nach Merge: df_scoring_full hat nun: ['Cluster', 'Segment']
Fehlende Werte (NaN) in Cluster: 0
Fehlende Werte (NaN) in Segment: 0

=== SEGMENT/CLUSTER REPORT (

In [39]:
# ============================================================
# Zelle 31 — QUALITY GATE / PLAUSI-CHECK (Segment & Budget View)
# ============================================================
# Ziel dieser Zelle:
# - Prüfen, ob die Targetliste operativ plausibel ist (nicht nur "Modell hat AUC")
# - Segment-Mix: Wer landet in der Targetliste und ist das erwartbar?
# - Budget-/ROI-Sicht: Wo entstehen erwartete Responses, Kosten und Profit?
# - Optionaler Reality-Check: Falls Ist-Labels verfügbar sind (Antwort_Letzte_Kampagne),
#   vergleichen wir erwartete vs tatsächliche Antwortquoten (nur zur Plausibilisierung).

import pandas as pd
import numpy as np

# ----------------------------
# 1) Parameter (Fallbacks)
# ----------------------------
# Wenn du die Parameter schon in einer vorherigen Zelle gesetzt hast, werden diese übernommen.
cost_per_contact = globals().get("cost_per_contact", 0.8)
value_per_response = globals().get("value_per_response", 5.0)
cal_method = globals().get("cal_method", "isotonic")

# Targetliste: df_target_topK ist die TopK*-Liste; df_scoring_full ist die Full-Scoring-Tabelle
df_all = df_scoring_full.copy()
df_tgt = df_target_topK.copy()

# Sicherstellen: ID als int (für saubere Merges/Groupbys)
df_all["ID"] = pd.to_numeric(df_all["ID"], errors="coerce")
df_tgt["ID"] = pd.to_numeric(df_tgt["ID"], errors="coerce")

# ----------------------------
# 2) Optional: Ist-Label anreichern (aus df oder aus vorhandenen Spalten)
# ----------------------------
label_col = None

# Kandidaten im Full-Scoring
for c in ["ist_antwort", "Antwort_Letzte_Kampagne", "antwort", "y", "target"]:
    if c in df_all.columns:
        label_col = c
        break

# Wenn nicht im Scoring: versuche aus Original-df zu mergen
if label_col is None and "df" in globals():
    if "Antwort_Letzte_Kampagne" in df.columns:
        tmp = df[["ID", "Antwort_Letzte_Kampagne"]].copy()
        tmp["ID"] = pd.to_numeric(tmp["ID"], errors="coerce")
        df_all = df_all.merge(tmp, on="ID", how="left")
        df_tgt = df_tgt.merge(tmp, on="ID", how="left")
        label_col = "Antwort_Letzte_Kampagne"

# Label als 0/1 numeric normalisieren (falls vorhanden)
if label_col is not None:
    df_all[label_col] = pd.to_numeric(df_all[label_col], errors="coerce")
    df_tgt[label_col] = pd.to_numeric(df_tgt[label_col], errors="coerce")

# ----------------------------
# 3) Basis-Kennzahlen (Full & Target)
# ----------------------------
n_all = len(df_all)
n_tgt = len(df_tgt)

base_rate = None
tgt_rate = None

if label_col is not None and df_all[label_col].notna().any():
    base_rate = float(df_all[label_col].mean())
    tgt_rate = float(df_tgt[label_col].mean()) if df_tgt[label_col].notna().any() else None

print("=== QUALITY GATE: BASIS ===")
print(f"cal_method: {cal_method}")
print(f"cost_per_contact: {cost_per_contact} | value_per_response: {value_per_response}")
print(f"Full:   n={n_all}")
print(f"Target: n={n_tgt} | contact_rate={n_tgt/n_all:.4f}")
if base_rate is not None:
    print(f"base_rate (Full, Ist): {base_rate:.4f}")
if tgt_rate is not None:
    print(f"target_rate (Targets, Ist): {tgt_rate:.4f}")
if base_rate is not None and tgt_rate is not None and base_rate > 0:
    print(f"lift (Targets vs Full): {tgt_rate/base_rate:.2f}")
print()

# ----------------------------
# 4) Segment/Cluster Quality Report
# ----------------------------
def build_quality_table(group_col: str) -> pd.DataFrame:
    if group_col not in df_all.columns or group_col not in df_tgt.columns:
        return None

    g_all = df_all.groupby(group_col, dropna=False).agg(
        n_all=("ID", "count"),
        mean_score_all=("score_calibrated", "mean"),
        exp_resp_all=("score_calibrated", "sum"),
    ).reset_index()
    g_all["share_all"] = g_all["n_all"] / g_all["n_all"].sum()

    g_tgt = df_tgt.groupby(group_col, dropna=False).agg(
        n_tgt=("ID", "count"),
        mean_score_tgt=("score_calibrated", "mean"),
        exp_resp_tgt=("score_calibrated", "sum"),
    ).reset_index()
    g_tgt["share_tgt"] = g_tgt["n_tgt"] / g_tgt["n_tgt"].sum()

    out = g_all.merge(g_tgt, on=group_col, how="left")
    out["n_tgt"] = out["n_tgt"].fillna(0).astype(int)
    out["share_tgt"] = out["share_tgt"].fillna(0.0)
    out["mean_score_tgt"] = out["mean_score_tgt"].fillna(0.0)
    out["exp_resp_tgt"] = out["exp_resp_tgt"].fillna(0.0)

    # Over/Under-Representation
    out["target_vs_all_ratio"] = np.where(out["share_all"] > 0, out["share_tgt"] / out["share_all"], np.nan)

    # Budget/Value/Profit (expected, score-basiert)
    out["cost_tgt"] = out["n_tgt"] * cost_per_contact
    out["value_tgt_expected"] = out["exp_resp_tgt"] * value_per_response
    out["profit_tgt_expected"] = out["value_tgt_expected"] - out["cost_tgt"]
    out["roi_tgt_expected"] = np.where(out["cost_tgt"] > 0, out["profit_tgt_expected"] / out["cost_tgt"], np.nan)

    # Optional: Ist-Rate je Gruppe (Full vs Target)
    if label_col is not None and df_all[label_col].notna().any():
        r_all = df_all.groupby(group_col, dropna=False)[label_col].mean().reset_index().rename(columns={label_col: "actual_rate_all"})
        r_tgt = df_tgt.groupby(group_col, dropna=False)[label_col].mean().reset_index().rename(columns={label_col: "actual_rate_tgt"})
        out = out.merge(r_all, on=group_col, how="left").merge(r_tgt, on=group_col, how="left")

        out["lift_actual_tgt_vs_all"] = np.where(
            (out["actual_rate_all"].notna()) & (out["actual_rate_all"] > 0),
            out["actual_rate_tgt"] / out["actual_rate_all"],
            np.nan
        )

        # Expected vs Actual (nur Plausi)
        # Sum actual in targets: "tatsächliche Responses" innerhalb der Targetliste (retrospektiv)
        a_tgt = df_tgt.groupby(group_col, dropna=False)[label_col].sum().reset_index().rename(columns={label_col: "actual_resp_tgt"})
        out = out.merge(a_tgt, on=group_col, how="left")
        out["actual_resp_tgt"] = out["actual_resp_tgt"].fillna(0.0)
        out["gap_expected_minus_actual_resp_tgt"] = out["exp_resp_tgt"] - out["actual_resp_tgt"]

    # Sortierung: operativ sinnvoll (wer dominiert die Targets / Profit)
    out = out.sort_values(["profit_tgt_expected", "target_vs_all_ratio"], ascending=False)
    return out

seg_quality = build_quality_table("Segment")
clu_quality = build_quality_table("Cluster")

print("=== QUALITY TABLE: SEGMENT ===")
if seg_quality is None:
    print("Segment-Spalte nicht vorhanden.")
else:
    with pd.option_context("display.max_rows", 100, "display.max_columns", 60):
        print(seg_quality)
print()

print("=== QUALITY TABLE: CLUSTER ===")
if clu_quality is None:
    print("Cluster-Spalte nicht vorhanden.")
else:
    with pd.option_context("display.max_rows", 100, "display.max_columns", 60):
        print(clu_quality)
print()

# ----------------------------
# 5) Operative Plausi: Top-Beispiele pro Segment (Targets)
# ----------------------------
print("=== PLAUSI SAMPLES: TOP 5 je Segment (Targets) ===")
if "Segment" not in df_tgt.columns:
    print("Keine Segment-Spalte in Targets vorhanden.")
else:
    cols_show = ["ID", "Segment", "score_calibrated"]
    if label_col is not None and label_col in df_tgt.columns:
        cols_show.append(label_col)

    df_tgt_sorted = df_tgt.sort_values("score_calibrated", ascending=False).copy()

    for seg in df_tgt_sorted["Segment"].dropna().unique():
        sub = df_tgt_sorted[df_tgt_sorted["Segment"] == seg].head(5)
        print()
        print(f"Segment: {seg}")
        with pd.option_context("display.max_rows", 20, "display.max_columns", 20):
            print(sub[cols_show])
print()

# ----------------------------
# 6) Kurzfazit (textuell, datengetrieben)
# ----------------------------
def _fmt(x):
    try:
        return f"{x:.4f}"
    except Exception:
        return str(x)

print("=== KURZFAZIT (DATA-DRIVEN) ===")
if seg_quality is not None:
    # Dominantes Segment in Targets (nach Anteil)
    dominant = seg_quality.sort_values("share_tgt", ascending=False).iloc[0]
    print(f"Dominantes Segment in Targets: {dominant['Segment']} | share_tgt={_fmt(dominant['share_tgt'])} | ratio={_fmt(dominant['target_vs_all_ratio'])}")

    # Segment mit stärkstem expected Profit
    best_profit = seg_quality.sort_values("profit_tgt_expected", ascending=False).iloc[0]
    print(f"Höchster erwarteter Profit (Targets): {best_profit['Segment']} | profit_expected={best_profit['profit_tgt_expected']:.2f} | ROI={best_profit['roi_tgt_expected']:.2f}")

    # Optionaler Ist-Lift
    if "lift_actual_tgt_vs_all" in seg_quality.columns and seg_quality["lift_actual_tgt_vs_all"].notna().any():
        best_lift = seg_quality.sort_values("lift_actual_tgt_vs_all", ascending=False).iloc[0]
        print(f"Höchster Ist-Lift (Targets vs All): {best_lift['Segment']} | lift_actual={_fmt(best_lift['lift_actual_tgt_vs_all'])}")
else:
    print("Kein Segment-Report vorhanden. (Segment-Spalte fehlt)")

print()
print("Objekte bereitgestellt:")
print("- seg_quality (oder None)")
print("- clu_quality (oder None)")
print("- df_all (angereichert)")
print("- df_tgt (angereichert)")


=== QUALITY GATE: BASIS ===
cal_method: isotonic
cost_per_contact: 0.8 | value_per_response: 5
Full:   n=2240
Target: n=700 | contact_rate=0.3125
base_rate (Full, Ist): 0.1491
target_rate (Targets, Ist): 0.4771
lift (Targets vs Full): 3.20

=== QUALITY TABLE: SEGMENT ===
                             Segment  n_all  mean_score_all  exp_resp_all  \
0   Segment A: High-Value / Frequent   1109        0.225689    250.288699   
1  Segment D: Low-Value / Infrequent   1131        0.104802    118.531036   

   share_all  n_tgt  mean_score_tgt  exp_resp_tgt  share_tgt  \
0   0.495089    476        0.464627    221.162520       0.68   
1   0.504911    224        0.429887     96.294765       0.32   

   target_vs_all_ratio  cost_tgt  value_tgt_expected  profit_tgt_expected  \
0             1.373490     380.8         1105.812598           725.012598   
1             0.633775     179.2          481.473824           302.273824   

   roi_tgt_expected  actual_rate_all  actual_rate_tgt  lift_actual_tgt_

In [40]:
# ============================================================
# Zelle 32 — Kampagnen-Policy (2-Wellen) + Export (Wave1/Wave2)
# ============================================================
"""
Analyst-Notiz:
Aus Zelle 31 wissen wir:
- Segment A dominiert Profit/Volumen (stabil, planbar)
- Segment D zeigt sehr hohen Ist-Lift (Hidden Gems), obwohl es insgesamt "low-value" ist

Policy: 2-Wellen-Kampagne
- Wave 1: Fokus Profit/Stabilität (primär Segment A)
- Wave 2: Fokus Exploration/Lift (gezielt Segment D)
=> Ergebnis: zwei Targetlisten + Report

Wichtig:
- Wir verändern kein Modell mehr.
- Wir nutzen nur df_scoring_full (kalibrierter Score), plus Segment/Cluster-Merge aus Zelle 30.
"""

import pandas as pd
import numpy as np

# ----------------------------
# 0) Voraussetzungen
# ----------------------------
need = ["df_scoring_full", "df_target_topK", "K_star_full"]
missing = [x for x in need if x not in globals()]
if missing:
    raise RuntimeError("Fehlende Objekte: " + ", ".join(missing) + ". Bitte Zelle 29-31 ausführen.")

df_all = df_scoring_full.copy()

if "Segment" not in df_all.columns:
    raise RuntimeError("Segment fehlt in df_scoring_full. Bitte Zelle 30 (Merge Fix) ausführen.")

# ----------------------------
# 1) Policy-Parameter (anpassen)
# ----------------------------
K_total = int(K_star_full)          # Gesamtumfang Kampagne
share_wave2 = 0.30                 # Anteil für Wave 2 (Exploration/Lift)
min_share_D_wave2 = 0.80           # In Wave2 soll mind. dieser Anteil Segment D sein
stable_tie_break = True            # Reproduzierbarkeit bei gleichen Scores

# Optional: Wenn du harte Budgets fährst, könntest du K_total hier überschreiben:
# K_total = 500

K_wave2 = int(round(K_total * share_wave2))
K_wave1 = K_total - K_wave2

print("=== POLICY PARAMS ===")
print("K_total:", K_total, "| K_wave1:", K_wave1, "| K_wave2:", K_wave2)
print("share_wave2:", share_wave2, "| min_share_D_wave2:", min_share_D_wave2)
print()

# ----------------------------
# 2) Sortierung: Score desc + stabiler Tie-Break (ID asc)
# ----------------------------
sort_cols = ["score_calibrated"]
asc = [False]
if stable_tie_break and "ID" in df_all.columns:
    sort_cols.append("ID")
    asc.append(True)

df_all = df_all.sort_values(sort_cols, ascending=asc).reset_index(drop=True)

# ----------------------------
# 3) Wave 1: "Profit/Stabilität" (primär Segment A)
# - wir nehmen die besten Scores innerhalb Segment A zuerst
# - falls Segment A nicht reicht (sollte es aber), füllen wir mit den nächsten besten overall auf
# ----------------------------
df_A = df_all[df_all["Segment"].astype(str).str.contains("Segment A", na=False)].copy()
df_not_A = df_all[~df_all.index.isin(df_A.index)].copy()

wave1 = df_A.head(K_wave1).copy()

if len(wave1) < K_wave1:
    # Fallback: Auffüllen mit den besten aus dem Rest
    need_fill = K_wave1 - len(wave1)
    fill = df_not_A.head(need_fill).copy()
    wave1 = pd.concat([wave1, fill], ignore_index=True)

wave1["wave"] = "WAVE_1"
wave1["policy_reason"] = "Profit/Stabilität (primär Segment A, Top Score)"

# ----------------------------
# 4) Wave 2: "Exploration/Lift" (gezielt Segment D)
# - wir schließen Wave1-Kunden aus
# - wir wollen in Wave2 mind. min_share_D_wave2 aus Segment D
# - Rest füllen wir mit besten Scores aus übrigen Kunden (egal welches Segment)
# ----------------------------
used_ids = set(wave1["ID"].tolist())
df_remaining = df_all[~df_all["ID"].isin(used_ids)].copy()

df_D_rem = df_remaining[df_remaining["Segment"].astype(str).str.contains("Segment D", na=False)].copy()
df_other_rem = df_remaining[~df_remaining.index.isin(df_D_rem.index)].copy()

K_D_wave2 = int(round(K_wave2 * min_share_D_wave2))
K_other_wave2 = K_wave2 - K_D_wave2

w2_D = df_D_rem.head(K_D_wave2).copy()
w2_other = df_other_rem.head(K_other_wave2).copy()

wave2 = pd.concat([w2_D, w2_other], ignore_index=True)

# Falls Segment D im Remaining nicht reicht: Rest aus Remaining auffüllen
if len(wave2) < K_wave2:
    need_fill = K_wave2 - len(wave2)
    df_fill_pool = df_remaining[~df_remaining["ID"].isin(set(wave2["ID"].tolist()))].copy()
    wave2 = pd.concat([wave2, df_fill_pool.head(need_fill)], ignore_index=True)

wave2 = wave2.head(K_wave2).copy()
wave2["wave"] = "WAVE_2"
wave2["policy_reason"] = f"Exploration/Lift (>= {min_share_D_wave2:.0%} Segment D, Top Score)"

# ----------------------------
# 5) Finales Policy-Targeting
# ----------------------------
policy_targets = pd.concat([wave1, wave2], ignore_index=True)

# Re-rank innerhalb der Policy-Liste für Export (Wave1 zuerst, dann Wave2)
policy_targets["policy_rank"] = np.arange(1, len(policy_targets) + 1)

# ----------------------------
# 6) Mini-Report: Mix + Expected Responses
# ----------------------------
def _seg_summary(df_in, label):
    out = df_in.groupby("Segment", dropna=False).agg(
        kunden=("ID", "count"),
        mean_score=("score_calibrated", "mean"),
        exp_responses=("score_calibrated", "sum")
    ).reset_index()
    out["share"] = out["kunden"] / out["kunden"].sum()
    out["label"] = label
    return out

sum_w1 = _seg_summary(wave1, "WAVE_1")
sum_w2 = _seg_summary(wave2, "WAVE_2")
sum_all = _seg_summary(policy_targets, "TOTAL")

seg_policy_report = pd.concat([sum_all, sum_w1, sum_w2], ignore_index=True)

print("=== POLICY MIX REPORT (Segment) ===")
with pd.option_context("display.max_rows", 50, "display.max_columns", 20):
    print(seg_policy_report)

print("\n=== EXPECTED (Score-based) ===")
print("WAVE_1 expected_responses:", round(float(wave1["score_calibrated"].sum()), 3))
print("WAVE_2 expected_responses:", round(float(wave2["score_calibrated"].sum()), 3))
print("TOTAL  expected_responses:", round(float(policy_targets["score_calibrated"].sum()), 3))
print()

# ----------------------------
# 7) Export
# ----------------------------
wave1_path = f"targetliste_wave1_top{len(wave1)}.csv"
wave2_path = f"targetliste_wave2_top{len(wave2)}.csv"
policy_path = f"targetliste_policy_total{len(policy_targets)}.csv"
policy_report_path = "kampagnen_policy_report.txt"

# Export mit ; (deutsches Excel-freundlich)
wave1.to_csv(wave1_path, index=False, sep=";")
wave2.to_csv(wave2_path, index=False, sep=";")
policy_targets.to_csv(policy_path, index=False, sep=";")

lines = [
    "KAMPAGNEN-POLICY REPORT (2-Wellen)",
    f"K_total={K_total}",
    f"K_wave1={len(wave1)}",
    f"K_wave2={len(wave2)}",
    f"share_wave2={share_wave2}",
    f"min_share_D_wave2={min_share_D_wave2}",
    "",
    f"WAVE_1 expected_responses_sum_score={float(wave1['score_calibrated'].sum()):.6f}",
    f"WAVE_2 expected_responses_sum_score={float(wave2['score_calibrated'].sum()):.6f}",
    f"TOTAL  expected_responses_sum_score={float(policy_targets['score_calibrated'].sum()):.6f}",
    "",
    "Export-Dateien:",
    f"- {wave1_path}",
    f"- {wave2_path}",
    f"- {policy_path}",
    f"- {policy_report_path}",
]

with open(policy_report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("=== EXPORT POLICY FERTIG ===")
print("-", wave1_path)
print("-", wave2_path)
print("-", policy_path)
print("-", policy_report_path)

print("\nPreview Policy Targets (Top 10):")
print(policy_targets.head(10)[["ID", "Segment", "score_calibrated", "wave", "policy_reason"]])


=== POLICY PARAMS ===
K_total: 700 | K_wave1: 490 | K_wave2: 210
share_wave2: 0.3 | min_share_D_wave2: 0.8

=== POLICY MIX REPORT (Segment) ===
                             Segment  kunden  mean_score  exp_responses  \
0   Segment A: High-Value / Frequent     532    0.431336     229.470843   
1  Segment D: Low-Value / Infrequent     168    0.505340      84.897146   
2   Segment A: High-Value / Frequent     490    0.455968     223.424117   
3   Segment A: High-Value / Frequent      42    0.143970       6.046726   
4  Segment D: Low-Value / Infrequent     168    0.505340      84.897146   

   share   label  
0   0.76   TOTAL  
1   0.24   TOTAL  
2   1.00  WAVE_1  
3   0.20  WAVE_2  
4   0.80  WAVE_2  

=== EXPECTED (Score-based) ===
WAVE_1 expected_responses: 223.424
WAVE_2 expected_responses: 90.944
TOTAL  expected_responses: 314.368

=== EXPORT POLICY FERTIG ===
- targetliste_wave1_top490.csv
- targetliste_wave2_top210.csv
- targetliste_policy_total700.csv
- kampagnen_policy_report.txt

In [41]:
# ============================================================
# Zelle 33 — Mini A/B-Test-Plan für Wave1 vs Wave2 (Messkonzept)
# ============================================================
"""
Analyst-Notiz:
Wir haben zwei Target-Wellen:
- Wave 1: Profit/Stabilität (primär Segment A)
- Wave 2: Exploration/Lift (gezielt Segment D)

Jetzt definieren wir einen messbaren Testplan:
1) Welche KPIs messen wir?
2) Wie vergleichen wir Wave1 vs Wave2?
3) Was sind sinnvolle Stop/Go-Regeln?
4) Optional: einfache Signifikanz (2-Proportionen-Test) für Response-Rate.
"""

import numpy as np
import pandas as pd
from math import sqrt

# ----------------------------
# 0) Voraussetzungen
# ----------------------------
need = ["wave1", "wave2", "cost_per_contact", "value_per_response"]
missing = [x for x in need if x not in globals()]
if missing:
    raise RuntimeError("Fehlende Objekte: " + ", ".join(missing) + ". Bitte Zelle 32 ausführen.")

# Optionales Ist-Label (falls historisch vorhanden)
label_col = None
for c in ["Antwort_Letzte_Kampagne", "ist_antwort"]:
    if c in wave1.columns and c in wave2.columns:
        label_col = c
        break

# ----------------------------
# 1) KPI-Definitionen (operativ)
# ----------------------------
# Response-Rate: responses / contacts
# CPA (Cost per Acquisition/Response): Kosten / Responses
# Expected Profit: (Responses * value_per_response) - (Contacts * cost_per_contact)
# ROI: Profit / Kosten

def kpi_table(df_in, label="WAVE"):
    n = len(df_in)
    exp_resp = float(df_in["score_calibrated"].sum())  # erwartete Responses (score-basiert)
    cost = n * cost_per_contact
    exp_value = exp_resp * value_per_response
    exp_profit = exp_value - cost
    exp_roi = exp_profit / cost if cost > 0 else np.nan

    out = {
        "label": label,
        "contacts": n,
        "expected_responses": exp_resp,
        "expected_value": exp_value,
        "cost_total": cost,
        "expected_profit": exp_profit,
        "expected_ROI": exp_roi
    }

    # Optional: tatsächliche Responses aus historischem Label (nur Plausi)
    if label_col is not None and df_in[label_col].notna().any():
        actual_resp = float(df_in[label_col].sum())
        actual_rate = float(df_in[label_col].mean())
        actual_value = actual_resp * value_per_response
        actual_profit = actual_value - cost
        actual_roi = actual_profit / cost if cost > 0 else np.nan

        out.update({
            "actual_responses_hist": actual_resp,
            "actual_rate_hist": actual_rate,
            "actual_profit_hist": actual_profit,
            "actual_ROI_hist": actual_roi
        })

    return out

kpi_w1 = kpi_table(wave1, "WAVE_1")
kpi_w2 = kpi_table(wave2, "WAVE_2")
kpi_total = kpi_table(pd.concat([wave1, wave2], ignore_index=True), "TOTAL")

kpi_df = pd.DataFrame([kpi_w1, kpi_w2, kpi_total])

print("=== KPI TABLE (Expected + optional Hist-Actual) ===")
with pd.option_context("display.max_columns", 50):
    print(kpi_df)
print()

# ----------------------------
# 2) Test-Plan (Textbaustein)
# ----------------------------
plan_lines = [
    "A/B-TEST-PLAN (Wave1 vs Wave2)",
    "",
    "Hypothesen:",
    "- H1: Wave1 hat stabilere, planbare Response-Rate und Profit (Segment A-lastig).",
    "- H2: Wave2 hat höheren Lift/Response-Rate in Segment D (Exploration), aber evtl. andere Profitstruktur.",
    "",
    "KPIs (Primär):",
    "- Response-Rate = Responses / Kontakte",
    "- Profit = (Responses * value_per_response) - (Kontakte * cost_per_contact)",
    "- ROI = Profit / Kosten",
    "",
    "KPIs (Sekundär):",
    "- CPA = Kosten / Responses",
    "- Segment-Mix in Responses (wer antwortet?)",
    "",
    "Stop/Go (praktisch):",
    "- Wenn Wave2 nach z.B. 20% Ausspielung deutlich schlechtere Profit/ROI zeigt -> Budget in Wave1 verschieben.",
    "- Wenn Wave2 deutlich bessere Response-Rate/ROI zeigt -> Wave2-Logik skalieren (mehr D-Targets).",
    "",
    "Messung:",
    "- Beide Wellen müssen das gleiche Creative/Offer-Setup haben (sonst verzerrt).",
    "- Responses sauber tracken (ID, wave, timestamp, response_flag).",
]

print("=== TEST-PLAN (Text) ===")
print("\n".join(plan_lines))
print()

# ----------------------------
# 3) Optional: einfache Signifikanz für Response-Rate (2-Proportionen-Test)
# Nur möglich, wenn wir echte Kampagnen-Responses haben.
# Hier rechnen wir eine Schablone vor: du trägst später observed Responses ein.
# ----------------------------
print("=== OPTIONAL: SIGNIFIKANZ-SCHABLONE (2-Proportionen) ===")
print("Trage nach Kampagne observed_responses_wave1 / wave2 ein und rechne den z-Score.")
print()

def two_prop_ztest(x1, n1, x2, n2):
    # pooled proportion
    p_pool = (x1 + x2) / (n1 + n2)
    se = sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
    if se == 0:
        return np.nan
    z = (x1/n1 - x2/n2) / se
    return z

# Platzhalter (nach Kampagne ersetzen)
observed_responses_wave1 = None
observed_responses_wave2 = None

if observed_responses_wave1 is not None and observed_responses_wave2 is not None:
    z = two_prop_ztest(observed_responses_wave1, len(wave1), observed_responses_wave2, len(wave2))
    print(f"z-Score: {z:.3f} (|z|>1.96 ~ signifikant bei 5%)")
else:
    print("Noch keine observed Responses gesetzt -> nur Vorlage.")

print("\nObjekte bereitgestellt:")
print("- kpi_df")
print("- plan_lines")


=== KPI TABLE (Expected + optional Hist-Actual) ===
    label  contacts  expected_responses  expected_value  cost_total  \
0  WAVE_1       490          223.424117     1117.120583       392.0   
1  WAVE_2       210           90.943872      454.719361       168.0   
2   TOTAL       700          314.367989     1571.839943       560.0   

   expected_profit  expected_ROI  
0       725.120583      1.849797  
1       286.719361      1.706663  
2      1011.839943      1.806857  

=== TEST-PLAN (Text) ===
A/B-TEST-PLAN (Wave1 vs Wave2)

Hypothesen:
- H1: Wave1 hat stabilere, planbare Response-Rate und Profit (Segment A-lastig).
- H2: Wave2 hat höheren Lift/Response-Rate in Segment D (Exploration), aber evtl. andere Profitstruktur.

KPIs (Primär):
- Response-Rate = Responses / Kontakte
- Profit = (Responses * value_per_response) - (Kontakte * cost_per_contact)
- ROI = Profit / Kosten

KPIs (Sekundär):
- CPA = Kosten / Responses
- Segment-Mix in Responses (wer antwortet?)

Stop/Go (praktisch):
-

In [42]:
# ============================================================
# Zelle 34 — FINAL REPORT Export (kampagnen_final_report.txt)
# ============================================================
"""
Analyst-Notiz:
Diese Zelle fasst die wichtigsten Ergebnisse des Projekts in einem Report zusammen,
damit das Ergebnis "prüfungs-/abgabefähig" dokumentiert ist.

Output:
- kampagnen_final_report.txt
"""

import os
import pandas as pd
import numpy as np
from datetime import datetime

# ----------------------------
# 0) Voraussetzungen
# ----------------------------
need = [
    "df_scoring_full", "df_target_topK",
    "seg_quality", "kpi_df",
    "cost_per_contact", "value_per_response", "cal_method"
]
missing = [x for x in need if x not in globals()]
if missing:
    raise RuntimeError("Fehlende Objekte: " + ", ".join(missing) + ". Bitte Zelle 29-33 ausführen.")

# Optional vorhandene Objekte
K_star_full = globals().get("K_star_full", None)
score_cutoff = globals().get("score_cutoff", None)
best_model_name = globals().get("best_model_name", "BestModel")
wave1_path = globals().get("wave1_path", "targetliste_wave1_top490.csv")
wave2_path = globals().get("wave2_path", "targetliste_wave2_top210.csv")
policy_path = globals().get("policy_path", "targetliste_policy_total700.csv")
policy_report_path = globals().get("policy_report_path", "kampagnen_policy_report.txt")

# Label-Info (falls vorhanden)
label_col = None
for c in ["Antwort_Letzte_Kampagne", "ist_antwort"]:
    if c in df_scoring_full.columns:
        label_col = c
        break
if label_col is None and "df" in globals() and "Antwort_Letzte_Kampagne" in df.columns:
    label_col = "Antwort_Letzte_Kampagne"

# ----------------------------
# 1) Kennzahlen (aus existierenden Ergebnissen)
# ----------------------------
n_all = len(df_scoring_full)
n_tgt = len(df_target_topK)
contact_rate = n_tgt / n_all

base_rate = None
tgt_rate = None
lift = None

if label_col is not None:
    # Falls Label im Scoring fehlt, versuchen wir minimal aus df zu mergen
    if label_col not in df_scoring_full.columns and "df" in globals():
        tmp = df[["ID", label_col]].copy()
        tmp["ID"] = pd.to_numeric(tmp["ID"], errors="coerce")
        tmp[label_col] = pd.to_numeric(tmp[label_col], errors="coerce")
        df_scoring_tmp = df_scoring_full.merge(tmp, on="ID", how="left")
        df_target_tmp = df_target_topK.merge(tmp, on="ID", how="left")
        base_rate = float(df_scoring_tmp[label_col].mean())
        tgt_rate = float(df_target_tmp[label_col].mean())
    else:
        base_rate = float(pd.to_numeric(df_scoring_full[label_col], errors="coerce").mean())
        tgt_rate = float(pd.to_numeric(df_target_topK[label_col], errors="coerce").mean())

    if base_rate and base_rate > 0:
        lift = tgt_rate / base_rate

# Segment-Kurzwerte
seg_tbl = seg_quality.copy() if seg_quality is not None else None

# KPI Tabellenwerte
kpi_out = kpi_df.copy()

# ----------------------------
# 2) Report-Text bauen
# ----------------------------
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

lines = []
lines.append("KAMPAGNEN-ABSCHLUSSREPORT")
lines.append("=" * 60)
lines.append(f"Zeitpunkt: {now_str}")
lines.append("")
lines.append("1) SETUP")
lines.append("-" * 60)
lines.append(f"Calibration-Methode: {cal_method}")
lines.append(f"Kosten pro Kontakt: {cost_per_contact}")
lines.append(f"Wert pro Response:  {value_per_response}")
if K_star_full is not None:
    lines.append(f"K* (Full): {K_star_full}")
if score_cutoff is not None:
    lines.append(f"Score-Cutoff: {score_cutoff}")
lines.append(f"Modell (Name): {best_model_name}")
lines.append("")
lines.append("2) BASIS & LIFT (Quality Gate)")
lines.append("-" * 60)
lines.append(f"Full n={n_all}")
lines.append(f"Targets n={n_tgt} | contact_rate={contact_rate:.4f}")
if base_rate is not None:
    lines.append(f"Base-Rate (Full, Ist):   {base_rate:.4f}")
if tgt_rate is not None:
    lines.append(f"Target-Rate (Targets):   {tgt_rate:.4f}")
if lift is not None:
    lines.append(f"Lift (Targets vs Full):  {lift:.2f}")
lines.append("")
lines.append("3) SEGMENT-INSIGHTS (Targets vs Gesamt)")
lines.append("-" * 60)

if seg_tbl is not None:
    # kurze Tabelle als Text
    show_cols = [
        "Segment","n_all","share_all","n_tgt","share_tgt","target_vs_all_ratio",
        "exp_resp_tgt","profit_tgt_expected","roi_tgt_expected"
    ]
    # Optional Ist-Spalten
    for c in ["actual_rate_all","actual_rate_tgt","lift_actual_tgt_vs_all"]:
        if c in seg_tbl.columns:
            show_cols.append(c)

    seg_view = seg_tbl[show_cols].copy()

    # Pretty formatting
    def fmt_df(df_in):
        df2 = df_in.copy()
        for c in df2.columns:
            if c in ["share_all","share_tgt","target_vs_all_ratio","roi_tgt_expected","actual_rate_all","actual_rate_tgt","lift_actual_tgt_vs_all"]:
                df2[c] = df2[c].astype(float).map(lambda x: f"{x:.4f}")
            if c in ["exp_resp_tgt","profit_tgt_expected"]:
                df2[c] = df2[c].astype(float).map(lambda x: f"{x:.2f}")
        return df2

    seg_view = fmt_df(seg_view)
    lines.append(seg_view.to_string(index=False))
else:
    lines.append("Kein Segment-Report vorhanden.")

lines.append("")
lines.append("4) POLICY (2-Wellen) — Expected KPIs")
lines.append("-" * 60)
lines.append(kpi_out.to_string(index=False))
lines.append("")
lines.append("5) TESTPLAN (Kurz)")
lines.append("-" * 60)
if "plan_lines" in globals():
    lines.extend(plan_lines)
else:
    lines.append("Kein Testplan-Text verfügbar (plan_lines fehlt).")

lines.append("")
lines.append("6) EXPORTS")
lines.append("-" * 60)
lines.append("Targetlisten & Reports:")
lines.append(f"- {wave1_path}")
lines.append(f"- {wave2_path}")
lines.append(f"- {policy_path}")
lines.append(f"- {policy_report_path}")
lines.append("")
lines.append("Hinweis:")
lines.append("Expected Responses basieren auf kalibriertem Score (Summe der Wahrscheinlichkeiten).")
lines.append("Für eine echte Kampagne: Responses tracken und Wave1/Wave2 getrennt evaluieren.")

# ----------------------------
# 3) Schreiben
# ----------------------------
final_report_path = "kampagnen_final_report.txt"
with open(final_report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("=== FINAL REPORT EXPORT ===")
print("-", final_report_path)

print("\nPreview (erste 40 Zeilen):")
preview = "\n".join(lines[:40])
print(preview)

print("\nObjekte bereitgestellt:")
print("- final_report_path")


=== FINAL REPORT EXPORT ===
- kampagnen_final_report.txt

Preview (erste 40 Zeilen):
KAMPAGNEN-ABSCHLUSSREPORT
Zeitpunkt: 2026-01-29 18:36:23

1) SETUP
------------------------------------------------------------
Calibration-Methode: isotonic
Kosten pro Kontakt: 0.8
Wert pro Response:  5
K* (Full): 700
Score-Cutoff: 0.17090521442495127
Modell (Name): RandomForest_SAFE

2) BASIS & LIFT (Quality Gate)
------------------------------------------------------------
Full n=2240
Targets n=700 | contact_rate=0.3125
Base-Rate (Full, Ist):   0.1491
Target-Rate (Targets):   0.4771
Lift (Targets vs Full):  3.20

3) SEGMENT-INSIGHTS (Targets vs Gesamt)
------------------------------------------------------------
                          Segment  n_all share_all  n_tgt share_tgt target_vs_all_ratio exp_resp_tgt profit_tgt_expected roi_tgt_expected actual_rate_all actual_rate_tgt lift_actual_tgt_vs_all
 Segment A: High-Value / Frequent   1109    0.4951    476    0.6800              1.3735       221.1

In [43]:
# ============================================================
# Zelle 34 — FINAL REPORT Export (kampagnen_final_report.txt)
# ============================================================
"""
Analyst-Notiz:
Diese Zelle fasst die wichtigsten Ergebnisse des Projekts in einem Report zusammen,
damit das Ergebnis "prüfungs-/abgabefähig" dokumentiert ist.

Output:
- kampagnen_final_report.txt
"""

import pandas as pd
import numpy as np
from datetime import datetime

# ----------------------------
# 0) Voraussetzungen
# ----------------------------
need = [
    "df_scoring_full", "df_target_topK",
    "seg_quality", "kpi_df",
    "cost_per_contact", "value_per_response", "cal_method"
]
missing = [x for x in need if x not in globals()]
if missing:
    raise RuntimeError("Fehlende Objekte: " + ", ".join(missing) + ". Bitte Zelle 29-33 ausführen.")

# Optional vorhandene Objekte (falls in vorherigen Zellen gesetzt)
K_star_full = globals().get("K_star_full", None)
score_cutoff = globals().get("score_cutoff", None)
best_model_name = globals().get("best_model_name", "BestModel")

wave1_path = globals().get("wave1_path", "targetliste_wave1_top490.csv")
wave2_path = globals().get("wave2_path", "targetliste_wave2_top210.csv")
policy_path = globals().get("policy_path", "targetliste_policy_total700.csv")
policy_report_path = globals().get("policy_report_path", "kampagnen_policy_report.txt")

# Label-Info (falls vorhanden)
label_col = None
for c in ["Antwort_Letzte_Kampagne", "ist_antwort"]:
    if c in df_scoring_full.columns:
        label_col = c
        break
if label_col is None and "df" in globals() and "Antwort_Letzte_Kampagne" in df.columns:
    label_col = "Antwort_Letzte_Kampagne"

# ----------------------------
# 1) Kennzahlen (Quality Gate)
# ----------------------------
n_all = len(df_scoring_full)
n_tgt = len(df_target_topK)
contact_rate = n_tgt / n_all

base_rate = None
tgt_rate = None
lift = None

if label_col is not None:
    # Falls Label im Scoring fehlt, versuchen wir minimal aus df zu mergen
    if label_col not in df_scoring_full.columns and "df" in globals():
        tmp = df[["ID", label_col]].copy()
        tmp["ID"] = pd.to_numeric(tmp["ID"], errors="coerce")
        tmp[label_col] = pd.to_numeric(tmp[label_col], errors="coerce")
        df_scoring_tmp = df_scoring_full.merge(tmp, on="ID", how="left")
        df_target_tmp = df_target_topK.merge(tmp, on="ID", how="left")
        base_rate = float(df_scoring_tmp[label_col].mean())
        tgt_rate = float(df_target_tmp[label_col].mean())
    else:
        base_rate = float(pd.to_numeric(df_scoring_full[label_col], errors="coerce").mean())
        tgt_rate = float(pd.to_numeric(df_target_topK[label_col], errors="coerce").mean())

    if base_rate and base_rate > 0:
        lift = tgt_rate / base_rate

# Segment-Tabelle + KPI Tabelle
seg_tbl = seg_quality.copy() if seg_quality is not None else None
kpi_out = kpi_df.copy()

# ----------------------------
# 2) Report-Text bauen
# ----------------------------
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

lines = []
lines.append("KAMPAGNEN-ABSCHLUSSREPORT")
lines.append("=" * 60)
lines.append(f"Zeitpunkt: {now_str}")
lines.append("")
lines.append("1) SETUP")
lines.append("-" * 60)
lines.append(f"Calibration-Methode: {cal_method}")
lines.append(f"Kosten pro Kontakt: {cost_per_contact}")
lines.append(f"Wert pro Response:  {value_per_response}")
if K_star_full is not None:
    lines.append(f"K* (Full): {K_star_full}")
if score_cutoff is not None:
    lines.append(f"Score-Cutoff: {score_cutoff}")
lines.append(f"Modell (Name): {best_model_name}")
lines.append("")
lines.append("2) BASIS & LIFT (Quality Gate)")
lines.append("-" * 60)
lines.append(f"Full n={n_all}")
lines.append(f"Targets n={n_tgt} | contact_rate={contact_rate:.4f}")
if base_rate is not None:
    lines.append(f"Base-Rate (Full, Ist):   {base_rate:.4f}")
if tgt_rate is not None:
    lines.append(f"Target-Rate (Targets):   {tgt_rate:.4f}")
if lift is not None:
    lines.append(f"Lift (Targets vs Full):  {lift:.2f}")
lines.append("")
lines.append("3) SEGMENT-INSIGHTS (Targets vs Gesamt)")
lines.append("-" * 60)

if seg_tbl is not None:
    show_cols = [
        "Segment","n_all","share_all","n_tgt","share_tgt","target_vs_all_ratio",
        "exp_resp_tgt","profit_tgt_expected","roi_tgt_expected"
    ]
    for c in ["actual_rate_all","actual_rate_tgt","lift_actual_tgt_vs_all"]:
        if c in seg_tbl.columns:
            show_cols.append(c)

    seg_view = seg_tbl[show_cols].copy()

    # Formatierung
    for c in ["share_all","share_tgt","target_vs_all_ratio","roi_tgt_expected","actual_rate_all","actual_rate_tgt","lift_actual_tgt_vs_all"]:
        if c in seg_view.columns:
            seg_view[c] = seg_view[c].astype(float).map(lambda x: f"{x:.4f}")
    for c in ["exp_resp_tgt","profit_tgt_expected"]:
        if c in seg_view.columns:
            seg_view[c] = seg_view[c].astype(float).map(lambda x: f"{x:.2f}")

    lines.append(seg_view.to_string(index=False))
else:
    lines.append("Kein Segment-Report vorhanden.")

lines.append("")
lines.append("4) POLICY (2-Wellen) — Expected KPIs")
lines.append("-" * 60)
lines.append(kpi_out.to_string(index=False))
lines.append("")
lines.append("5) TESTPLAN (Kurz)")
lines.append("-" * 60)
if "plan_lines" in globals():
    lines.extend(plan_lines)
else:
    lines.append("Kein Testplan-Text verfügbar (plan_lines fehlt).")

lines.append("")
lines.append("6) EXPORTS")
lines.append("-" * 60)
lines.append("Targetlisten & Reports:")
lines.append(f"- {wave1_path}")
lines.append(f"- {wave2_path}")
lines.append(f"- {policy_path}")
lines.append(f"- {policy_report_path}")
lines.append("")
lines.append("Hinweis:")
lines.append("Expected Responses basieren auf kalibriertem Score (Summe der Wahrscheinlichkeiten).")
lines.append("Für eine echte Kampagne: Responses tracken und Wave1/Wave2 getrennt evaluieren.")

# ----------------------------
# 3) Schreiben
# ----------------------------
final_report_path = "kampagnen_final_report.txt"
with open(final_report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("=== FINAL REPORT EXPORT ===")
print("-", final_report_path)

print("\nPreview (erste 40 Zeilen):")
print("\n".join(lines[:40]))

print("\nObjekt bereitgestellt:")
print("- final_report_path")


=== FINAL REPORT EXPORT ===
- kampagnen_final_report.txt

Preview (erste 40 Zeilen):
KAMPAGNEN-ABSCHLUSSREPORT
Zeitpunkt: 2026-01-29 18:37:26

1) SETUP
------------------------------------------------------------
Calibration-Methode: isotonic
Kosten pro Kontakt: 0.8
Wert pro Response:  5
K* (Full): 700
Score-Cutoff: 0.17090521442495127
Modell (Name): RandomForest_SAFE

2) BASIS & LIFT (Quality Gate)
------------------------------------------------------------
Full n=2240
Targets n=700 | contact_rate=0.3125
Base-Rate (Full, Ist):   0.1491
Target-Rate (Targets):   0.4771
Lift (Targets vs Full):  3.20

3) SEGMENT-INSIGHTS (Targets vs Gesamt)
------------------------------------------------------------
                          Segment  n_all share_all  n_tgt share_tgt target_vs_all_ratio exp_resp_tgt profit_tgt_expected roi_tgt_expected actual_rate_all actual_rate_tgt lift_actual_tgt_vs_all
 Segment A: High-Value / Frequent   1109    0.4951    476    0.6800              1.3735       221.1

In [44]:
# ============================================================
# Zelle X — MANAGEMENT HTML REPORT (Charts + Tabellen + Text)
# Output: management_report.html
# ============================================================

import os
import io
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# ---------- Helpers ----------
def fig_to_base64(fig, dpi=160):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")

def df_to_html_table(df, max_rows=30):
    if df is None:
        return "<p><em>(Keine Tabelle verfügbar)</em></p>"
    dfx = df.copy()
    if len(dfx) > max_rows:
        dfx = dfx.head(max_rows)
    return dfx.to_html(index=False, escape=True, classes="tbl", border=0)

def safe_get(name):
    return globals().get(name, None)

# ---------- Collect Objects (optional) ----------
kpi_df = safe_get("kpi_df")
seg_quality = safe_get("seg_quality")
gains_holdout_table = safe_get("gains_holdout_table")
topk_holdout_table = safe_get("topk_holdout_table")
roi_table = safe_get("roi_table")
profit_curve_extended = safe_get("profit_curve_extended")  # falls vorhanden
breakeven_table = safe_get("breakeven_table")

cost_per_contact = safe_get("cost_per_contact")
value_per_response = safe_get("value_per_response")
cal_method = safe_get("cal_method")

# ---------- Build Charts ----------
charts = []

# 1) Gains Curve (Holdout) if available
if isinstance(gains_holdout_table, pd.DataFrame) and {"Anteil_Kontaktiert", "Gain(Recall_kumuliert)", "Modell"}.issubset(gains_holdout_table.columns):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    for model_name, g in gains_holdout_table.groupby("Modell"):
        ax.plot(g["Anteil_Kontaktiert"], g["Gain(Recall_kumuliert)"], label=str(model_name))
    ax.set_xlabel("Anteil kontaktiert")
    ax.set_ylabel("Kumulative Gains (Recall)")
    ax.set_title("Cumulative Gains (Holdout)")
    ax.legend()
    charts.append(("Cumulative Gains (Holdout)", fig_to_base64(fig)))

# 2) Precision@K (Holdout) if available
if isinstance(topk_holdout_table, pd.DataFrame) and {"K", "Precision@K", "Modell"}.issubset(topk_holdout_table.columns):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    for model_name, g in topk_holdout_table.groupby("Modell"):
        ax.plot(g["K"], g["Precision@K"], label=str(model_name))
    ax.set_xlabel("K (Top-K Kontakte)")
    ax.set_ylabel("Precision@K")
    ax.set_title("Precision@K (Holdout)")
    ax.legend()
    charts.append(("Precision@K (Holdout)", fig_to_base64(fig)))

# 3) ROI / Profit vs K (wenn roi_table verfügbar)
if isinstance(roi_table, pd.DataFrame) and {"K", "expected_profit", "ROI"}.issubset(roi_table.columns):
    # Profit
    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.plot(roi_table["K"], roi_table["expected_profit"])
    ax.set_xlabel("K (Kontakte)")
    ax.set_ylabel("Expected Profit")
    ax.set_title("Expected Profit vs K")
    charts.append(("Expected Profit vs K", fig_to_base64(fig)))

    # ROI
    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.plot(roi_table["K"], roi_table["ROI"])
    ax.set_xlabel("K (Kontakte)")
    ax.set_ylabel("ROI")
    ax.set_title("ROI vs K")
    charts.append(("ROI vs K", fig_to_base64(fig)))

# 4) Segment-Mix (Targets vs All) aus seg_quality (wenn vorhanden)
if isinstance(seg_quality, pd.DataFrame) and {"Segment", "share_all", "share_tgt"}.issubset(seg_quality.columns):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    seg = seg_quality.copy()
    seg["share_all"] = pd.to_numeric(seg["share_all"], errors="coerce")
    seg["share_tgt"] = pd.to_numeric(seg["share_tgt"], errors="coerce")
    x = np.arange(len(seg))
    width = 0.35
    ax.bar(x - width/2, seg["share_all"], width, label="All")
    ax.bar(x + width/2, seg["share_tgt"], width, label="Targets")
    ax.set_xticks(x)
    ax.set_xticklabels(seg["Segment"], rotation=15, ha="right")
    ax.set_ylabel("Anteil")
    ax.set_title("Segment-Mix: All vs Targets")
    ax.legend()
    charts.append(("Segment-Mix", fig_to_base64(fig)))

# ---------- Executive Summary numbers (best effort) ----------
# If kpi_df exists, we pick TOTAL row
summary = {}
if isinstance(kpi_df, pd.DataFrame) and "label" in kpi_df.columns:
    total_row = kpi_df.loc[kpi_df["label"].astype(str).str.upper().eq("TOTAL")]
    if len(total_row) == 1:
        r = total_row.iloc[0].to_dict()
        summary = {
            "contacts_total": int(r.get("contacts", np.nan)),
            "expected_responses_total": float(r.get("expected_responses", np.nan)),
            "expected_profit_total": float(r.get("expected_profit", np.nan)),
            "expected_roi_total": float(r.get("expected_ROI", np.nan)),
        }

now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ---------- HTML Template ----------
css = """
<style>
@page { size: A4; margin: 18mm; }
body { font-family: Arial, sans-serif; color: #111; }
h1,h2,h3 { margin: 0 0 8px 0; }
p { line-height: 1.35; margin: 6px 0; }
.small { font-size: 12px; color: #444; }
.box { border: 1px solid #ddd; padding: 10px 12px; margin: 10px 0; border-radius: 6px; }
.grid { display: grid; grid-template-columns: 1fr 1fr; gap: 10px; }
.kpi { font-size: 14px; }
.kpi b { font-size: 18px; }
.tbl { border-collapse: collapse; width: 100%; font-size: 12px; }
.tbl th, .tbl td { border: 1px solid #ddd; padding: 6px 8px; }
.tbl th { background: #f5f5f5; }
.figure { page-break-inside: avoid; margin: 10px 0 16px 0; }
.figure img { width: 100%; height: auto; border: 1px solid #eee; }
hr { border: none; border-top: 1px solid #ddd; margin: 14px 0; }
</style>
"""

def fmt(x, nd=2):
    try:
        return f"{float(x):.{nd}f}"
    except Exception:
        return "n/a"

exec_box = f"""
<div class="box">
  <h2>Executive Summary</h2>
  <div class="grid">
    <div class="kpi">Kontakte (TOTAL)<br><b>{summary.get("contacts_total","n/a")}</b></div>
    <div class="kpi">Expected Responses (TOTAL)<br><b>{fmt(summary.get("expected_responses_total", np.nan), 1)}</b></div>
    <div class="kpi">Expected Profit (TOTAL)<br><b>{fmt(summary.get("expected_profit_total", np.nan), 2)}</b></div>
    <div class="kpi">Expected ROI (TOTAL)<br><b>{fmt(summary.get("expected_roi_total", np.nan), 2)}</b></div>
  </div>
  <p class="small">
    Hinweis: Expected Responses = Summe kalibrierter Wahrscheinlichkeiten (Score). 
    Szenario-Parameter: cost_per_contact={cost_per_contact}, value_per_response={value_per_response}, cal_method={cal_method}.
  </p>
</div>
"""

# Charts HTML
charts_html = ""
for title, b64 in charts:
    charts_html += f"""
    <div class="figure">
      <h3>{title}</h3>
      <img src="data:image/png;base64,{b64}" alt="{title}">
    </div>
    """

html = f"""
<html>
<head>
<meta charset="utf-8">
<title>Management Report – Marktkampagne</title>
{css}
</head>
<body>
  <h1>Management Report – Marktkampagne (Targeting & ROI)</h1>
  <p class="small">Stand: {now}</p>

  {exec_box}

  <hr>

  <h2>1) KPI-Übersicht (Policy / Wellen)</h2>
  {df_to_html_table(kpi_df, max_rows=10)}

  <h2>2) Segment-Insights</h2>
  {df_to_html_table(seg_quality, max_rows=10)}

  <h2>3) Model Performance (Holdout)</h2>
  {df_to_html_table(topk_holdout_table, max_rows=20)}
  {df_to_html_table(gains_holdout_table, max_rows=20)}

  <h2>4) ROI / Budget Logik</h2>
  {df_to_html_table(roi_table, max_rows=20)}
  {df_to_html_table(breakeven_table, max_rows=20)}

  <h2>5) Diagramme</h2>
  {charts_html}

  <hr>
  <h2>Empfehlung (kurz)</h2>
  <p>
    Wir nutzen ein priorisiertes Targeting (Top-K) mit kalibrierten Scores zur Budgetsteuerung.
    Die Policy kann als 2-Wellen-Rollout umgesetzt werden (Wave1 stabil, Wave2 Exploration).
    Erfolg wird über Tracking (ID, Wave, Kontaktzeitpunkt, Response-Flag) gemessen.
  </p>

  <p class="small">
    Technischer Hinweis: Diagramme sind statisch (PNG) für PDF-freundlichen Export.
    PDF-Erstellung: Datei im Browser öffnen → Drucken → Als PDF speichern.
  </p>
</body>
</html>
"""

out_path = "management_report.html"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(html)

print("=== HTML REPORT FERTIG ===")
print(out_path)
print("Öffnen im Browser und 'Drucken → Als PDF speichern' für die GL-Version.")


=== HTML REPORT FERTIG ===
management_report.html
Öffnen im Browser und 'Drucken → Als PDF speichern' für die GL-Version.


In [45]:
# ============================================================
# Zelle 35 — MANAGEMENT REPORT als HTML (Charts + Text + Tabellen)
# Output: management_report.html
# ============================================================
"""
Analyst-Notiz:
Diese Zelle erzeugt einen kompakten, GL-tauglichen Management-Report als HTML.
Der Report enthält:
- Executive Summary (Kernzahlen)
- KPI-Übersicht (Wave1/Wave2/Total)
- Holdout-Performance (Top-K + Gains)
- ROI/Profit-Logik
- Segment-Insights
- Calibration-Überblick (falls vorhanden)
- Diagramme als eingebettete PNGs (Base64), damit die HTML aus einer Datei funktioniert.

PDF-Erstellung:
- management_report.html im Browser öffnen -> Drucken -> Als PDF speichern
"""

import io
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# ------------------------------------------------------------
# 0) Safe Getter (Zelle läuft auch, wenn einzelne Objekte fehlen)
# ------------------------------------------------------------
def safe_get(name):
    return globals().get(name, None)

df_scoring_full       = safe_get("df_scoring_full")
df_target_topK        = safe_get("df_target_topK")
kpi_df                = safe_get("kpi_df")
seg_quality           = safe_get("seg_quality")
topk_holdout_table    = safe_get("topk_holdout_table")
gains_holdout_table   = safe_get("gains_holdout_table")

roi_table             = safe_get("roi_table")
profit_curve_extended = safe_get("profit_curve_extended")
breakeven_table       = safe_get("breakeven_table")

brier_scores_df       = safe_get("brier_scores_df")
rel_uncal             = safe_get("rel_uncal")
rel_sig               = safe_get("rel_sig")
rel_iso               = safe_get("rel_iso")

cost_per_contact      = safe_get("cost_per_contact")
value_per_response    = safe_get("value_per_response")
cal_method            = safe_get("cal_method")

# ------------------------------------------------------------
# 1) Kleine Helper
# ------------------------------------------------------------
def fig_to_base64(fig, dpi=170):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")

def df_to_html_table(df, max_rows=25):
    if not isinstance(df, pd.DataFrame) or df.empty:
        return "<p><em>(Keine Tabelle verfügbar)</em></p>"
    dfx = df.copy()
    if len(dfx) > max_rows:
        dfx = dfx.head(max_rows)
    return dfx.to_html(index=False, escape=True, classes="tbl", border=0)

def fmt(x, nd=2):
    try:
        return f"{float(x):.{nd}f}"
    except Exception:
        return "n/a"

def fmt_pct(x, nd=1):
    try:
        return f"{100*float(x):.{nd}f}%"
    except Exception:
        return "n/a"

def has_cols(df, cols):
    return isinstance(df, pd.DataFrame) and set(cols).issubset(df.columns)

now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ------------------------------------------------------------
# 2) Executive Summary ableiten (best effort)
# ------------------------------------------------------------
summary = {
    "n_all": len(df_scoring_full) if isinstance(df_scoring_full, pd.DataFrame) else None,
    "n_tgt": len(df_target_topK)  if isinstance(df_target_topK, pd.DataFrame) else None,
    "contact_rate": None,
    "base_rate": None,
    "target_rate": None,
    "lift": None,
}

# Base/Target-Rate über bekannte Label-Spalten (falls vorhanden)
label_col = None
if isinstance(df_scoring_full, pd.DataFrame):
    for c in ["Antwort_Letzte_Kampagne", "ist_antwort"]:
        if c in df_scoring_full.columns:
            label_col = c
            break

if summary["n_all"] and summary["n_tgt"]:
    summary["contact_rate"] = summary["n_tgt"] / summary["n_all"]

if label_col and isinstance(df_scoring_full, pd.DataFrame) and isinstance(df_target_topK, pd.DataFrame):
    base_rate = pd.to_numeric(df_scoring_full[label_col], errors="coerce").mean()
    tgt_rate  = pd.to_numeric(df_target_topK[label_col], errors="coerce").mean()
    summary["base_rate"]   = float(base_rate) if pd.notna(base_rate) else None
    summary["target_rate"] = float(tgt_rate)  if pd.notna(tgt_rate)  else None
    if summary["base_rate"] and summary["base_rate"] > 0 and summary["target_rate"] is not None:
        summary["lift"] = summary["target_rate"] / summary["base_rate"]

# KPI TOTAL aus kpi_df (falls vorhanden)
kpi_total = {}
if isinstance(kpi_df, pd.DataFrame) and "label" in kpi_df.columns:
    total_row = kpi_df.loc[kpi_df["label"].astype(str).str.upper().eq("TOTAL")]
    if len(total_row) == 1:
        r = total_row.iloc[0].to_dict()
        kpi_total = {
            "contacts": int(r.get("contacts", np.nan)) if pd.notna(r.get("contacts", np.nan)) else None,
            "expected_responses": float(r.get("expected_responses", np.nan)),
            "expected_profit": float(r.get("expected_profit", np.nan)),
            "expected_roi": float(r.get("expected_ROI", np.nan)),
            "cost_total": float(r.get("cost_total", np.nan)),
            "expected_value": float(r.get("expected_value", np.nan)),
        }

# ------------------------------------------------------------
# 3) Diagramme bauen (nur wenn Daten verfügbar)
# ------------------------------------------------------------
charts = []

# (A) Cumulative Gains (Holdout)
if has_cols(gains_holdout_table, ["Anteil_Kontaktiert", "Gain(Recall_kumuliert)", "Modell"]):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    for model_name, g in gains_holdout_table.groupby("Modell"):
        ax.plot(g["Anteil_Kontaktiert"], g["Gain(Recall_kumuliert)"], label=str(model_name))
    ax.set_xlabel("Anteil kontaktiert")
    ax.set_ylabel("Kumulative Gains (Recall)")
    ax.set_title("Cumulative Gains (Holdout)")
    ax.legend()
    charts.append(("Cumulative Gains (Holdout)", fig_to_base64(fig)))

# (B) Precision@K (Holdout)
if has_cols(topk_holdout_table, ["K", "Precision@K", "Modell"]):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    for model_name, g in topk_holdout_table.groupby("Modell"):
        ax.plot(g["K"], g["Precision@K"], label=str(model_name))
    ax.set_xlabel("K (Top-K Kontakte)")
    ax.set_ylabel("Precision@K")
    ax.set_title("Precision@K (Holdout)")
    ax.legend()
    charts.append(("Precision@K (Holdout)", fig_to_base64(fig)))

# (C) Profit/ROI vs K (ROI-Tabelle)
if has_cols(roi_table, ["K", "expected_profit", "ROI"]):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.plot(roi_table["K"], roi_table["expected_profit"])
    ax.set_xlabel("K (Kontakte)")
    ax.set_ylabel("Expected Profit")
    ax.set_title("Expected Profit vs K")
    charts.append(("Expected Profit vs K", fig_to_base64(fig)))

    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.plot(roi_table["K"], roi_table["ROI"])
    ax.set_xlabel("K (Kontakte)")
    ax.set_ylabel("ROI")
    ax.set_title("ROI vs K")
    charts.append(("ROI vs K", fig_to_base64(fig)))

# (D) Segment-Mix: All vs Targets
if has_cols(seg_quality, ["Segment", "share_all", "share_tgt"]):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    seg = seg_quality.copy()
    seg["share_all"] = pd.to_numeric(seg["share_all"], errors="coerce")
    seg["share_tgt"] = pd.to_numeric(seg["share_tgt"], errors="coerce")
    x = np.arange(len(seg))
    width = 0.35
    ax.bar(x - width/2, seg["share_all"], width, label="All")
    ax.bar(x + width/2, seg["share_tgt"], width, label="Targets")
    ax.set_xticks(x)
    ax.set_xticklabels(seg["Segment"], rotation=15, ha="right")
    ax.set_ylabel("Anteil")
    ax.set_title("Segment-Mix: All vs Targets")
    ax.legend()
    charts.append(("Segment-Mix: All vs Targets", fig_to_base64(fig)))

# (E) KPI Bars: Expected Profit / ROI pro Wave
if has_cols(kpi_df, ["label", "expected_profit", "expected_ROI", "contacts"]):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    d = kpi_df.copy()
    d["expected_profit"] = pd.to_numeric(d["expected_profit"], errors="coerce")
    ax.bar(d["label"].astype(str), d["expected_profit"])
    ax.set_xlabel("Wave")
    ax.set_ylabel("Expected Profit")
    ax.set_title("Expected Profit nach Wave")
    charts.append(("Expected Profit nach Wave", fig_to_base64(fig)))

    fig = plt.figure()
    ax = fig.add_subplot(111)
    d["expected_ROI"] = pd.to_numeric(d["expected_ROI"], errors="coerce")
    ax.bar(d["label"].astype(str), d["expected_ROI"])
    ax.set_xlabel("Wave")
    ax.set_ylabel("Expected ROI")
    ax.set_title("Expected ROI nach Wave")
    charts.append(("Expected ROI nach Wave", fig_to_base64(fig)))

# (F) Calibration (Brier + Reliability) falls vorhanden
if has_cols(brier_scores_df, ["Variante", "Brier_Score"]):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    x = brier_scores_df["Variante"].astype(str)
    y = pd.to_numeric(brier_scores_df["Brier_Score"], errors="coerce")
    ax.bar(x, y)
    ax.set_xlabel("Variante")
    ax.set_ylabel("Brier Score (niedriger = besser)")
    ax.set_title("Calibration-Check: Brier Score")
    charts.append(("Calibration-Check: Brier Score", fig_to_base64(fig)))

def reliability_plot(rel_df, title):
    if not has_cols(rel_df, ["mean_proba", "actual_rate"]):
        return None
    fig = plt.figure()
    ax = fig.add_subplot(111)
    x = pd.to_numeric(rel_df["mean_proba"], errors="coerce")
    y = pd.to_numeric(rel_df["actual_rate"], errors="coerce")
    ax.plot([0,1],[0,1], linestyle="--")
    ax.plot(x, y, marker="o")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Actual rate")
    ax.set_title(title)
    return fig

for rel_df, title in [(rel_uncal, "Reliability: Uncalibrated"),
                      (rel_sig, "Reliability: Sigmoid (Platt)"),
                      (rel_iso, "Reliability: Isotonic")]:
    fig = reliability_plot(rel_df, title)
    if fig is not None:
        charts.append((title, fig_to_base64(fig)))

# ------------------------------------------------------------
# 4) HTML bauen
# ------------------------------------------------------------
css = """
<style>
@page { size: A4; margin: 18mm; }
body { font-family: Arial, sans-serif; color: #111; }
h1 { margin: 0 0 6px 0; font-size: 22px; }
h2 { margin: 18px 0 8px 0; font-size: 16px; }
h3 { margin: 10px 0 6px 0; font-size: 14px; }
p { margin: 6px 0; line-height: 1.35; }
.small { font-size: 12px; color: #444; }
.box { border: 1px solid #ddd; padding: 10px 12px; margin: 10px 0; border-radius: 6px; }
.grid { display: grid; grid-template-columns: 1fr 1fr; gap: 10px; }
.kpi { font-size: 13px; }
.kpi b { font-size: 18px; }
.tbl { border-collapse: collapse; width: 100%; font-size: 12px; }
.tbl th, .tbl td { border: 1px solid #ddd; padding: 6px 8px; }
.tbl th { background: #f5f5f5; }
.figure { page-break-inside: avoid; margin: 10px 0 16px 0; }
.figure img { width: 100%; height: auto; border: 1px solid #eee; }
hr { border: none; border-top: 1px solid #ddd; margin: 14px 0; }
.note { font-size: 12px; color: #333; }
</style>
"""

# Executive Summary Box
exec_lines = []
exec_lines.append(f"<div class='kpi'>Kontakte (Full)<br><b>{summary['n_all'] if summary['n_all'] is not None else 'n/a'}</b></div>")
exec_lines.append(f"<div class='kpi'>Targets (K)<br><b>{summary['n_tgt'] if summary['n_tgt'] is not None else 'n/a'}</b></div>")
exec_lines.append(f"<div class='kpi'>Contact Rate<br><b>{fmt_pct(summary['contact_rate'],1) if summary['contact_rate'] is not None else 'n/a'}</b></div>")
exec_lines.append(f"<div class='kpi'>Lift (Targets vs Full)<br><b>{fmt(summary['lift'],2) if summary['lift'] is not None else 'n/a'}</b></div>")

exec_kpi_2 = []
exec_kpi_2.append(f"<div class='kpi'>Expected Responses (TOTAL)<br><b>{fmt(kpi_total.get('expected_responses', np.nan),1)}</b></div>")
exec_kpi_2.append(f"<div class='kpi'>Expected Profit (TOTAL)<br><b>{fmt(kpi_total.get('expected_profit', np.nan),2)}</b></div>")
exec_kpi_2.append(f"<div class='kpi'>Expected ROI (TOTAL)<br><b>{fmt(kpi_total.get('expected_roi', np.nan),2)}</b></div>")
exec_kpi_2.append(f"<div class='kpi'>Kosten / Wert<br><b>{cost_per_contact} / {value_per_response}</b></div>")

exec_box = f"""
<div class="box">
  <h2>Executive Summary</h2>
  <div class="grid">
    {exec_lines[0]}{exec_lines[1]}{exec_lines[2]}{exec_lines[3]}
  </div>
  <div class="grid" style="margin-top:10px;">
    {exec_kpi_2[0]}{exec_kpi_2[1]}{exec_kpi_2[2]}{exec_kpi_2[3]}
  </div>
  <p class="small">
    Stand: {now} | cal_method={cal_method} | Expected Responses = Summe kalibrierter Scores (Wahrscheinlichkeiten).
  </p>
</div>
"""

# Charts HTML
charts_html = ""
for title, b64 in charts:
    charts_html += f"""
    <div class="figure">
      <h3>{title}</h3>
      <img src="data:image/png;base64,{b64}" alt="{title}">
    </div>
    """

# Target Preview Tabelle (Top 20)
tgt_preview_html = "<p><em>(Keine Targetliste verfügbar)</em></p>"
if isinstance(df_target_topK, pd.DataFrame) and not df_target_topK.empty:
    cols_pref = [c for c in ["ID","rank","score_calibrated","Segment","Cluster"] if c in df_target_topK.columns]
    if cols_pref:
        tgt_preview_html = df_target_topK[cols_pref].head(20).to_html(index=False, escape=True, classes="tbl", border=0)
    else:
        tgt_preview_html = df_target_topK.head(20).to_html(index=False, escape=True, classes="tbl", border=0)

html = f"""
<html>
<head>
<meta charset="utf-8">
<title>Management Report – Marktkampagne</title>
{css}
</head>
<body>
  <h1>Management Report – Marktkampagne (Targeting, ROI, Policy)</h1>
  <p class="small">Erstellt am: {now}</p>

  {exec_box}

  <h2>1) KPI-Übersicht (Policy / Wellen)</h2>
  {df_to_html_table(kpi_df, max_rows=10)}

  <h2>2) Diagramme (Management-Übersicht)</h2>
  {charts_html}

  <h2>3) Segment-Insights (All vs Targets)</h2>
  {df_to_html_table(seg_quality, max_rows=20)}

  <h2>4) Holdout-Performance (Top-K & Gains)</h2>
  {df_to_html_table(topk_holdout_table, max_rows=25)}
  {df_to_html_table(gains_holdout_table, max_rows=25)}

  <h2>5) ROI / Budget Logik</h2>
  {df_to_html_table(roi_table, max_rows=30)}
  {df_to_html_table(breakeven_table, max_rows=30)}

  <h2>6) Targetliste (Preview Top 20)</h2>
  {tgt_preview_html}

  <hr>

  <h2>Empfehlung (kurz)</h2>
  <p class="note">
    Empfehlung: Kampagne als 2-Wellen-Policy ausrollen (Wave1 stabil/profitgetrieben, Wave2 Exploration).
    Erfolgsmessung über sauberes Tracking (ID, Wave, Timestamp, Response-Flag). Nach erster Teilausspielung Review
    und Budgetumschichtung nach ROI/Profit.
  </p>

  <p class="small">
    PDF: Datei im Browser öffnen -> Drucken -> Als PDF speichern (A4).
  </p>
</body>
</html>
"""

out_path = "management_report.html"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(html)

print("=== HTML REPORT FERTIG ===")
print(out_path)
print("Öffnen im Browser und 'Drucken -> Als PDF speichern' für die Geschäftsleitung.")


=== HTML REPORT FERTIG ===
management_report.html
Öffnen im Browser und 'Drucken -> Als PDF speichern' für die Geschäftsleitung.


In [46]:
# ============================================================
# Zelle — 15-Minuten Präsentation als HTML (Seite für Seite)
# Output: praesentation_15min.html
# ============================================================
"""
Analyst-Notiz:
Dieses HTML ist als "Vortragsdokument" gedacht:
- Jede Seite entspricht einer Präsentationsseite (mit Page Break)
- Pro Seite ein kleines Diagramm (rechts) + Sprechtext (links)
- Im Browser nutzbar, und als PDF exportierbar (Drucken -> PDF)
"""

import io
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# ----------------------------
# 0) Safe Getter
# ----------------------------
def safe_get(name):
    return globals().get(name, None)

df_scoring_full       = safe_get("df_scoring_full")
df_target_topK        = safe_get("df_target_topK")

kpi_df                = safe_get("kpi_df")
seg_quality           = safe_get("seg_quality")

topk_holdout_table    = safe_get("topk_holdout_table")
gains_holdout_table   = safe_get("gains_holdout_table")

roi_table             = safe_get("roi_table")
brier_scores_df       = safe_get("brier_scores_df")

cost_per_contact      = safe_get("cost_per_contact")
value_per_response    = safe_get("value_per_response")
cal_method            = safe_get("cal_method")

# ----------------------------
# 1) Helpers
# ----------------------------
def fig_to_base64(fig, dpi=170):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")

def has_cols(df, cols):
    return isinstance(df, pd.DataFrame) and set(cols).issubset(df.columns)

def fmt(x, nd=2):
    try:
        return f"{float(x):.{nd}f}"
    except Exception:
        return "n/a"

def pct(x, nd=1):
    try:
        return f"{100*float(x):.{nd}f}%"
    except Exception:
        return "n/a"

def chart_placeholder(msg="(Chart nicht verfügbar)"):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.axis("off")
    ax.text(0.5, 0.5, msg, ha="center", va="center")
    return fig_to_base64(fig)

now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ----------------------------
# 2) Kennzahlen (best effort)
# ----------------------------
n_all = len(df_scoring_full) if isinstance(df_scoring_full, pd.DataFrame) else None
n_tgt = len(df_target_topK)  if isinstance(df_target_topK, pd.DataFrame) else None
contact_rate = (n_tgt / n_all) if (n_all and n_tgt) else None

# Label-Spalte finden (falls vorhanden)
label_col = None
if isinstance(df_scoring_full, pd.DataFrame):
    for c in ["Antwort_Letzte_Kampagne", "ist_antwort"]:
        if c in df_scoring_full.columns:
            label_col = c
            break

base_rate = None
target_rate = None
lift = None
if label_col and isinstance(df_scoring_full, pd.DataFrame) and isinstance(df_target_topK, pd.DataFrame):
    base_rate = pd.to_numeric(df_scoring_full[label_col], errors="coerce").mean()
    target_rate = pd.to_numeric(df_target_topK[label_col], errors="coerce").mean()
    if pd.notna(base_rate) and base_rate > 0 and pd.notna(target_rate):
        lift = float(target_rate) / float(base_rate)
    base_rate = float(base_rate) if pd.notna(base_rate) else None
    target_rate = float(target_rate) if pd.notna(target_rate) else None

# KPI TOTAL aus kpi_df (falls vorhanden)
kpi_total = {}
if isinstance(kpi_df, pd.DataFrame) and "label" in kpi_df.columns:
    total_row = kpi_df.loc[kpi_df["label"].astype(str).str.upper().eq("TOTAL")]
    if len(total_row) == 1:
        r = total_row.iloc[0].to_dict()
        kpi_total = {
            "contacts": int(r.get("contacts", np.nan)) if pd.notna(r.get("contacts", np.nan)) else None,
            "expected_responses": float(r.get("expected_responses", np.nan)),
            "expected_profit": float(r.get("expected_profit", np.nan)),
            "expected_roi": float(r.get("expected_ROI", np.nan)),
            "cost_total": float(r.get("cost_total", np.nan)),
            "expected_value": float(r.get("expected_value", np.nan)),
        }

# ----------------------------
# 3) Charts (klein, pro Seite)
# ----------------------------

# Seite 2: Base vs Target Rate (Balken)
def make_chart_base_vs_target():
    if base_rate is None or target_rate is None:
        return chart_placeholder("Base/Target Rate fehlen")
    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.bar(["Full (Base)","Targets"], [base_rate, target_rate])
    ax.set_ylim(0, max(target_rate, base_rate) * 1.2)
    ax.set_ylabel("Response-Rate")
    ax.set_title("Base vs Target Response-Rate")
    return fig_to_base64(fig)

# Seite 3: Gains (Holdout)
def make_chart_gains_holdout():
    if not has_cols(gains_holdout_table, ["Anteil_Kontaktiert", "Gain(Recall_kumuliert)", "Modell"]):
        return chart_placeholder("Gains Holdout fehlt")
    fig = plt.figure()
    ax = fig.add_subplot(111)
    for model_name, g in gains_holdout_table.groupby("Modell"):
        ax.plot(g["Anteil_Kontaktiert"], g["Gain(Recall_kumuliert)"], label=str(model_name))
    ax.set_xlabel("Anteil kontaktiert")
    ax.set_ylabel("Kumulative Gains")
    ax.set_title("Cumulative Gains (Holdout)")
    ax.legend(fontsize=8)
    return fig_to_base64(fig)

# Seite 4: Profit vs K (ROI Table)
def make_chart_profit_vs_k():
    if not has_cols(roi_table, ["K", "expected_profit"]):
        return chart_placeholder("ROI/Profit Tabelle fehlt")
    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.plot(roi_table["K"], roi_table["expected_profit"])
    ax.set_xlabel("K (Kontakte)")
    ax.set_ylabel("Expected Profit")
    ax.set_title("Expected Profit vs K")
    return fig_to_base64(fig)

# Seite 5: Segment-Mix
def make_chart_segment_mix():
    if not has_cols(seg_quality, ["Segment", "share_all", "share_tgt"]):
        return chart_placeholder("Segment-Mix fehlt")
    seg = seg_quality.copy()
    seg["share_all"] = pd.to_numeric(seg["share_all"], errors="coerce")
    seg["share_tgt"] = pd.to_numeric(seg["share_tgt"], errors="coerce")
    fig = plt.figure()
    ax = fig.add_subplot(111)
    x = np.arange(len(seg))
    width = 0.35
    ax.bar(x - width/2, seg["share_all"], width, label="All")
    ax.bar(x + width/2, seg["share_tgt"], width, label="Targets")
    ax.set_xticks(x)
    ax.set_xticklabels(seg["Segment"], rotation=15, ha="right")
    ax.set_ylabel("Anteil")
    ax.set_title("Segment-Mix: All vs Targets")
    ax.legend(fontsize=8)
    return fig_to_base64(fig)

# Seite 6: Wave Profit Balken
def make_chart_wave_profit():
    if not has_cols(kpi_df, ["label", "expected_profit"]):
        return chart_placeholder("KPI/Waves fehlen")
    d = kpi_df.copy()
    d["expected_profit"] = pd.to_numeric(d["expected_profit"], errors="coerce")
    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.bar(d["label"].astype(str), d["expected_profit"])
    ax.set_xlabel("Wave")
    ax.set_ylabel("Expected Profit")
    ax.set_title("Expected Profit nach Wave")
    return fig_to_base64(fig)

# Seite 7: Calibration (Brier)
def make_chart_brier():
    if not has_cols(brier_scores_df, ["Variante","Brier_Score"]):
        return chart_placeholder("Brier Score fehlt")
    fig = plt.figure()
    ax = fig.add_subplot(111)
    x = brier_scores_df["Variante"].astype(str)
    y = pd.to_numeric(brier_scores_df["Brier_Score"], errors="coerce")
    ax.bar(x, y)
    ax.set_ylabel("Brier (niedriger besser)")
    ax.set_title("Calibration-Check: Brier Score")
    return fig_to_base64(fig)

charts = {
    "s2": make_chart_base_vs_target(),
    "s3": make_chart_gains_holdout(),
    "s4": make_chart_profit_vs_k(),
    "s5": make_chart_segment_mix(),
    "s6": make_chart_wave_profit(),
    "s7": make_chart_brier(),
}

# ----------------------------
# 4) Seiteninhalte (Sprechtext)
# ----------------------------
# Hinweis: Das ist "Vortragsdokument": links Sprechertext, rechts Chart
pages = []

# Seite 1 — Titel / Agenda
pages.append({
    "title": "Marktkampagne – Management Update (15 Minuten)",
    "subtitle": f"Stand: {now}",
    "bullets": [
        "Ziel: Kontakte priorisieren, um mehr Responses pro Euro zu erzielen.",
        "Vorgehen: Scoring → Holdout-Check → Kalibrierung → ROI/Budget → Policy (2 Wellen).",
        "Ergebnis: Konkrete Targetliste + Steuerungslogik für Kampagnenausspielung."
    ],
    "chart_b64": None
})

# Seite 2 — Ausgangslage & Lift
pages.append({
    "title": "Ausgangslage & warum Targeting Sinn macht",
    "subtitle": "Base Rate vs. Target Rate",
    "bullets": [
        f"Gesamtbestand: {n_all if n_all is not None else 'n/a'} Kunden.",
        f"Basis-Antwortquote (Full): {pct(base_rate,1) if base_rate is not None else 'n/a'}.",
        f"Target-Antwortquote (Targets): {pct(target_rate,1) if target_rate is not None else 'n/a'}.",
        f"Lift: {fmt(lift,2) if lift is not None else 'n/a'}× → höhere Response-Dichte in der Zielgruppe.",
        "Kernaussage: Wir erhöhen Effizienz, ohne das Budget einfach nur zu erhöhen."
    ],
    "chart_b64": charts["s2"]
})

# Seite 3 — Holdout Performance
pages.append({
    "title": "Validierung: Performance im Holdout",
    "subtitle": "Cumulative Gains (Holdout)",
    "bullets": [
        "Holdout = Datenanteil, der nicht fürs Training genutzt wurde → realistischere Leistungsprüfung.",
        "Cumulative Gains zeigt: Wie viele Responses wir finden, wenn wir nur einen Anteil der Kunden kontaktieren.",
        "Interpretation: Bei begrenztem Budget liefert Top-K deutlich mehr Responses als Zufallsauswahl."
    ],
    "chart_b64": charts["s3"]
})

# Seite 4 — ROI / Profit Steuerung
pages.append({
    "title": "Wirtschaftliche Übersetzung: Profit & ROI als Steuerungsgröße",
    "subtitle": "Expected Profit vs K",
    "bullets": [
        f"Szenario-Parameter: cost_per_contact={cost_per_contact}, value_per_response={value_per_response}, calibration={cal_method}.",
        "Expected Responses = Summe der kalibrierten Wahrscheinlichkeiten (Score-Summe).",
        "Damit wird K (Anzahl Kontakte) zu einem echten Budget-Regler: klein starten, messen, dann skalieren.",
        f"Wenn vorhanden: TOTAL Expected Profit ≈ {fmt(kpi_total.get('expected_profit', np.nan),2)} bei TOTAL Contacts={kpi_total.get('contacts','n/a')}."
    ],
    "chart_b64": charts["s4"]
})

# Seite 5 — Segment-Insights
pages.append({
    "title": "Segment-Insights: Wer steckt in den Targets?",
    "subtitle": "All vs Targets (Segment-Mix)",
    "bullets": [
        "Segment A (High-Value/Frequent) liefert typischerweise Stabilität und planbaren Profit.",
        "Segment D (Low-Value/Infrequent) kann hohen Lift liefern, ist aber potenziell volatiler.",
        "Konsequenz: Wir nutzen Segment-Infos für eine kontrollierte Rollout-Strategie (Wellen)."
    ],
    "chart_b64": charts["s5"]
})

# Seite 6 — Policy / Wellen
pages.append({
    "title": "Empfehlung: 2-Wellen-Policy für kontrollierten Rollout",
    "subtitle": "Wave1 (stabil) + Wave2 (Exploration)",
    "bullets": [
        "Wave 1: Fokus auf Stabilität/Profit (A-lastig) → schnelle, planbare Ergebnisse.",
        "Wave 2: Exploration (gezielt D-Anteil) → Lernen, ob Skalierung sinnvoll ist.",
        "Steuerung: gleiche Creatives/Offer pro Wave, damit Vergleich fair bleibt.",
        "Stop/Go: Wenn Wave2 nach Teil-Ausspielung schlechter ist → Budget in Wave1 verschieben."
    ],
    "chart_b64": charts["s6"]
})

# Seite 7 — Calibration Kurz
pages.append({
    "title": "Warum Kalibrierung wichtig ist (kurz)",
    "subtitle": "Brier Score als Qualitätscheck",
    "bullets": [
        "Scores sollen nicht nur ranken, sondern (annähernd) echte Wahrscheinlichkeiten abbilden.",
        "Brier Score: niedriger = bessere Übereinstimmung von Prognose und Realität.",
        "Praktischer Nutzen: ROI/Budget-Rechnungen werden belastbarer, weil die Score-Summe realistischer ist."
    ],
    "chart_b64": charts["s7"]
})

# Seite 8 — Entscheidung / Next Steps
pages.append({
    "title": "Entscheidung & nächste Schritte",
    "subtitle": "Was wir jetzt konkret tun",
    "bullets": [
        "1) Kampagne nach Targetliste ausspielen (Top-K / Policy-Wellen).",
        "2) Tracking sicherstellen: ID, Wave, Timestamp, Response-Flag.",
        "3) Nach erster Teilausspielung Review: Response-Rate, Profit, ROI; Budget nachziehen.",
        "Ergebnis: datengetriebene Kampagnensteuerung – messbar, kontrolliert, skalierbar."
    ],
    "chart_b64": None
})

# ----------------------------
# 5) HTML Layout (A4, Page Break)
# ----------------------------
css = """
<style>
@page { size: A4 landscape; margin: 12mm; }
body { font-family: Arial, sans-serif; color: #111; }
.page { page-break-after: always; border: 1px solid #eee; padding: 12px 14px; }
.header { display: flex; justify-content: space-between; align-items: baseline; }
h1 { font-size: 24px; margin: 0 0 6px 0; }
h2 { font-size: 18px; margin: 0 0 10px 0; color: #333; }
.small { font-size: 12px; color: #555; }
.grid { display: grid; grid-template-columns: 1.25fr 1fr; gap: 14px; align-items: start; }
ul { margin: 8px 0 0 18px; font-size: 14px; line-height: 1.35; }
li { margin: 6px 0; }
.chart { border: 1px solid #ddd; padding: 6px; }
.chart img { width: 100%; height: auto; display: block; }
.footer { margin-top: 8px; font-size: 12px; color: #666; }
.note { font-size: 13px; color: #222; }
</style>
"""

def page_html(p, idx, total):
    left = "<ul>" + "".join([f"<li>{b}</li>" for b in p["bullets"]]) + "</ul>"
    if p["chart_b64"] is None:
        right = "<div class='chart'><div class='note'>(Kein Chart auf dieser Seite)</div></div>"
    else:
        right = f"<div class='chart'><img src='data:image/png;base64,{p['chart_b64']}' alt='chart'></div>"
    return f"""
    <div class="page">
      <div class="header">
        <div>
          <h1>{p['title']}</h1>
          <h2>{p['subtitle']}</h2>
        </div>
        <div class="small">Seite {idx}/{total}</div>
      </div>

      <div class="grid">
        <div>{left}</div>
        <div>{right}</div>
      </div>

      <div class="footer">
        Notiz: Diese Seite ist als Sprechfolie gedacht. Export: Browser → Drucken → PDF.
      </div>
    </div>
    """

html_pages = "".join([page_html(p, i+1, len(pages)) for i, p in enumerate(pages)])

html = f"""
<html>
<head>
<meta charset="utf-8">
<title>Praesentation 15min – Marktkampagne</title>
{css}
</head>
<body>
{html_pages}
</body>
</html>
"""

out_path = "praesentation_15min.html"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(html)

print("=== PRÄSENTATION HTML FERTIG ===")
print(out_path)
print("Tipp: Im Browser öffnen -> Drucken -> Als PDF speichern (Landscape).")


=== PRÄSENTATION HTML FERTIG ===
praesentation_15min.html
Tipp: Im Browser öffnen -> Drucken -> Als PDF speichern (Landscape).
